In [3]:
"""
Music-video generator — v11, lyric-driven edition.

THE LYRICS DRIVE EVERYTHING
---------------------------------------------------------------------------
author_scenes() turns each song into a short film:
  * a film bible invented from the lyrics: logline, chapters, the
    PERFORMER(S) whose voice we hear (matched to the CLAP-detected voice),
    and whatever cast the lyrics call for -- any gender, age, kind, number;
  * every scene is "performance" (the performer sings and dances), "story"
    (characters act out what the lyric describes -- an angel pulling a man
    from a wreck, not a person singing about it), or "performance_in_story";
  * settings are literal: a park song is in a park, a Paris lyric is in
    recognizable Paris in the mood of that line;
  * HEAT follows each scene's own lines (none / romantic / sensual /
    explicit), capped by CONFIG["max_scene_heat"]; sexual content is adults
    only.

WHAT THIS DOES
---------------------------------------------------------------------------
All MiniMax H3 rendering happens OUTSIDE this process, via your Wan2GP fork
(https://github.com/fastflair/Wan2GP). This script only does the local work:
Whisper transcription, music analysis (tempo/beats/energy), LLM scene
authoring, Z-Image reference stills, and per-scene audio slicing -- then
hands the whole song's worth of scenes to `wgp.py` as ONE batch and lets
Wan2GP do the actual H3 generation. There is ZERO diffusers / ModularPipeline
/ torchao code here; all of that lives in Wan2GP's own `wgp.py`.

ONE SCENE = ONE IMAGE = ONE ANIMATION
---------------------------------------------------------------------------
Each scene is a single Z-Image still that H3 animates in ONE ref2va call,
driven by that scene's audio slice. No shot subdivision, no video_source
continuation. Scenes run h3_min_scene_seconds..h3_max_scene_seconds; a single ref2va call spans up to frames_maximum=737 frames
(~30.7s @ 24fps), so every scene fits one window. Every scene becomes one
task in a single JSON list sent to `wgp.py --process` in ONE subprocess call
(_parse_settings_json() accepts a plain JSON list of task dicts and runs them
as one queue, in order).

BEAT-DRIVEN CUTTING + ENERGY-AWARE CAMERA GRAMMAR
---------------------------------------------------------------------------
Scene cuts land on the music (downbeats/section boundaries) and cut RATE
follows energy -- quiet passages hold long, intense passages cut fast (see
analyze_music() / build_beat_aligned_boundaries()). Each scene is tagged with
an energy band, and write_scene_prompts() authors its camera move + action
from a band-specific vocabulary (CAMERA_GRAMMAR_BY_ENERGY): slow/stable when
quiet, kinetic when intense. Gated by CONFIG["beat_driven_cutting"].

TIMING / ALIGNMENT
---------------------------------------------------------------------------
Each scene's video_length (frames) is derived from the REAL span of song it
covers, snapped to H3's 17*k+5 frame grid with a running cursor so per-scene
rounding never accumulates. Clips concatenate gaplessly to just past the song
end (see build_scene_jobs()).

WGP.PY INTEGRATION -- FIELD NAMES CONFIRMED FROM SOURCE
---------------------------------------------------------------------------
Read directly out of wgp.py and models/minimax_h3/minimax_h3_handler.py:
  - `model_type`: "minimax_h3_ref2va" (or "minimax_h3_ref2va_pruned")
  - `video_prompt_type: "KI"` + `image_refs: [path]` -- enables the reference
    image (the "I"/"K" flag lives in video_prompt_type, NOT image_prompt_type,
    which is left empty; validate_settings() nulls image_refs otherwise).
  - `audio_prompt_type: "A"` + `audio_guide: path`  -- one audio reference.
  - `resolution: "WxH"`, `video_length` (frames), `seed`, `num_inference_steps`.
  - NO `negative_prompt` -- minimax_h3_handler.py sets "no_negative_prompt": True.
See `_wgp_task_from_scene()` -- the one function to edit if a future update
to your fork changes any of this.

Because `wgp.py`'s CLI queue mode doesn't expose a way to force an exact
output filename per task, `render_all_scenes_via_wgp()` matches finished
renders back to the right scene by task number and creation order.

Before a real run, set `CONFIG["wgp_dry_run"] = True` once and inspect the
printed command plus `audio_videos_dir/wgp_render/batch_manifest.json` --
that's exactly what gets sent to `wgp.py --process`.
"""

import gc
import json
import json_repair
import numpy as np
import os
import re
import random
import secrets
import shlex
import shutil
import subprocess
import tempfile
import time
import torch

from datetime import datetime
from diffusers import Krea2Pipeline
from diffusers import BitsAndBytesConfig as DiffusersBitsAndBytesConfig
from diffusers.quantizers import PipelineQuantizationConfig
from moviepy.audio.io.AudioFileClip import AudioFileClip
from openai import OpenAI
from transformers import BitsAndBytesConfig as TransformersBitsAndBytesConfig
from transformers import WhisperProcessor, WhisperForConditionalGeneration

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["XFORMERS_IGNORE_FLASH_VERSION_CHECK"] = "1"
_NVENC_SUPPORTED = None
_REALESRGAN_AVAILABLE = None
_REALESRGAN_IMPORT_ERROR = None

# -----------------------------------------------------------------------------
# Dependency check -- only what THIS process still does locally: Whisper,
# Z-Image, the LLM client, and audio I/O. No diffusers MiniMax H3 code lives
# here any more; that runs inside Wan2GP's own environment via `wgp.py`.
# -----------------------------------------------------------------------------
_MISSING = []
try:
    from huggingface_hub import hf_hub_download
except ImportError:
    _MISSING.append("huggingface_hub")
try:
    import soundfile as sf
except ImportError:
    _MISSING.append("soundfile")

if _MISSING:
    print("=" * 70)
    print("MISSING/INCOMPATIBLE DEPENDENCIES:")
    for m in _MISSING:
        print(f"  - {m}")
    print("\nInstall with:")
    print("  pip install  huggingface_hub soundfile")
    print("=" * 70)
    raise SystemExit(1)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16

retry_limit = 3

USE_GROK = False

IMAGE_THEME = "3D digital art, photorealistic"

# THEME is the creative directive every authoring prompt reads.
THEME = """
THE SONG IS THE SCREENPLAY.
Decode THIS song's lyrics into an amazing, creative short film before
decorating anything. Who is in this story, where are they, what happens to
them, and what changes by the last chorus? Invent a story a viewer would
want to rewatch: a clear protagonist, stakes, a turn, a payoff, and one or
two recurring images that come back transformed at the end.

SHOW WHAT THE LYRIC SAYS. When the lyric describes people doing things --
an angel pulling a stranger out of a burning car, a father teaching his
daughter to drive, two kids running from the rain -- SHOW THAT EVENT with
those characters, not a performer standing still singing about it.

THE PLACE IS THE LYRIC'S PLACE. A song about a park happens in a park. When
the lyric names Paris, we are in Paris, recognizably, in the mood of that
line (a rain-soaked Montmartre staircase for heartbreak, a sunlit Seine
quay for joy). Named cities, streets, rivers, bars, churches, highways and
eras are filmed literally. Invented settings fill the gaps the lyric leaves.

THE PERFORMERS PERFORM. Choruses, hooks and first-person emotional lines
belong to whoever is singing: they sing to camera and DANCE -- moves that
belong to this genre and this energy, from a slow sway to full
choreography. They may also walk through the story world, singing while
the story happens around them.

CAST FOLLOWS THE SONG: whoever and however many the lyrics call for --
any gender, any age, humans, angels, animals, machines -- each with a
locked look that stays consistent across the film.

HEAT FOLLOWS THE LYRIC. Each scene is exactly as sexual as its own lines:
not at all, romantic, sensual, or explicit. Sexual content is adults only.
"""


audio_filenames = ['''FrankensteinsAccordion.mp3''']
# Optional look for the performer(s) whose voice we hear
# (e.g. "a gravel-voiced man in his 50s, silver beard, denim jacket").
# Empty -> designed automatically from the detected voice + lyrics.
PROTAGONIST = "Dr. Frankenstein, a german mad scientist that loves his creations, however horrific they may be."

# Per-song creative direction, keyed by audio basename (no extension). Every
# field is optional and AUTHORITATIVE when set; every authoring stage reads it.
#   image_theme    -- rendering medium for this song (overrides IMAGE_THEME)
#   world          -- the places and era the film lives in
#   performer      -- look AND how they perform (overrides PROTAGONIST)
#   voice          -- male|female|duet|group|male_rap|female_rap|instrumental
#                     (overrides CLAP vocal detection)
#   director_notes -- what this video is visually about
#   must_show      -- set pieces the film must contain, each at least once
#   performance_target_ratio -- share of shots where the performer sings
#   lyrics         -- the real lyrics; Whisper lines are snapped to them
SONG_DIRECTION = {
    "FrankensteinsAccordion": {
        "image_theme": ("stylized 3D CGI animated feature film, gothic "
                        "stop-motion-inspired character design with exaggerated "
                        "proportions, sculpted clay and cloth textures, physically "
                        "based materials, volumetric lighting"),
        "world": ("A storm-lashed Gothic castle laboratory on a jagged Bavarian "
                  "crag: brass Tesla coils, bubbling glass retorts, workbenches "
                  "heaped with bellows leather, piano-key ivory, bones and copper "
                  "wire. Below it a crooked Alpine village of misty cobbled lanes "
                  "and shuttered windows, and a moonlit graveyard behind iron "
                  "gates. 19th century, Halloween night."),
        "performer": ("Dr. Frankenstein, a wiry German mad scientist in his 50s "
                      "with wild white hair, round brass goggles pushed up on his "
                      "forehead, a hooked nose and a gleeful manic grin, wearing a "
                      "stained leather apron over a burgundy waistcoat with rolled "
                      "shirt sleeves. He adores his creations, however horrific. He "
                      "sings while PLAYING the patchwork accordion he built; he is "
                      "never a singer fronting a band on a concert stage."),
        "voice": "",   # CLAP heard "duet" (51%); set "male" if it is one singer
        "director_notes": (
            "An animated horror-comedy. The story is the BUILD and what it "
            "unleashes: Dr. Frankenstein builds an accordion the way he built "
            "his monster, from scavenged parts, brings it to life with "
            "lightning, and its music raises the dead and sets every monster in "
            "the land dancing. Most shots are story: the workshop and the build "
            "in close detail, the accordion as a living character, ensembles of "
            "dancing monsters, and paranormal phenomena caused by the music. "
            "Dr. Frankenstein singing to camera is the minority. Spooky and "
            "playful, never gory. No concert stage, no microphones, no human "
            "rock band: any band is made of monsters drumming on bones and skulls."),
        "must_show": [
            "the workshop: wide establishing shots of the laboratory crowded with machines, jars and half-built instruments",
            "the build, step by step: blueprints, stitching bellows from patchwork leather, fitting bone and ivory keys, bolting on brass and electrodes",
            "the awakening: lightning strikes the rooftop rods and electrifies the accordion on the operating table; its bellows breathe like lungs",
            "the accordion as a living character: glowing eyes behind the grille, bellows that breathe, keys that ripple on their own",
            "Frankenstein's monster lumbering through misty village streets with the accordion",
            "monster ensemble dances: skeletons, ghosts, zombies, werewolves, witches and goblins dancing a polka together",
            "paranormal effects caused by the music: visible spectral notes, ectoplasm ribbons, ghosts pouring out of the bellows, witches riding spectral gales",
            "the dead rising from their graves in the moonlit graveyard to the tune",
            "frightened villagers hiding behind shutters, later won over and dancing",
            "a ghoulish monster band drumming on bones and tapping skulls",
            "finale: a Halloween feast and monster dance party where Frankenstein and his creations are celebrated",
        ],
        "performance_target_ratio": 0.2,
        # Paste the real lyrics here to fix Whisper mishearings
        # ("polka whales", "poker souls", "mencarp", ...).
        "lyrics": "",
    },
}


W1017 09:31:25.807000 95279 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1017 09:31:25.842000 95279 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


In [4]:

CONFIG = {
    "openai_api_key": os.environ.get("OPENAI_API_KEY", ""),
    "openai_model": "gpt-5.6-luna",
    "openai_model_large": "gpt-5.6-luna",
    "hf_token": os.environ.get("HF_TOKEN", ""),
    "grok_api_key": os.environ.get("XAI_API_KEY", ""),
    "grok_model": "grok-4.7",
    # Grok 4.7 always reasons (no "none"); low | medium | high (xAI default) | xhigh.
    "grok_reasoning_effort": "low",
    # Per-request LLM timeout (seconds). Grok responses are streamed, so this
    # bounds the silence between chunks, including the reasoning phase.
    "llm_timeout_seconds": 900,
    "base_working_dir": "./images",
    "base_video_dir": "./output",
    "audio_files": [
        f"/mnt/d/Share/Audio/{audio_filenames[0]}",
        #f"/mnt/d/Share/Audio/{audio_filenames[1]}",
        #f"/mnt/d/Share/Audio/{audio_filenames[2]}",
        #f"/mnt/d/Share/Audio/{audio_filenames[3]}",
        #f"/mnt/d/Share/Audio/{audio_filenames[4]}",
        #f"/mnt/d/Share/Audio/{audio_filenames[5]}",
        #f"/mnt/d/Share/Audio/{audio_filenames[6]}",
        #f"/mnt/d/Share/Audio/{audio_filenames[7]}",
    ],
    # ---- Scenes ----------------------------------------------------------
    # Highest heat any scene may reach (none|romantic|sensual|explicit). Each
    # scene is still capped by its OWN lyric lines; this is a global ceiling.
    "max_scene_heat": "explicit",
    # Rough share of scenes that show the performer(s) singing. The planner
    # decides per lyric; this is a target, not a quota.
    "performance_target_ratio": 0.35,
    "plan_batch_size": 16,
    "write_batch_size": 8,
    "audit_enabled": True,
    "audit_min_score": 3,          # 0-5; lower scenes are rewritten once
    "audit_batch_size": 12,
    # ---- Visual variety guards ---------------------------------------------
    # Every scene stands on one set piece from the film bible's menu; none is
    # used more than this many times or in two adjacent scenes.
    "set_piece_max_uses": 2,
    # Longest allowed run of back-to-back "performance" shots.
    "max_performance_run": 2,
    # Principal cast keys per frame; the rest become background extras so
    # the still prompt stays readable for the image model.
    "max_cast_in_frame": 4,
    # Abort authoring (before any stills render) when more than this share of
    # scenes fall back to template prompts -- the LLM is failing.
    "max_fallback_ratio": 0.25,
    # LLM pass that fixes Whisper mishearings (snaps to song_metadata lyrics
    # when given) before anything reads the lyrics.
    "transcript_cleanup": True,
    "krea_negative_prompt": "panels, black bars, low detail, blurry, low resolution, flat lighting, text, watermark, extra limbs, deformed hands",
    # CLAP vocal detection (male / female / duet / group / rap / none) tells
    # the film bible whose voice we hear. song_metadata "performer" overrides.
    "clap_vocal_detection": True,
    # ---- Genre / world grounding ----------------------------------------
    # Manual genre OVERRIDE. Leave "" so CLAP classifies the audio and, when
    # a runner-up is close, automatically uses a hybrid like "pop hip-hop".
    # A non-empty value here (or in song_metadata[basename].genre) skips CLAP.
    "genre": "",
    # Per-song metadata keyed by audio basename (no extension). "" = default.
    # genre left empty -> CLAP fills it (hybrid when warranted).
    "song_metadata": SONG_DIRECTION,
#    "song_metadata": {
#        "": {"genre": "", "world": "", "performer": ""},
#        "01_ItsCountry": {"genre": "country folk",
#                          "performer": "a man in his 40s, ..."},
#    },
    # CLAP zero-shot audio genre detection (fallback when metadata genre is
    # empty). Runs on a few 10s slices from the middle of the song.
    "clap_genre_detection": True,
    "clap_model_id": "laion/clap-htsat-unfused",
    "clap_num_slices": 3,
    "clap_slice_seconds": 10.0,
    "clap_min_confidence": 0.25,   # below this, still emit a hybrid if the runner-up is close
    # Hybrid genre: keep the runner-up when it is a real second voice, not noise.
    # Example: pop 42% + hip hop 20% -> "pop hip-hop" instead of just "pop".
    "clap_hybrid_enabled": True,
    "clap_hybrid_max_labels": 2,          # 2 = top + runner-up; 3 includes 3rd if close
    "clap_hybrid_min_second": 0.12,       # runner-up must have at least this probability
    "clap_hybrid_min_ratio": 0.35,        # runner-up / top (20/42 ≈ 0.48 -> include)
    "clap_hybrid_joiner": " ",            # "pop hip-hop"; use " / " for "pop / hip-hop"
    # ---------------------------------------------------------------------
    # Canvas / timing constants shared with the render step. These describe
    # WHAT to generate; the H3 model itself now lives entirely in Wan2GP.
    # ---------------------------------------------------------------------
    "h3_width": 1344,
    "h3_height": 768,
    "h3_fps": 24.0,
    # Krea-2-Turbo is TDM-distilled: official recipe is few steps and
    # guidance_scale=0.0. At guidance 0 the pipeline IGNORES negative_prompt.
    # Anti-clone work MUST live in the POSITIVE prompt + a post-gen
    # similarity retry (see generate_image). If you switch to krea/Krea-2-Raw,
    # set krea_guidance_scale to 4.5 and krea_num_inference_steps to 28 so
    # negatives actually fire.
    "krea_num_inference_steps": 12,
    "krea_guidance_scale": 0.0,
    "krea_complexity": 0.35,          # passed only if the pipeline accepts it
    "krea_anti_duplicate": True,
    "krea_clone_retry": 3,
    "krea_clone_similarity_max": 0.90,
    "krea_skip_existing": False,      # True -> keep stills already on disk

    # ---- Scene duration -------------------------------------------------
    # Scene lengths are clamped to [h3_min_scene_seconds, h3_max_scene_seconds]
    # by normalize_scene_durations(); build_scene_jobs() then snaps each to
    # H3's frame grid. Both bounds are chosen to sit inside what a single
    # ref2va call can render (frames_minimum 107 = 4.458s .. frames_maximum
    # 737 = 30.667s), so no scene is ever silently clamped shorter than planned.
    # 5.0s minimum: safely above the 4.458s render floor (a 5s request snaps
    # to 124 frames = 5.167s), so normalize never emits a sub-renderable scene.
    "h3_min_scene_seconds": 3.0,
    # 30.0s maximum: 30s x 24fps = 720 -> snaps to 719 frames (5 + 17*42) =
    # 29.958s, safely under the single-window cap of 736 frames. Staying under
    # one window means every clip renders in a single pass -- no sliding-window
    # stitching, no window-boundary seams, faster. Scenes longer than this are
    # split into consecutive scenes by normalize_scene_durations(), each with
    # its own reference image and animation.
    "h3_max_scene_seconds": 26.0,

    # ---- CGI movie magic ---------------------------------------------------
    "cgi_magic_enabled": True,
    "cgi_magic_probability": 0.78,
    "cgi_magic_peak_probability": 0.95,

    # ---- Upscale method ---------------------------------------------------
    # Choose ONE path. They are mutually exclusive:
    #   "none"       : leave H3 clips at native h3_width x h3_height.
    #   "realesrgan" : AFTER-generation 4x pixel restorer (current path).
    #   "h3_latent"  : DURING-generation MiniMax H3 two-phase latent upscale
    #                  via Wan2GP (LBH-123-AI/Minimax_h3_latent_Upscaler
    #                  3d_conv_v1, already shipped with Wan2GP).
    #                  Phase 1 denoises a small canvas, the 3D network
    #                  resizes [B,24,T,H,W] spatially, phase 2 refines at
    #                  the target resolution.
    # Legal h3_latent scales: 1.5, 2.0, 2.5, 3.0 (1.0 / "none" disables).
    # Stock Wan2GP drafts at target/2.0 (H3_TWO_PHASE_SCALE). This script
    # sets FINAL resolution to native * scale and exports
    # H3_TWO_PHASE_SCALE in the wgp.py env. If your fork reads that env
    # var in models/minimax_h3/pipeline.py:
    #   H3_TWO_PHASE_SCALE = float(os.getenv("H3_TWO_PHASE_SCALE", "2.0"))
    # the draft stays at h3_width x h3_height for any factor.
    "upscale_scene_clips": True,          # only used when backend=realesrgan
    "upscale_backend": "realesrgan",      # "none" | "realesrgan" | "h3_latent"
    "upscale_scale": 4,                   # realesrgan only (x4plus is 4x)
    "h3_latent_upscale_scale": 3.0,       # 1.0 | 1.5 | 2.0 | 2.5 | 3.0
    "h3_latent_upscale_tiling": True,    # True -> Wan2GP tiled phase-2
    "h3_latent_upscale_switch_threshold": 0.9035,
    "h3_latent_upscale_align": 32,
    "upscale_tile": 192,
    "upscale_tile_pad": 16,
    "upscale_use_half": True,
    "upscale_weights_dir": "./models/realesrgan",
    "upscale_output_suffix": "_4x",
    "upscale_codec": "h264_nvenc",
    "upscale_nvenc_preset": "p5",
    "upscale_nvenc_cq": 16,
    "upscale_audio_copy": True,
    "upscale_skip_existing": True,
    "final_video_codec": "h264_nvenc",
    "final_nvenc_preset": "p5",
    "final_nvenc_cq": 16,

    # Keep the individual per-scene clips (in scene_clips/) rather than
    # deleting them after the optional final concat.
    "h3_keep_scene_clips": True,
    # If False, skip the final ffmpeg concat entirely -- individual scene
    # clips under audio_videos_dir/scene_clips/ are still produced either
    # way; this just controls whether they also get stitched into one
    # final_<timestamp>.mp4. Set False if you only want the per-scene clips.
    "concat_final_video": False,

    # ---- Beat-driven cutting + energy-aware camera grammar ----------------
    # Scene cuts land on the music (downbeats/sections), and cut RATE follows
    # energy: quiet passages hold long, intense passages cut fast. Camera
    # MOTION is likewise chosen from an energy/emotion-specific vocabulary.
    "beat_driven_cutting": True,       # False -> fall back to lyric-gap scenes
    "music_beats_per_bar": 4,          # assumed time signature (4/4)
    "music_num_sections": 8,           # structural sections to detect
    "music_energy_hop_seconds": 0.1,   # energy-envelope resolution
    # Target scene length per energy band (seconds). The high value can't go
    # below h3_min_scene_seconds (the fastest cut the model can render), so
    # lower values are clamped up to it by build_beat_aligned_boundaries().
    "cut_target_low_seconds": 11.0,    # quiet: long, contemplative holds
    "cut_target_mid_seconds": 8.0,     # building: medium pacing
    "cut_target_high_seconds": 5.0,    # peak/drop: cut as fast as allowed (min scene)


    # ---------------------------------------------------------------------
    # Wan2GP (external render) settings -- THIS is the new integration
    # surface. See the module docstring above, and _wgp_task_from_scene()
    # below, before a real run.
    # ---------------------------------------------------------------------
    "wgp_repo_dir": os.path.expanduser("~/repos/Wan2GP"),
    # Wan2GP runs inside its own conda env; "python" here is resolved AFTER
    # `conda activate wgp_conda_env`, not this script's own interpreter.
    "wgp_python_executable": "python",
    "wgp_conda_env": "wan2gp",
    # Restored after the wgp.py call finishes (success or failure).
    "wgp_conda_base_env": "base",
    "wgp_script": "wgp.py",
    # TODO verify this matches the model key your fork registers for the
    # MiniMax H3 ref2va (omni-reference) checkpoint partition.
    "wgp_model_type": "minimax_h3_ref2va",   # or "minimax_h3_ref2va_pruned" (20B pruned ckpt)
    # Reference-image mode flag for ref2va, placed in video_prompt_type (NOT
    # image_prompt_type). "KI" = use reference image(s), with the first one
    # as the main subject that also defines output dimensions -- matches the
    # handler's image_ref_choices "letters_filter": "KI". Use plain "I" if
    # you don't want the reference image to drive output dimensions.
    "wgp_video_prompt_type": "KI",
    "wgp_num_inference_steps": 8,   # 8 if the Turbo LoRA is active, ~20 (model default) otherwise
    # H3 ref2va's real frame grid/limits, read directly out of
    # models/minimax_h3/minimax_h3_handler.py's query_model_def():
    # frames = frames_offset + k * frames_steps, clamped to
    # [frames_minimum, frames_maximum]. video_length is NOT limited by the
    # audio reference's length (video_length_not_limited_by_audio=True), so
    # one call really can cover a whole scene.
    "wgp_frames_offset": 5,
    "wgp_frames_steps": 17,
    "wgp_frames_minimum": 107,
    "wgp_frames_maximum": 737,   # ~30.7s @ 24fps -- SINGLE-WINDOW cap
    # Maximum TOTAL video_length (frames) the scene planner may request for a
    # single scene clip. Set equal to frames_maximum (737) so every clip
    # stays within ONE sliding window -- no multi-window stitching, no seams.
    # (ref2va CAN stitch windows for longer clips if you raise this above
    # frames_maximum, e.g. 855 for 35s, but that is slower and can show a
    # faint boundary seam; keeping it at 737 avoids that entirely.)
    "wgp_video_length_max_frames": 737,   # single window; 736f = 30.667s max
    # The AUDIO REFERENCE clip (not the rendered video) must be 2-15s per
    # WanGP's own validation for MiniMax H3 -- it's a tone/rhythm reference,
    # not the full soundtrack. (This is independent of the 5-35s SCENE length;
    # the reference is just the opening slice used for tone/rhythm.)
    "wgp_audio_ref_min_seconds": 2.0,
    "wgp_audio_ref_max_seconds": 15.0,
    "wgp_prompt_max_words": 350,

    # Extra flags appended verbatim to the wgp.py invocation. Confirmed
    # against shared/cli_args.py (fastflair fork) and mmgp 3.7.12's
    # offload.py profile() definitions (the exact mmgp version in the logs):
    #   --profile N : mmgp memory profile.
    #     1 HighRAM_HighVRAM : pin everything, no VRAM budget cap. Fastest,
    #                          wants ~48GB RAM + up to 24GB VRAM.
    #     2 HighRAM_LowVRAM  : pin ALL models in reserved RAM, cap VRAM
    #                          budget (~12GB working set). Fast + VRAM-safe
    #                          when you have lots of RAM. <-- best for a
    #                          24GB 4090 + 128GB RAM box.
    #     3 LowRAM_HighVRAM  : pin only the transformer, extra quantization,
    #                          no VRAM cap (wants a big-VRAM card).
    #     4 LowRAM_LowVRAM   : pin only the transformer, extra quantization,
    #                          AND cap VRAM. For RAM-starved machines; slower.
    #     5 VeryLowRAM_LowVRAM: no pinning at all, tiny transformer budget.
    #                          Slowest; only if even 4 won't fit.
    #   --perc-reserved-mem-max 0.5 : raise the reserved-RAM ceiling so mmgp
    #                          can FULLY pin the weights. The log warned it
    #                          fell back to partial pinning because it thought
    #                          only ~32GB was reservable; with 128GB RAM,
    #                          0.5 (=64GB) lets it pin the full ~63GB and
    #                          removes that fallback.
    #   --vram-safety-coefficient 0.8 : default; lower it (e.g. 0.7) only if
    #                          you still see VRAM pressure.
    # Tuned for a 4090 (24GB VRAM) + 128GB system RAM:
    "wgp_extra_cli_args": ["--profile", "2", "--perc-reserved-mem-max", "0.5"],
    # If True, no subprocess call is made -- the batch manifest is written
    # and the command printed only. Use this first to sanity-check field
    # names against an "Export Settings" JSON from your UI.
    "wgp_dry_run": False,
    "wgp_call_timeout_seconds": 3600,
    # Automatic retry passes for scenes that fail to render. Errors like
    # "CUDA driver error: device not ready" are transient GPU/driver hiccups
    # (common when VRAM is heavily oversubscribed, e.g. on WSL2) rather than
    # bad scene configs, so a failed scene usually succeeds on a fresh
    # attempt. Each retry only re-submits scenes still missing an output;
    # anything already rendered is skipped. Set wgp_max_render_attempts to 1
    # to disable retrying.
    "wgp_max_render_attempts": 3,
    "wgp_retry_sleep_seconds": 20,
    # Environment sanitization for the wgp.py subprocess. This script may
    # itself run inside a Jupyter/IPython kernel, which sets MPLBACKEND to
    # a notebook-only value ("module://matplotlib_inline.backend_inline")
    # that isn't a valid matplotlib backend outside a notebook -- and
    # wgp.py's own import chain (models/wan/scail/...) imports matplotlib
    # at startup, so that value leaking through crashes wgp.py before a
    # single task runs. Forced to a real headless backend here; add more
    # keys to either dict if your environment injects other vars that
    # don't belong in a plain subprocess.
    "wgp_env_overrides": {"MPLBACKEND": "Agg"},
    "wgp_env_unset": [],

    # ---- Turbo LoRA (best-effort; same caveat as before: published
    # against the fl2va checkpoint, ref2va compatibility unconfirmed) ----
    "wgp_use_turbo_lora": True,
    "wgp_turbo_lora_repo": "larryvrh/MiniMax-H3-Turbo-Lora",
    "wgp_turbo_lora_filename": "minimax_h3_turbo_4step_ema_ckpt850.safetensors",
    # Subfolder under wgp_repo_dir that Wan2GP scans for MiniMax H3 loras.
    # Confirmed from models/minimax_h3/minimax_h3_handler.py:
    # get_lora_dir() = os.path.join(lora_root, 'minimax_h3'), and
    # DEFAULT_LORA_ROOT in wgp.py is "loras" -- so this is "loras/minimax_h3",
    # NOT the bare "loras" folder. Override with --lora-dir-minimax-h3 on
    # your fork if you've relocated it.
    "wgp_lora_subdir": os.path.join("loras", "minimax_h3"),
    # Extra directories (relative to wgp_repo_dir) ensure_turbo_lora_in_wgp()
    # checks for an already-existing copy before downloading -- e.g. the
    # flat "loras" folder, if that's where you've placed loras directly.
    "wgp_lora_search_dirs": ["loras"],
    "wgp_lora_multiplier": "1.0",
    # ---- Lyric reading ----------------------------------------------------
    "lyric_fidelity_enabled": True,        # per-scene lyric decoding pass
    "lyric_interpret_batch_size": 24,
    "lyric_line_max_seconds": 8.0,         # cap on how long one transcript line is assumed to last
}

HEIGHT = CONFIG["h3_height"]
WIDTH = CONFIG["h3_width"]

os.makedirs(CONFIG["base_working_dir"], exist_ok=True)
os.makedirs(CONFIG["base_video_dir"], exist_ok=True)


# Ordered heat scale shared by the song classifier and per-scene lyric reads.
HEAT_LEVELS = ("none", "romantic", "sensual", "explicit")

CGI_MAGIC_DIRECTIVE = """
PREMIUM CGI MOVIE-MAGIC (in service of the story)
Use advanced 3D-CGI effects as storytelling tools with a visible cause and
response: the angel's light cutting through wreck smoke, rain that freezes
mid-air when two hands touch, a city skyline rebuilding itself as hope
returns, autumn leaves spiralling around a dancer on the downbeat. Effects
interact physically with characters, architecture, weather and light.
Vary the effect family; never repeat the same glow or particle cloud in
adjacent scenes. Quiet scenes get subtle physical magic; musical peaks may
become spectacular set pieces. Keep faces, hands and the main story action
readable. Grounded songs get grounded effects (weather, light, practical
fire and water); fantastical lyrics earn fantastical spectacle.
""".strip()

MAGIC_EFFECT_FAMILIES = [
    "volumetric light and god-rays cutting through dust, smoke, mist or rain",
    "weather as emotion: rain, snow, wind or fog that swells and breaks with the music",
    "particle trails and light-painting effects generated by a dancer's movement",
    "fluid simulation: water, ink or liquid light reacting causally to action",
    "embers, sparks, leaves or petals reacting to movement and the downbeat",
    "environmental transformation: streets bloom, ruins rebuild, landscapes shift with the story",
    "celestial phenomena: aurora, meteor trails, eclipses, impossible skies",
    "portals, mirrors or doorways revealing a memory or another layer of the world",
    "time effects: frozen moments, rewinding debris, slow-motion droplets",
    "scale illusion: tiny figures against colossal landscapes or architecture",
    "stylized smoke, clouds or fire briefly forming meaningful shapes",
    "subtle magical physics in hair, fabric, reflections and light",
]

# Per-energy camera grammar: quiet passages breathe with slow, stable moves;
# peaks explode with fast, kinetic ones. _camera_grammar_block() hands the
# right vocabulary to the scene writer.
CAMERA_GRAMMAR_BY_ENERGY = {
    "low": {
        "feel": "calm, intimate, contemplative -- the camera breathes",
        "moves": ["Static Shot", "Push In", "slow Pull Out", "gentle Tilt Up",
                  "slow Pan Left", "slow Pan Right", "Pedestal Up"],
        "modifiers": "with small amplitude at slow speed",
        "subject_action": ("a small, specific, emotionally loaded action -- a "
                           "glance, a hand reaching, a slow sway while singing, "
                           "a character pausing at a threshold; never a freeze"),
    },
    "mid": {
        "feel": "building, purposeful, flowing with the groove",
        "moves": ["Tracking Shot", "Arc Shot", "Truck Left", "Truck Right",
                  "Push In", "Pan Left", "Pan Right", "Tilt Down"],
        "modifiers": "with medium amplitude",
        "subject_action": ("rhythmic purposeful movement -- the performer grooving "
                           "and stepping on the backbeat, characters walking, "
                           "working, chasing, embracing; the story advances"),
    },
    "high": {
        "feel": "explosive, kinetic, urgent -- the peak of the song",
        "moves": ["Zoom In", "fast Push In", "Arc Shot", "Shake Slightly",
                  "Shake Strongly", "Roll Clockwise", "Roll Counterclockwise",
                  "fast Tracking Shot", "Whip Pan Left", "Whip Pan Right"],
        "modifiers": "with large amplitude at fast speed",
        "subject_action": ("a full-body event that ACCENTS the peak -- a "
                           "performer's big dance break, a rescue, a leap, a "
                           "crash, a run, a crowd erupting; hits on the downbeats"),
    },
}

SCENE_TYPES = ("performance", "story", "performance_in_story")

FILM_MANDATE = """
FILM MANDATE (hard constraints for this music video)
SCENE TYPES -- every scene is exactly one of:
- "performance": the performer(s) whose voice we hear sing THIS scene's
  lyric to camera and DANCE to the music in a performance setting that
  belongs to the song. Use for choruses, hooks, first-person feeling lines
  with no concrete event, and instrumental breaks that need energy.
- "story": the characters the lyric describes DO what the lyric describes.
  When a line tells of someone doing something (an angel rescuing someone
  from a car wreck, a couple dancing in a kitchen, a soldier coming home)
  SHOW that event. Nobody in a story scene sings or lip-syncs.
- "performance_in_story": the performer is physically inside the story
  world, singing while the story event happens around them.
PERFORMERS: match the voice we hear -- a solo voice is one performer, a duet
  is two, a group is a group. Each has a locked look; wardrobe may change
  between chapters, face and body never do. Performers TEND TO DANCE: groove,
  step, spin, sway or full choreography native to this genre, sized to the
  energy band, landing on the beat.
PERFORMANCE IS NOT A STAGE: unless the lyric puts the performer on a stage,
  they perform INSIDE the story world -- in its rooms, streets and set
  pieces -- playing their instrument, conducting or leading the others in
  the dance. A generic singer-on-a-stage shot is a failure.
CAST: everyone else comes from the lyrics. Gender, age, kind (human, angel,
  animal, machine) and how many people share a frame are whatever the
  lyric and the story need -- one lone figure, a couple, a family, a crowd.
  Each named character has a locked look repeated verbatim. Do not add
  characters the story does not need; do not leave out ones it does.
PLACE: settings are LITERAL whenever the lyric names or clearly implies one.
  A park lyric is in a park. "Paris" means recognizable Paris (landmarks,
  streets, rooftops, cafes, the Seine) in the light and weather that fit the
  mood of that line. Named cities, streets, buildings, vehicles, eras and
  weather are filmed as named. Stay in a place while the lyric stays there;
  move when the lyric moves. Invent specific, filmable places only where the
  lyric leaves a gap -- and make them belong to THIS song.
STORY CRAFT: build an amazing, creative short film -- a protagonist with
  something at stake, a turn, a payoff. Plant one or two recurring images
  early and pay them off at the end. Mix wide, medium and close shots.
  Flashbacks and dreams are allowed when they deepen the lyric.
HEAT follows each scene's OWN lines (scene_heat), never the song's
  reputation and never a quota:
  - "none": non-sexual; everyone dressed for the story.
  - "romantic": affection, longing, an embrace or a kiss.
  - "sensual": suggestive, intimate touch; partial undress is allowed.
  - "explicit": openly sexual content, staged inside the story with the
    people the lyric is about -- alone or together, any gender, consenting.
  Sexual content of any level involves adults only.
""".strip()

LYRIC_FIDELITY_RULES = """
LYRIC FIDELITY
- "literal": the lyric describes a concrete event, person, place or object.
  SHOW IT recognizably; a viewer hearing the line must connect it to the frame.
- "analogy": the lyric is figurative ("a heart of stone"). Show the human
  reality behind it, never a silly literal rendering.
- "atmospheric": a feeling with no specific image -- carry it through
  performance, light, weather and body language.
- "instrumental": nothing sung -- dance break, story montage, or a breath
  that bridges the lines before and after.
- Weather, time of day and place words are SETTING: if the lyric says it
  rains in Paris at night, it rains in Paris at night.
""".strip()

FORMAT_EXAMPLES = """
FORMAT EXAMPLES (copy STRUCTURE and SPECIFICITY, never these exact people):
scene_description: Wide shot, low angle. A woman angel with storm-grey wings, braided copper hair and a cracked silver breastplate kneels on rain-slick asphalt beside an overturned blue pickup truck, gripping the driver's door as it tears free in a shower of sparks; inside, a bearded man in a red flannel shirt reaches for her hand. Headlights rake through wreck smoke on a two-lane highway at night, pine forest on both shoulders. Lighting: cold rain, sodium streetlight, warm angel-light blooming from her wings.
motion: The angel wrenches the door away on the downbeat and pulls the man out as her wings flare, scattering rain into a ring of light. The camera arcs around them with large amplitude. Nobody sings.
scene_description: Medium-wide shot on a Montmartre staircase at blue hour, Sacre-Coeur glowing above. The singer -- a tall man in his thirties with a close-cropped beard, deep brown skin and a mustard corduroy jacket -- sings to camera while dancing down the wet steps, one hand on the iron rail, street lamps reflecting in the puddles.
motion: The singer sings the line to camera and dances down the steps, sliding his hand along the rail and spinning on the landing on the backbeat. Tracking shot with medium amplitude.
""".strip()


In [5]:

# =============================================================================
# Utility / memory
# =============================================================================
def reset_memory(dev):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(dev)
        torch.cuda.reset_accumulated_memory_stats(dev)
        torch.cuda.ipc_collect()


def _strip_code_fence(text: str) -> str:
    """Drop a ```json ... ``` wrapper around an LLM response, if present."""
    cleaned = (text or "").strip()
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        cleaned = "\n".join(lines)
    return cleaned


def safe_json_parse(response_text):
    """Robustly extract and parse a JSON list from an LLM response."""
    response_text = response_text or ""
    cleaned_text = _strip_code_fence(response_text)
    try:
        data = json_repair.loads(cleaned_text)
        if isinstance(data, list):
            if data and isinstance(data[0], list):
                data = [item for sublist in data for item in sublist]
            data = [d for d in data if isinstance(d, dict)]
            if data:
                return data
        elif isinstance(data, dict):
            for key in data:
                if isinstance(data[key], list):
                    inner = data[key]
                    if inner and isinstance(inner[0], list):
                        inner = [item for sublist in inner for item in sublist]
                    inner = [d for d in inner if isinstance(d, dict)]
                    if inner:
                        return inner
            return [data]
    except Exception as e:
        print(f"JSON Repair failed: {e}")
    try:
        match = re.search(r'\[.*\]', response_text, re.DOTALL)
        if match:
            data = json_repair.loads(match.group())
            if isinstance(data, list):
                if data and isinstance(data[0], list):
                    data = [item for sublist in data for item in sublist]
                data = [d for d in data if isinstance(d, dict)]
                return data
    except Exception:
        pass
    return []
  


# =============================================================================
# FILM MANDATE + LYRIC HEAT. classify_song_content() reads the whole lyric
# (summary, shape, overall heat) as context for the film bible; each scene's
# own heat comes from generate_lyric_interpretation() and is capped by
# gate_scene_heat(), so content is sexual only where the lines are.
# =============================================================================


def _film_mandate_block() -> str:
    return FILM_MANDATE + "\n\n" + LYRIC_FIDELITY_RULES


def _heat_rank(heat) -> int:
    h = str(heat or "").strip().lower()
    return HEAT_LEVELS.index(h) if h in HEAT_LEVELS else 0


def _norm_heat(heat, default: str = "none") -> str:
    h = str(heat or "").strip().lower()
    return h if h in HEAT_LEVELS else default


def classify_song_content(text: str, config: dict) -> dict:
    """One LLM read of the WHOLE lyric: what it is about, its shape, and
    its overall heat (context for the film bible; scenes are gated per line)."""
    print("Reading the whole lyric (subject, shape, heat)...")
    if not (text or "").strip():
        print("  [content] no sung lyrics -> heat none.")
        return {"song_heat": "none", "story_shape": "performance",
                "summary": "instrumental", "reason": "no lyrics transcribed"}
    prompt = f"""
You are a music supervisor briefing a music-video director. Read the lyrics
(Whisper transcript, may contain small errors): what is the song about, what
shape is it, and how sexual is the SONG AS A WHOLE?

song_heat, one of:
- "none": no romance or sex (story, faith, party, protest, nature, grief...).
- "romantic": love, longing, heartbreak, kissing, tenderness, devotion.
- "sensual": suggestive desire, seduction, innuendo, bodies -- but sex is
  not the song's main subject.
- "explicit": the song is substantially ABOUT sex: explicit sexual acts,
  graphic body description, or sustained unmistakable sexual innuendo as
  its central subject.
A love song with one suggestive line is "romantic" or "sensual", never
"explicit". Do not inflate: most songs are "none" or "romantic".

Also return:
- "story_shape": "narrative" (events happen to characters), "anthology"
  (many lives / vignettes), "performance" (mostly feeling, little event),
  or "symbolic".
- "summary": one sentence: what the song is about.
- "reason": one sentence justifying song_heat.

Lyrics:
{text[:9000]}

Return ONLY a raw JSON object with keys song_heat, story_shape, summary,
reason.
"""
    data = _llm_dict(prompt, temperature=0.2, system_message=(
        "You are a precise, unsentimental music supervisor. "
        "You return only valid JSON."))
    data["song_heat"] = _norm_heat(data.get("song_heat"), "none")
    shape = str(data.get("story_shape") or "").strip().lower()
    data["story_shape"] = shape if shape in (
        "narrative", "anthology", "performance", "symbolic") else "narrative"
    data["summary"] = str(data.get("summary") or "").strip()
    data["reason"] = str(data.get("reason") or "").strip()
    print(f"  [content] heat={data['song_heat']}, shape={data['story_shape']} "
          f"-- {data['summary'][:140]}")
    return data


def _lyric_fields(s: dict) -> dict:
    """The per-scene lyric reading, as saved to lyric_interpretation.json."""
    return {
        "lyrics": (s.get("text") or "").strip(),
        "has_lyrics": bool(s.get("has_lyrics", bool((s.get("text") or "").strip()))),
        "lyric_before": (s.get("lyric_before") or "")[:120],
        "lyric_after": (s.get("lyric_after") or "")[:120],
        "lyric_meaning": s.get("lyric_meaning", ""),
        "lyric_emotion": s.get("lyric_emotion", ""),
        "lyric_visual_cues": list(s.get("lyric_visual_cues") or []),
        "lyric_literal_traps": list(s.get("lyric_literal_traps") or []),
        "lyric_atmosphere": s.get("lyric_atmosphere", ""),
        "lyric_fidelity_mode": s.get("lyric_fidelity_mode", ""),
        "lyric_places": list(s.get("lyric_places") or []),
        "lyric_heat": s.get("lyric_heat", "none"),
    }
  
def restamp_scene_lyrics(scenes: list, raw_segments: list, duration: float,
                         config: dict = None) -> list:
    """
    Overwrite every scene's "text" with EXACTLY the transcript lines sung
    during its [start, next_start) span. Each transcript line is treated as a
    span from its own timestamp to the next line's timestamp (capped at
    lyric_line_max_seconds) and attributed to the scene it overlaps MOST, so a
    line that starts a hair before a cut lands with the scene it is mostly
    sung in. Also records "has_lyrics" and the neighbouring lines
    ("lyric_before" / "lyric_after") so instrumental scenes know what they
    are bridging.
 
    Call this AFTER normalize_scene_durations(): reanchoring copies a draft's
    whole text onto whatever cut interval it maps to (and the dedup step can
    swap in a different draft's text), and splitting copies one text onto
    every piece -- so before this call, "text" is often lyrics from a
    completely different part of the song.
    """
    if not scenes:
        return scenes
    cfg = config or {}
    max_line = float(cfg.get("lyric_line_max_seconds", 8.0))
    segs = sorted((float(ts), str(txt).strip()) for ts, txt in (raw_segments or [])
                  if str(txt).strip())
    bounds = []
    for i, s in enumerate(scenes):
        start = float(s.get("start", 0.0))
        end = float(scenes[i + 1].get("start", duration)) if i + 1 < len(scenes) else float(duration)
        bounds.append((start, max(start, end)))
        s["text"] = ""
        s["_lyric_lines"] = []
 
    for k, (ts, txt) in enumerate(segs):
        nxt = segs[k + 1][0] if k + 1 < len(segs) else float(duration)
        te = min(nxt, ts + max_line, float(duration))
        best, best_ov = None, 0.0
        for i, (a, b) in enumerate(bounds):
            ov = min(te, b) - max(ts, a)
            if ov > best_ov:
                best, best_ov = i, ov
        if best is None:  # zero-length line: attribute by start time
            best = max((i for i, (a, b) in enumerate(bounds) if a <= ts), default=0)
        scenes[best]["_lyric_lines"].append((ts, txt))
 
    for i, s in enumerate(scenes):
        lines = [t for _, t in s.pop("_lyric_lines")]
        s["text"] = " ".join(lines).strip()
        s["has_lyrics"] = bool(lines)
        a, b = bounds[i]
        before = [t for ts, t in segs if ts < a]
        after = [t for ts, t in segs if ts >= b]
        s["lyric_before"] = before[-1] if before else ""
        s["lyric_after"] = after[0] if after else ""
 
    n_lyric = sum(1 for s in scenes if s["has_lyrics"])
    print(f"  [lyrics] restamped {len(scenes)} scene(s): {n_lyric} with sung "
          f"lyrics, {len(scenes) - n_lyric} instrumental.")
    return scenes
 
def generate_lyric_interpretation(scenes: list, full_lyrics: str,
                                  config: dict) -> None:
    """
    ONE decoding pass (batched) that turns each scene's raw lyric lines into
    something a director can shoot: what the line MEANS (analogy decoded),
    the emotion, 2-5 concrete visual cues that ARE that meaning inside the
    film's world, the words that must not be rendered literally, any
    weather/time/setting and named places, and how sexual THESE lines are
    (lyric_heat). Mutates scenes in place.
 
    Instrumental scenes get a bridging meaning derived from the lines before
    and after them, so intros, solos and outros still serve the story.
    """
    if not scenes or not config.get("lyric_fidelity_enabled", True):
        return
    print(f"Decoding lyrics per scene ({len(scenes)} scenes)...")
    batch_size = max(4, int(config.get("lyric_interpret_batch_size", 24)))
 
    def _defaults(s):
        s.setdefault("lyric_meaning", (s.get("text") or "").strip())
        s.setdefault("lyric_emotion", s.get("emotion", ""))
        s.setdefault("lyric_visual_cues", [])
        s.setdefault("lyric_literal_traps", [])
        s.setdefault("lyric_atmosphere", "")
        s.setdefault("lyric_fidelity_mode",
                     "atmospheric" if s.get("has_lyrics") else "instrumental")
        s.setdefault("lyric_places", [])
        s.setdefault("lyric_heat", "none")
 
    # Constant across batches: sent as its own message so it is cached.
    context = f"""
You are a lyric interpreter and story editor for a music video. For every
scene below you are given the EXACT lines sung during that scene (empty
"lyrics" = instrumental passage) plus the lines just before and after.
Prefer LITERAL story staging when the lyric names or clearly implies a
concrete event, place, presence, or object. Invent those proofs from the
words themselves. Only use analogy mode when the phrase is clearly
figurative and would look silly filmed word-for-word. Do not flatten a
specific lyric into a generic location just because the musical genre
usually lives there. Show the people and places the lyric names.

### {_brief_block()}
Theme: {THEME}

### {_film_mandate_block()}
Full lyrics for context: {full_lyrics[:6000]}
 
For EACH scene decode what is really being said so a director can film it:
- "lyric_meaning": 1-2 plain sentences: what the singer is actually
  saying or feeling here. Decode analogies and idioms ("whiskey river" ->
  a man drowning his grief in whiskey, glass after glass; "dust on the
  Bible" -> a faith abandoned so long the book is never opened).
- "lyric_emotion": 1-3 words.
- "lyric_visual_cues": 2-5 CONCRETE, filmable images that ARE this
  meaning inside the world above (people, objects, gestures, places).
  These must be recognisable to someone hearing the line.
- "lyric_literal_traps": words/phrases that must NOT be rendered
  literally (e.g. "no actual river of whiskey"). [] if none.
- "lyric_atmosphere": weather / time of day / physical setting the lyric
  literally states or strongly implies (rain, night, winter, a church, a
  kitchen at dawn) -- these SHOULD be rendered literally. "" if none.
- "lyric_fidelity_mode": one of
    "literal"      - the line describes a concrete event/image; show it.
    "analogy"      - figurative; show the inferred human reality.
    "atmospheric"  - a feeling with no specific image; carry the mood.
    "instrumental" - nothing sung; bridge lyric_before -> lyric_after.
- "lyric_places": every place the line names or clearly implies, as short
  filmable phrases ("Paris", "the park", "a hospital corridor",
  "Highway 61"). [] if none.
- "lyric_heat": how sexual THESE lines are -- "none", "romantic" (love,
  longing, a kiss), "sensual" (suggestive desire), "explicit" (sexual acts).
  Rate only these lines; do not inherit heat from the rest of the song.
For instrumental scenes, derive lyric_meaning/emotion/cues from the
surrounding lines (what the music is holding, anticipating, or releasing);
their lyric_places is [] and lyric_heat is "none".
 
Return ONLY a raw JSON array, one object per scene, in order, each with:
"index" (copied), "lyric_meaning", "lyric_emotion", "lyric_visual_cues",
"lyric_literal_traps", "lyric_atmosphere", "lyric_fidelity_mode",
"lyric_places", "lyric_heat".
 """

    for start in range(0, len(scenes), batch_size):
        batch = scenes[start:start + batch_size]
        rows = [{
            "index": start + k,
            "start_sec": round(float(s.get("start", 0.0)), 1),
            "lyrics": (s.get("text") or "").strip(),
            "lyric_before": (s.get("lyric_before") or "")[:160],
            "lyric_after": (s.get("lyric_after") or "")[:160],
        } for k, s in enumerate(batch)]
 
        prompt = f"""
Scenes:
{json.dumps(rows, indent=2, ensure_ascii=False)}
"""
        got = _by_index(_llm_items(prompt, temperature=0.5, use_large=True,
                                   context=context))
        for k, s in enumerate(batch):
            it = got.get(start + k)
            if it:
                s["lyric_meaning"] = str(it.get("lyric_meaning") or "").strip()
                s["lyric_emotion"] = str(it.get("lyric_emotion") or "").strip()
                s["lyric_visual_cues"] = [str(x).strip() for x in
                                          (it.get("lyric_visual_cues") or []) if str(x).strip()][:5]
                s["lyric_literal_traps"] = [str(x).strip() for x in
                                            (it.get("lyric_literal_traps") or []) if str(x).strip()][:5]
                s["lyric_atmosphere"] = str(it.get("lyric_atmosphere") or "").strip()
                mode = str(it.get("lyric_fidelity_mode") or "").strip().lower()
                if mode not in ("literal", "analogy", "atmospheric", "instrumental"):
                    mode = "instrumental" if not s.get("has_lyrics") else "atmospheric"
                if not s.get("has_lyrics"):
                    mode = "instrumental"
                s["lyric_fidelity_mode"] = mode
                places = it.get("lyric_places") or []
                if isinstance(places, str):
                    places = [places]
                s["lyric_places"] = ([str(x).strip() for x in places if str(x).strip()][:4]
                                     if s.get("has_lyrics") else [])
                s["lyric_heat"] = (_norm_heat(it.get("lyric_heat"), "none")
                                   if s.get("has_lyrics") else "none")
            _defaults(s)
 
    # A decoded lyric emotion is a better scene emotion than the draft's
    # guess, which was made against a different slice of the song.
    for s in scenes:
        if s.get("has_lyrics") and s.get("lyric_emotion"):
            s["emotion"] = s["lyric_emotion"]
    modes = [s.get("lyric_fidelity_mode") for s in scenes]
    print(f"  [lyrics] modes: literal={modes.count('literal')} "
          f"analogy={modes.count('analogy')} atmospheric={modes.count('atmospheric')} "
          f"instrumental={modes.count('instrumental')}")
  

# =============================================================================
# LLM helpers
# =============================================================================
def _llm_items(prompt: str, temperature: float = 0.3, use_large: bool = False,
               system_message: str = None, context: str = None) -> list:
    """One LLM call parsed as a JSON list of dicts ([] on any failure).
    `context` is the part of a batched prompt that never changes between
    batches; it goes in its own message so the provider can cache it."""
    try:
        items = safe_json_parse(get_openai_prompt_response(
            prompt, temperature=temperature, use_grok=USE_GROK,
            use_large=use_large, context=context,
            system_message=system_message or H3_CINEMATOGRAPHER_SYSTEM_MESSAGE))
    except Exception as e:
        print(f"  [llm] call failed ({e})")
        return []
    return [it for it in (items or []) if isinstance(it, dict)]


def _llm_dict(prompt: str, temperature: float = 0.3, use_large: bool = False,
              system_message: str = None, context: str = None) -> dict:
    """One LLM call parsed as a single JSON object ({} on any failure)."""
    try:
        return safe_json_dict(get_openai_prompt_response(
            prompt, temperature=temperature, use_grok=USE_GROK,
            use_large=use_large, context=context,
            system_message=system_message or H3_CINEMATOGRAPHER_SYSTEM_MESSAGE))
    except Exception as e:
        print(f"  [llm] call failed ({e})")
        return {}


def _by_index(items: list) -> dict:
    """{index: item} for LLM rows that carry an integer "index"."""
    return {it["index"]: it for it in items or []
            if isinstance(it, dict) and isinstance(it.get("index"), int)}


def safe_json_dict(response_text: str) -> dict:
    """
    Robustly extract and parse a single JSON OBJECT (dict) from an LLM
    response. Kept separate from safe_json_parse() because the style/entity
    bibles contain list values (emotional_arc) that the list-parser would
    incorrectly unwrap or reject.
    """
    response_text = response_text or ""
    cleaned_text = _strip_code_fence(response_text)
    try:
        data = json_repair.loads(cleaned_text)
        if isinstance(data, dict):
            return data
        if isinstance(data, list) and data and isinstance(data[0], dict):
            return data[0]
    except Exception as e:
        print(f"JSON dict repair failed: {e}")
    try:
        match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if match:
            d = json_repair.loads(match.group())
            if isinstance(d, dict):
                return d
    except Exception:
        pass
    return {}


def _truncate_prompt(prompt: str, max_words: int = 350) -> str:
    """Truncate prompt to stay under the text-encoder's token budget."""
    words = prompt.split()
    if len(words) > max_words:
        truncated = " ".join(words[:max_words])
        print(f"  [prompt] Truncated from {len(words)} to {max_words} words.")
        return truncated
    return prompt


# =============================================================================
# Audio helpers
# =============================================================================
def load_audio_mono_16k(audio_file: str) -> tuple:
    """Mono 16kHz decode, used only for Whisper transcription."""
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp_path = tmp.name
    try:
        subprocess.run(
            ["ffmpeg", "-y", "-i", audio_file,
             "-ar", "16000", "-ac", "1", "-f", "wav", tmp_path],
            check=True, capture_output=True,
        )
        audio, sr = sf.read(tmp_path, dtype="float32", always_2d=False)
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)
    return audio, sr


def load_full_audio(audio_file: str) -> tuple:
    """
    Decode the ENTIRE song once, at its native sample rate, into memory as
    (channels, samples) float32 -- used to slice per-shot audio references
    without re-reading the file from disk on every one of the (many) shot
    slices across a full song.
    """
    audio, sr = sf.read(audio_file, dtype="float32", always_2d=True)
    audio = audio.T  # (samples, channels) -> (channels, samples)
    return audio, sr


def slice_audio_tensor(full_audio: np.ndarray, sr: int, start_sec: float,
                       duration_sec: float) -> tuple:
    """Slice (channels, samples) float32 audio already in memory. Returns
    (chunk, actual_duration_sec) -- the chunk may be shorter than requested
    near the end of the song, which the caller should account for."""
    n_ch, n_samp = full_audio.shape
    start = min(int(round(start_sec * sr)), n_samp)
    end = min(int(round((start_sec + duration_sec) * sr)), n_samp)
    chunk = full_audio[:, start:end]
    return chunk, chunk.shape[1] / sr


def get_audio_duration(audio_file: str) -> float:
    with AudioFileClip(audio_file) as clip:
        return clip.duration


# =============================================================================
# Music analysis -- the foundation of beat-driven cutting and energy-aware
# camera grammar. Extracts, from the actual waveform:
#   - tempo (BPM) and beat times
#   - downbeat times (bar starts -- the strongest cut points)
#   - section boundaries (structural changes: verse->chorus->bridge, where a
#     film would naturally change scene)
#   - a normalized energy envelope (0..1) sampled on a fixed grid, so we can
#     ask "how intense is the song at time t" and let cut RATE and camera
#     MOTION follow the music instead of the lyrics.
# Uses librosa when available (full-quality analysis); falls back to a pure
# numpy RMS/flux estimator so the feature still works in environments without
# it. Everything returned is in SECONDS on the song timeline.
# =============================================================================
def camera_grammar_for(band: str) -> dict:
    return CAMERA_GRAMMAR_BY_ENERGY.get(band, CAMERA_GRAMMAR_BY_ENERGY["mid"])


def _camera_grammar_block(band: str) -> str:
    """Render one energy band's camera vocabulary as an instruction block for
    the motion-authoring LLM call."""
    g = camera_grammar_for(band)
    moves = ", ".join(g["moves"])
    return (f"CAMERA REGISTER for this scene (energy = {band.upper()}: "
            f"{g['feel']}):\n"
            f"- Choose ONE dominant camera move from: {moves}.\n"
            f"- Default modifiers unless the moment calls otherwise: {g['modifiers']}.\n"
            f"- Subject action should be: {g['subject_action']}.\n"
            f"- Beat-sync: name which movement lands on the pulse or rides "
            f"the swell (a dance step, a footfall, a door slam, a turn of the "
            f"head). Show the emotion changing across the take.")

def _analyze_music_librosa(audio_file: str, config: dict) -> dict:
    import librosa
    y, sr = librosa.load(audio_file, sr=None, mono=True)
    duration = len(y) / sr

    # Tempo + beat frames.
    tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr, trim=False)
    beat_times = librosa.frames_to_time(beat_frames, sr=sr).tolist()
    tempo = float(np.atleast_1d(tempo)[0])

    # Downbeats: librosa has no bar tracker, so infer a 4/4 grid by taking
    # every 4th beat, phase-aligned to the strongest onset among the first
    # few beats (a decent, dependency-free approximation of bar starts).
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    onset_times_all = librosa.times_like(onset_env, sr=sr)
    beats_per_bar = int(config.get("music_beats_per_bar", 4))
    downbeats = []
    if beat_times:
        # pick the phase (0..beats_per_bar-1) whose beats carry the most onset energy
        def onset_at(t):
            idx = int(np.argmin(np.abs(onset_times_all - t)))
            return onset_env[idx]
        best_phase, best_score = 0, -1.0
        for phase in range(beats_per_bar):
            score = sum(onset_at(beat_times[i]) for i in range(phase, len(beat_times), beats_per_bar))
            if score > best_score:
                best_phase, best_score = phase, score
        downbeats = [beat_times[i] for i in range(best_phase, len(beat_times), beats_per_bar)]

    # Structural section boundaries via a self-similarity/agglomerative split
    # on chroma+MFCC -- these are the "a new part of the song starts here"
    # moments a director would cut on.
    try:
        chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        feat = np.vstack([librosa.util.normalize(chroma, axis=1),
                          librosa.util.normalize(mfcc, axis=1)])
        n_sections = int(config.get("music_num_sections", 8))
        bounds = librosa.segment.agglomerative(feat, n_sections)
        section_times = sorted(set(librosa.frames_to_time(bounds, sr=sr).tolist()))
        section_times = [t for t in section_times if 0.0 < t < duration]
    except Exception as e:
        print(f"  [music] section detection failed ({e}); using downbeats only.")
        section_times = []

    # Energy envelope on a fixed hop grid, RMS in dB -> normalized 0..1.
    hop_sec = float(config.get("music_energy_hop_seconds", 0.1))
    hop = max(1, int(sr * hop_sec))
    rms = librosa.feature.rms(y=y, frame_length=hop * 2, hop_length=hop)[0]
    env_db = librosa.amplitude_to_db(rms + 1e-6)
    env = (env_db - env_db.min()) / (env_db.max() - env_db.min() + 1e-9)
    env_times = (np.arange(len(env)) * hop_sec).tolist()

    return {
        "backend": "librosa",
        "duration": duration,
        "tempo": tempo,
        "beat_times": beat_times,
        "downbeat_times": downbeats,
        "section_times": section_times,
        "energy_times": env_times,
        "energy": env.tolist(),
        "beats_per_bar": beats_per_bar,
    }


def _analyze_music_numpy(audio_file: str, config: dict) -> dict:
    """Dependency-light fallback: RMS energy envelope + spectral-flux onset
    peaks, with a tempo/beat estimate from onset autocorrelation. Coarser
    than librosa but good enough to drive cutting when librosa is absent."""
    full_audio, sr = load_full_audio(audio_file)   # (channels, samples)
    mono = full_audio.mean(axis=0)
    duration = len(mono) / sr

    hop_sec = float(config.get("music_energy_hop_seconds", 0.1))
    hop = max(1, int(sr * hop_sec))
    n = len(mono) // hop
    if n < 2:
        return {"backend": "numpy", "duration": duration, "tempo": 0.0,
                "beat_times": [], "downbeat_times": [], "section_times": [],
                "energy_times": [0.0], "energy": [0.0],
                "beats_per_bar": int(config.get("music_beats_per_bar", 4))}

    rms = np.array([np.sqrt(np.mean(mono[i*hop:(i+1)*hop]**2) + 1e-9) for i in range(n)])
    times = np.arange(n) * hop_sec
    env = (rms - rms.min()) / (rms.max() - rms.min() + 1e-9)

    # onset strength = positive change in energy (crude spectral-flux proxy)
    flux = np.maximum(0.0, np.diff(rms, prepend=rms[0]))
    fl = (flux - flux.min()) / (flux.max() - flux.min() + 1e-9)
    thr = float(fl.mean() + 0.8 * fl.std())
    onset_times = [float(times[i]) for i in range(1, len(fl) - 1)
                   if fl[i] > thr and fl[i] >= fl[i-1] and fl[i] >= fl[i+1]]

    # tempo via autocorrelation of the onset envelope, searched 60-180 BPM
    tempo, beat_times, downbeats = 0.0, [], []
    if len(onset_times) >= 4:
        ac = np.correlate(fl - fl.mean(), fl - fl.mean(), mode="full")[len(fl)-1:]
        lags = np.arange(len(ac)) * hop_sec
        lo, hi = 60.0 / 180.0, 60.0 / 60.0  # seconds per beat for 180..60 BPM
        mask = (lags >= lo) & (lags <= hi)
        if mask.any():
            best_lag = lags[mask][int(np.argmax(ac[mask]))]
            if best_lag > 0:
                tempo = 60.0 / best_lag
                # phase-align beat grid to the first strong onset
                phase = onset_times[0] if onset_times else 0.0
                t = phase
                while t < duration:
                    beat_times.append(round(t, 3))
                    t += best_lag
                bpb = int(config.get("music_beats_per_bar", 4))
                downbeats = beat_times[::bpb]

    # sections: mark the largest sustained energy shifts as boundaries
    section_times = []
    if n > 20:
        win = max(5, int(2.0 / hop_sec))  # ~2s smoothing
        smooth = np.convolve(env, np.ones(win)/win, mode="same")
        d = np.abs(np.diff(smooth, prepend=smooth[0]))
        n_sections = int(config.get("music_num_sections", 8))
        if d.max() > 0:
            # take the top (n_sections-1) most separated change peaks
            order = np.argsort(d)[::-1]
            picked = []
            for idx in order:
                t = float(times[idx])
                if all(abs(t - p) > 8.0 for p in picked):  # >=8s apart
                    picked.append(t)
                if len(picked) >= n_sections - 1:
                    break
            section_times = sorted(picked)

    return {
        "backend": "numpy",
        "duration": duration,
        "tempo": float(tempo),
        "beat_times": beat_times,
        "downbeat_times": downbeats,
        "section_times": section_times,
        "energy_times": times.tolist(),
        "energy": env.tolist(),
        "beats_per_bar": int(config.get("music_beats_per_bar", 4)),
    }

def analyze_music(audio_file: str, config: dict) -> dict:
    """Analyze the song's rhythm/structure/energy. Prefers librosa; falls
    back to a numpy estimator. Result is cached to disk next to the images."""
    print("Analyzing music (tempo, beats, sections, energy)...")
    try:
        # _analyze_music_librosa imports librosa at its top; if librosa is
        # absent that raises ImportError, which we catch to fall back.
        analysis = _analyze_music_librosa(audio_file, config)
    except Exception as e:
        print(f"  [music] librosa unavailable or failed ({e}); "
              f"using the numpy fallback analyzer.")
        analysis = _analyze_music_numpy(audio_file, config)
    print(f"  [music] backend={analysis['backend']} tempo={analysis['tempo']:.1f} BPM, "
          f"{len(analysis['beat_times'])} beats, "
          f"{len(analysis['downbeat_times'])} downbeats, "
          f"{len(analysis['section_times'])} section boundaries.")
    return analysis

def energy_at(analysis: dict, t: float) -> float:
    """Normalized energy (0..1) of the song at time t seconds."""
    times = analysis.get("energy_times") or []
    energy = analysis.get("energy") or []
    if not times or not energy:
        return 0.5
    idx = int(np.searchsorted(times, t))
    idx = max(0, min(len(energy) - 1, idx))
    return float(energy[idx])

# =============================================================================
# Genre grounding: per-song metadata, CLAP audio classification (genre and
# lead voice), and the Creative Brief (GENRE, WORLD, METAPHOR POLICY) that
# every authoring prompt reads. Lyrics alone mislead: a country song about
# software reads as sci-fi on the page; the genre lives in the AUDIO.
# =============================================================================
CLAP_GENRE_LABELS = [
    "country", "folk acoustic", "rock", "pop", "hip hop",
    "electronic dance", "R&B and soul", "heavy metal", "jazz", "blues",
    "classical orchestral", "reggae", "latin",
]

# Module-level current brief, set once per song by author_scenes().
# _brief_block() reads it, so no downstream function signature has to change.
CURRENT_BRIEF = {}


def get_song_metadata(audio_file: str, config: dict) -> dict:
    """Producer-supplied genre/world/protagonist for this song, merged over
    the "" defaults. Authoritative when present; CLAP only fills genre when
    both CONFIG['genre'] and song_metadata genre are empty."""
    base = os.path.splitext(os.path.basename(audio_file))[0]
    meta = dict(config.get("song_metadata", {}).get("", {}))
    meta.update({k: v for k, v in
                 config.get("song_metadata", {}).get(base, {}).items() if v})
    # Top-level CONFIG["genre"] overrides per-song metadata when set.
    cfg_genre = str(config.get("genre") or "").strip()
    if cfg_genre:
        meta["genre"] = cfg_genre
        meta["genre_source"] = "config"
    elif str(meta.get("genre") or "").strip():
        meta["genre"] = str(meta["genre"]).strip()
        meta["genre_source"] = "song_metadata"
    else:
        meta["genre"] = ""
        meta["genre_source"] = ""
    return meta


def resolve_song_genre(audio_file: str, config: dict, meta: dict = None) -> dict:
    """
    Genre resolution, one place:
      1. CONFIG['genre'] or song_metadata genre, if set, WINS. CLAP is skipped.
      2. Otherwise CLAP classifies the audio and may return a hybrid
         ('pop hip-hop') when the runner-up is close.
    Mutates and returns meta with genre + genre_source filled in.
    """
    meta = dict(meta or get_song_metadata(audio_file, config))
    if str(meta.get("genre") or "").strip():
        print(f"[genre] using override from {meta.get('genre_source') or 'config'}: "
              f"'{meta['genre']}' (CLAP skipped)")
        return meta
    if not config.get("clap_genre_detection", True):
        print("[genre] no override and CLAP disabled; brief will infer from lyrics.")
        return meta
    detected = detect_genre_clap(audio_file, config)
    if detected:
        meta["genre"] = detected
        meta["genre_source"] = "clap"
    return meta


def _format_clap_genre_label(label: str) -> str:
    """Pretty-print a CLAP label for use as a style string."""
    raw = (label or "").strip()
    aliases = {
        "hip hop": "hip-hop",
        "r&b and soul": "R&B",
        "electronic dance": "electronic",
        "folk acoustic": "folk",
        "classical orchestral": "classical",
        "heavy metal": "metal",
    }
    return aliases.get(raw.lower(), raw)


def _compose_hybrid_genre(ranked: list, config: dict) -> str:
    """
    ranked: [(label, prob), ...] already sorted high -> low.
    Returns a single genre string: "pop", "pop hip-hop", or
    "pop (possibly hip-hop)" when even the top score is weak.
    """
    if not ranked:
        return ""
    hybrid_on = bool(config.get("clap_hybrid_enabled", True))
    max_n = max(1, int(config.get("clap_hybrid_max_labels", 2)))
    min_second = float(config.get("clap_hybrid_min_second", 0.12))
    min_ratio = float(config.get("clap_hybrid_min_ratio", 0.35))
    min_conf = float(config.get("clap_min_confidence", 0.25))
    joiner = str(config.get("clap_hybrid_joiner", " "))

    chosen = [ranked[0]]
    if hybrid_on:
        top_p = float(ranked[0][1])
        for label, prob in ranked[1:]:
            if len(chosen) >= max_n:
                break
            p = float(prob)
            if p < min_second:
                break
            if top_p > 0 and (p / top_p) < min_ratio:
                break
            chosen.append((label, p))

    names = [_format_clap_genre_label(lbl) for lbl, _ in chosen]
    # Dedup after aliasing ("R&B and soul" / "R&B")
    seen = set()
    pretty = []
    for name in names:
        key = name.lower()
        if key in seen:
            continue
        seen.add(key)
        pretty.append(name)
    genre = joiner.join(pretty) if pretty else _format_clap_genre_label(ranked[0][0])

    # Weak top score and no hybrid partner: keep the old hedge so the brief
    # LLM does not over-commit to a shaky single label.
    if (not hybrid_on or len(pretty) < 2) and float(ranked[0][1]) < min_conf and len(ranked) > 1:
        genre = (f"{_format_clap_genre_label(ranked[0][0])} "
                 f"(possibly {_format_clap_genre_label(ranked[1][0])})")
    return genre


def _clap_zero_shot(audio_file: str, text_labels: list, config: dict):
    """Average CLAP zero-shot probabilities over a few slices from the
    25%-75% span of the song (skipping intros/outros). Returns a numpy array
    aligned with text_labels, or None when the song is too short. The model
    is loaded, used, and freed inside this call (~600MB). Raises on failure."""
    from transformers import ClapModel, ClapProcessor
    n_slices = max(1, int(config.get("clap_num_slices", 3)))
    slice_sec = float(config.get("clap_slice_seconds", 10.0))
    model_id = config.get("clap_model_id", "laion/clap-htsat-unfused")
    full, sr = load_full_audio(audio_file)
    mono = full.mean(axis=0).astype(np.float32)
    duration = len(mono) / sr

    # CLAP expects 48kHz mono.
    target_sr = 48000
    if sr != target_sr:
        try:
            import librosa
            mono = librosa.resample(mono, orig_sr=sr, target_sr=target_sr)
        except Exception:
            idx = np.linspace(0, len(mono) - 1,
                              int(len(mono) * target_sr / sr))
            mono = np.interp(idx, np.arange(len(mono)), mono).astype(np.float32)
        sr = target_sr

    slices = []
    for frac in np.linspace(0.25, 0.75, n_slices):
        start = max(0.0, frac * duration - slice_sec / 2.0)
        chunk = mono[int(start * sr):int((start + slice_sec) * sr)]
        if len(chunk) >= sr:  # at least 1s of audio
            slices.append(chunk)
    if not slices:
        return None

    processor = ClapProcessor.from_pretrained(model_id)
    model = ClapModel.from_pretrained(model_id)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(dev).eval()
    try:
        with torch.no_grad():
            inputs = processor(text=list(text_labels), audio=slices,
                               sampling_rate=sr, return_tensors="pt",
                               padding=True)
            inputs = {k: (v.to(dev) if hasattr(v, "to") else v)
                      for k, v in inputs.items()}
            # logits_per_audio: (n_slices, n_labels); average across slices.
            probs = model(**inputs).logits_per_audio.softmax(dim=-1).mean(dim=0)
        return probs.float().cpu().numpy()
    finally:
        del model, processor
        reset_memory(device)


def detect_genre_clap(audio_file: str, config: dict) -> str:
    """
    Zero-shot musical-genre classification of the ACTUAL AUDIO using CLAP
    (laion/clap-htsat-unfused): scores a few slices from the middle of the
    song against CLAP_GENRE_LABELS and returns a genre string.

    When the runner-up is close enough (see clap_hybrid_*), the return value
    is a hybrid such as "pop hip-hop" rather than only the top label. A weak
    single-label result still hedges as "top (possibly runner-up)". Returns
    "" on any failure -- the pipeline then proceeds with lyrics-only
    inference, same as before this feature existed.
    """
    labels = list(config.get("clap_genre_labels") or CLAP_GENRE_LABELS)
    model_id = config.get("clap_model_id", "laion/clap-htsat-unfused")
    print(f"[genre] No genre in song_metadata -- classifying audio with "
          f"CLAP ({model_id})...")
    try:
        # The standard CLAP zero-shot template.
        probs = _clap_zero_shot(
            audio_file, [f"This audio is {lbl} music" for lbl in labels], config)
        if probs is None:
            print("[genre] Song too short to slice for CLAP; skipping.")
            return ""
        order = np.argsort(probs)[::-1]
        ranked = [(labels[i], float(probs[i])) for i in order]
        top, p_top = ranked[0]
        second, p_second = ranked[1] if len(ranked) > 1 else ("", 0.0)
        genre = _compose_hybrid_genre(ranked, config)
        print(f"[genre] CLAP: {top} ({p_top:.0%}), runner-up {second} "
              f"({p_second:.0%}) -> using '{genre}'")
        return genre
    except Exception as e:
        print(f"[genre] CLAP detection failed ({e}); continuing without a "
              f"detected genre.")
        try:
            reset_memory(device)
        except Exception:
            pass
        return ""


# Vocal type -> CLAP caption. Tells the film bible whose voice we hear.
CLAP_VOCAL_LABELS = {
    "male": "the sound of a man singing",
    "female": "the sound of a woman singing",
    "duet": "the sound of a man and a woman singing a duet",
    "group": "the sound of a choir or a group of singers",
    "male_rap": "the sound of a man rapping",
    "female_rap": "the sound of a woman rapping",
    "instrumental": "instrumental music with no vocals",
}


def detect_vocals_clap(audio_file: str, config: dict) -> dict:
    """Zero-shot vocal type ("male", "female", "duet", "group", "male_rap",
    "female_rap", "instrumental"). Returns {} on failure or when disabled."""
    if not config.get("clap_vocal_detection", True):
        return {}
    keys = list(CLAP_VOCAL_LABELS)
    print("[vocals] classifying the lead voice with CLAP...")
    try:
        probs = _clap_zero_shot(audio_file, [CLAP_VOCAL_LABELS[k] for k in keys], config)
        if probs is None:
            return {}
        order = np.argsort(probs)[::-1]
        ranked = [(keys[i], float(probs[i])) for i in order]
        print("[vocals] CLAP: " + ", ".join(f"{k} {p:.0%}" for k, p in ranked[:3]))
        return {"vocal": ranked[0][0], "confidence": round(ranked[0][1], 3),
                "ranked": [[k, round(p, 3)] for k, p in ranked]}
    except Exception as e:
        print(f"[vocals] CLAP vocal detection failed ({e}); the performer "
              f"will be inferred from the lyrics.")
        return {}


def _brief_block(brief: dict = None) -> str:
    """Render the creative brief as an instruction block. With no argument,
    reads the module-level CURRENT_BRIEF (set per song by
    author_scenes()), so prompt builders need no extra parameter."""
    b = brief if brief is not None else CURRENT_BRIEF
    if not b:
        return ("WORLD & GENRE GROUNDING: invent the world from the lyrics; "
                "named places are filmed as named.")
    mm = "\n".join(
        f'    - "{m.get("lyric_phrase","")}" -> {m.get("grounded_visual","")}'
        for m in b.get("metaphor_map", []) if isinstance(m, dict))
    lines = [
        "WORLD & GENRE GROUNDING (authoritative -- every scene lives in THIS world):",
        f"- Genre: {b.get('genre','')}"
        + (" (hybrid -- use BOTH styles, do not drop the second)"
           if " " in str(b.get("genre") or "")
           or "/" in str(b.get("genre") or "") else ""),
        f"- World: {b.get('world','')}",
        f"- Era & props: {b.get('era_and_props','')}",
        f"- Protagonist (narrator implied by lyrics/genre): {b.get('protagonist_note','')}",
        f"- Surrealism budget: {b.get('surrealism_budget','')}",
    ]
    if mm:
        lines.append("- Metaphor translations (lyrics are FIGURATIVE; render "
                     "them as these grounded visuals, never literally):\n" + mm)
    direction = _direction_block(b)
    if direction:
        lines.append(direction)
    return "\n".join(l for l in lines if not l.endswith(": "))


def _direction_block(d: dict) -> str:
    """The producer's director_notes and must_show list as a prompt block
    ("" when the song has none)."""
    d = d or {}
    notes = str(d.get("director_notes") or "").strip()
    must = _str_list(d.get("must_show"), 24)
    if not (notes or must):
        return ""
    lines = ["DIRECTOR'S BRIEF (authoritative -- overrides genre habits and "
             "the generic defaults):"]
    if notes:
        lines.append(notes)
    if must:
        lines.append("MUST SHOW (each at least once, spread across the film):")
        lines += [f"- {m}" for m in must]
    return "\n".join(lines)

def energy_band_thresholds(analysis: dict) -> tuple:
    """
    Tertile thresholds computed from THIS song's own energy distribution,
    so the low/mid/high split is always roughly 1/3-1/3-1/3 of the song
    regardless of envelope skew. Fixes the failure mode where a very quiet
    intro/outro drags the dB minimum down during min-max normalization,
    pushing the entire rest of the song above a fixed 0.66 cutoff and
    classifying every scene "high" (which flattens both the cut-rate
    scaling and the camera grammar into one uniform register).

    Returns (low_cut, high_cut): energy < low_cut is "low", energy >=
    high_cut is "high", between is "mid". Falls back to the old fixed
    (0.33, 0.66) when the envelope is too short to take percentiles of.
    Degenerate distributions (near-constant energy) also fall back, since
    percentiles of a flat envelope would make the bands arbitrary.
    """
    env = analysis.get("energy") or []
    if len(env) < 10:
        return (0.33, 0.66)
    arr = np.asarray(env, dtype=float)
    lo = float(np.percentile(arr, 33))
    hi = float(np.percentile(arr, 66))
    # A loudness-normalised master can sit at 0.85-0.92 for the whole song
    # (tertiles only 0.02 apart). Falling back to the FIXED (0.33, 0.66) cut
    # there tagged 28/29 scenes "high" and flattened camera grammar, cut rate
    # and CGI intensity into one register. Song-RELATIVE tertiles are still
    # the right answer for a compressed song -- the loudest third is "high",
    # the quietest third is "low" -- so keep them unless the envelope is
    # truly constant, in which case everything is "mid".
    if (float(arr.max()) - float(arr.min())) < 0.01:
        return (-1.0, 2.0)   # constant energy -> every scene "mid"
    if (hi - lo) < 1e-6:
        hi = lo + 1e-6
    return (lo, hi)

def energy_band(e: float, thresholds: tuple = (0.33, 0.66)) -> str:
    """Bucket an energy value into a coarse intensity band used to pick cut
    rate and camera grammar. `thresholds` is (low_cut, high_cut) -- pass the
    song-relative tertiles from energy_band_thresholds(analysis) so bands
    adapt to each song's dynamic range; the default preserves the old fixed
    behavior for callers without an analysis in hand."""
    low_cut, high_cut = thresholds
    if e < low_cut:
        return "low"
    if e < high_cut:
        return "mid"
    return "high"

def build_beat_aligned_boundaries(analysis: dict, lyric_starts: list,
                                  config: dict) -> list:
    """
    Turn the raw musical analysis into a list of SCENE-CUT times (seconds),
    the heart of beat-driven cutting. Strategy:
      1. Seed cuts from SECTION boundaries (structural changes) and the lyric
         starts (so scenes still track the song's narrative), unioned.
      2. Snap every seed cut to the nearest DOWNBEAT (or beat, if no downbeats)
         so cuts land on the music, not between it.
      3. Enforce a target scene length that SCALES WITH ENERGY: quiet passages
         hold longer (fewer cuts), intense passages cut faster (more cuts).
         Energy banding uses SONG-RELATIVE tertiles (energy_band_thresholds)
         rather than fixed 0.33/0.66 cutoffs, so a skewed envelope (quiet
         intro dragging the normalization floor down) can't classify the
         whole song as one band and flatten the pacing.
    Returns a sorted list of unique cut times starting at 0.0.
    """
    duration = float(analysis.get("duration") or 0.0)
    downbeats = list(analysis.get("downbeat_times") or [])
    beats = list(analysis.get("beat_times") or [])
    sections = list(analysis.get("section_times") or [])
    grid = downbeats if len(downbeats) >= 4 else beats  # snapping grid
    thresholds = energy_band_thresholds(analysis)

    def snap(t):
        if not grid:
            return t
        return min(grid, key=lambda g: abs(g - t))

    # 1) seed cuts
    seeds = set([0.0])
    for t in sections:
        seeds.add(t)
    for t in (lyric_starts or []):
        seeds.add(float(t))

    # 2) snap to musical grid
    snapped = sorted(set(snap(t) for t in seeds if 0.0 <= t < duration))
    if 0.0 not in snapped:
        snapped = [0.0] + snapped

    # 3) energy-scaled cut density
    min_hold = float(config.get("h3_min_scene_seconds", 4.5))
    max_hold = float(config.get("h3_max_scene_seconds", 30.0))
    # energy-band -> target scene length (seconds). High energy = short holds.
    target_by_band = {
        "low":  float(config.get("cut_target_low_seconds", 12.0)),
        "mid":  float(config.get("cut_target_mid_seconds", 7.0)),
        "high": float(config.get("cut_target_high_seconds", 3.5)),
    }

    cuts = [0.0]
    anchors = snapped + [duration]
    for a_idx in range(len(anchors) - 1):
        seg_start = anchors[a_idx]
        seg_end = anchors[a_idx + 1]
        if seg_end - seg_start <= 1e-3:
            continue
        # local energy sets how often to cut inside this segment
        band = energy_band(energy_at(analysis, (seg_start + seg_end) / 2.0),
                           thresholds)
        target = max(min_hold, min(max_hold, target_by_band[band]))
        # candidate interior grid points (downbeats, or beats as fallback)
        interior = [g for g in grid if seg_start + min_hold <= g <= seg_end - min_hold]
        last = cuts[-1]
        # Greedily place a cut at the grid point NEAREST each target cadence
        # mark (last + target, last + 2*target, ...), so cut rate tracks the
        # energy target as closely as the grid allows.
        next_mark = last + target
        for g in interior:
            if g - last < min_hold:
                continue
            if g >= next_mark:
                cuts.append(g)
                last = g
                next_mark = last + target
        # ensure the anchor (section/lyric boundary) itself is a cut if it
        # respects the min hold
        if seg_end < duration and (seg_end - cuts[-1]) >= min_hold:
            cuts.append(seg_end)

    cuts = sorted(set(round(c, 3) for c in cuts if 0.0 <= c < duration))
    if not cuts or cuts[0] != 0.0:
        cuts = [0.0] + cuts

    # final pass: drop any cut that violates the min hold (keep the earlier one)
    pruned = [cuts[0]]
    for c in cuts[1:]:
        if c - pruned[-1] >= min_hold:
            pruned.append(c)
    print(f"  [music] beat-aligned cutting: {len(pruned)} scene cuts "
          f"(avg hold {duration / max(1, len(pruned)):.1f}s, "
          f"band thresholds low<{thresholds[0]:.2f}, high>={thresholds[1]:.2f}, "
          f"energy-scaled from {target_by_band['low']:.0f}s quiet to "
          f"{target_by_band['high']:.0f}s intense).")
    return pruned

def scenes_from_cuts(cut_times: list, analysis: dict, duration: float) -> list:
    """One timing scene per beat-aligned cut interval, tagged with its musical
    energy: "energy" (0..1 at the interval midpoint) and "energy_band"
    (low|mid|high, using the same song-relative tertiles as
    build_beat_aligned_boundaries, so bands always split into a real mix).
    Lyric text is attached afterwards by restamp_scene_lyrics()."""
    thresholds = energy_band_thresholds(analysis)
    cuts = sorted(set(float(c) for c in cut_times if 0.0 <= c < duration))
    if not cuts or cuts[0] != 0.0:
        cuts = [0.0] + cuts
    out = []
    for seg_start, seg_end in zip(cuts, cuts[1:] + [duration]):
        e = energy_at(analysis, (seg_start + seg_end) / 2.0)
        out.append({"start": round(seg_start, 3), "text": "",
                    "energy": round(e, 3), "energy_band": energy_band(e, thresholds)})
    bands = [s["energy_band"] for s in out]
    print(f"  [music] {len(out)} beat-aligned scenes "
          f"(low={bands.count('low')}, mid={bands.count('mid')}, "
          f"high={bands.count('high')}).")
    return out

# =============================================================================
# Whisper transcription
# =============================================================================
def get_audio_pipe():
    model_id = "openai/whisper-large-v3-turbo"
    processor = WhisperProcessor.from_pretrained(model_id)
    model = WhisperForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=dtype, low_cpu_mem_usage=True
    ).to(device)
    return processor, model


# =============================================================================
# Z-Image still generation
# =============================================================================
def load_KREA2_model():
    model_id = "krea/Krea-2-Turbo"
    
    transformer_quant = DiffusersBitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    
    text_encoder_quant = TransformersBitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    
    quant_config = PipelineQuantizationConfig(
        quant_mapping={
            "transformer": transformer_quant,
            "text_encoder": text_encoder_quant,
        }
    )
    
    pipe = Krea2Pipeline.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        quantization_config=quant_config,
        safety_checker=None,
        requires_safety_checker=False     
    )
    # Always on: the NSFW LoRA improves overall image quality, not just heat.
    pipe.load_lora_weights("uzumix/krea2_nsfw")
    pipe.to("cuda")
    return pipe


def _krea_negative_prompt(config: dict) -> str:
    parts = [str(config.get("krea_negative_prompt") or "").strip(),
             "triptych, diptych, collage, contact sheet, split screen, "
             "comic panels, filmstrip, captions, titles, words, writing, "
             "duplicated person, clones, the same face repeated"]
    return ", ".join(p for p in parts if p)


def _call_krea_pipe(pipe, prompt, negative_prompt, height, width, seed, config):
    """Single txt2img call. Drops kwargs the installed pipeline does not accept."""
    steps = int(config.get("krea_num_inference_steps", 12))
    guidance = float(config.get("krea_guidance_scale", 0.0))
    gen = torch.Generator("cuda").manual_seed(int(seed))
    kwargs = dict(
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=int(height),
        width=int(width),
        num_inference_steps=steps,
        guidance_scale=guidance,
        generator=gen,
    )
    complexity = config.get("krea_complexity", None)
    if complexity is not None:
        kwargs["complexity"] = float(complexity)

    last_err = None
    for _ in range(4):
        try:
            return pipe(**kwargs).images[0]
        except TypeError as e:
            last_err = e
            msg = str(e)
            dropped = False
            for key in ("complexity", "negative_prompt", "guidance_scale"):
                if key in kwargs and key in msg:
                    kwargs.pop(key, None)
                    dropped = True
                    break
            if not dropped:
                raise
    if last_err:
        raise last_err
    raise RuntimeError("Krea2 pipeline call failed")


def _third_clone_score(image) -> float:
    """How much the left/center/right thirds look like the same figure.

    The common Krea failure on a 16:9 canvas is a triptych of one person.
    Comparing normalized grayscale thirds is a cheap detector that needs no
    second model. High score => regenerate.
    """
    try:
        w, h = image.size
        if w < 96 or h < 96:
            return 0.0
        y0, y1 = int(h * 0.12), int(h * 0.88)
        tw = w // 3
        vecs = []
        for i in range(3):
            crop = image.crop((i * tw, y0, (i + 1) * tw, y1)).convert("L").resize((48, 64))
            arr = np.asarray(crop, dtype=np.float32).ravel()
            arr = arr - arr.mean()
            n = float(np.linalg.norm(arr)) + 1e-6
            vecs.append(arr / n)
        dots = (
            float(np.dot(vecs[0], vecs[1])),
            float(np.dot(vecs[1], vecs[2])),
            float(np.dot(vecs[0], vecs[2])),
        )
        return max(dots)
    except Exception:
        return 0.0


@torch.inference_mode()
def generate_image(pipe, prompt: str, resolution: tuple,
                   seed: int, style_prefix: str, config: dict = None,
                   scene: dict = None) -> tuple:
    """One widescreen still. The prompt is the scene itself plus its visual
    style -- nothing that doesn't describe the picture. If the decoded frame
    looks like three copies of the same person, re-roll with a fresh seed.
    (Krea-2-Turbo ships with guidance_scale=0, so negative_prompt is a no-op.)
    """
    config = config or {}
    height, width = resolution
    full_prompt = prompt
    # Medium and style lead the prompt so the look is set before the
    # content; stated only at the tail it loses to the photographic default.
    if style_prefix and style_prefix not in full_prompt:
        full_prompt = f"{style_prefix}. {full_prompt}"
    theme = _image_theme()
    if theme and theme.lower() not in full_prompt.lower():
        full_prompt = f"{theme}. {full_prompt}"
    negative_prompt = _krea_negative_prompt(config)
    if seed == -1:
        seed = secrets.randbits(32)

    max_tries = max(1, int(config.get("krea_clone_retry", 3)))
    if not config.get("krea_anti_duplicate", True):
        max_tries = 1
    thresh = float(config.get("krea_clone_similarity_max", 0.90))
    best_image, best_seed, best_score = None, seed, 1.0
    attempt_seed = seed
    for attempt in range(1, max_tries + 1):
        image = _call_krea_pipe(
            pipe, full_prompt, negative_prompt, height, width,
            attempt_seed, config,
        )
        score = _third_clone_score(image)
        print(f"    [krea] clone-score={score:.3f} "
              f"(max {thresh:.2f}) attempt {attempt}/{max_tries} seed={attempt_seed}")
        if score < best_score:
            best_image, best_seed, best_score = image, attempt_seed, score
        if score < thresh:
            return image, attempt_seed
        attempt_seed = secrets.randbits(32)
    print(f"    [krea] WARNING: still looks cloned after {max_tries} "
          f"tries (best score {best_score:.3f}); keeping the least-similar frame.")
    return best_image, best_seed


# =============================================================================
# LLM CLIENT MANAGEMENT
# =============================================================================
_openai_client = None
_grok_client = None

H3_CINEMATOGRAPHER_SYSTEM_MESSAGE = (
    "You are an award-winning music-video director, cinematographer and "
    "MiniMax H3 prompt engineer. You turn THIS song's lyrics into an amazing, "
    "creative short film: the characters the lyrics describe doing what the "
    "lyrics describe, in the places the lyrics name, intercut with the "
    "performer(s) whose voice we hear singing and dancing to the track. "
    "Settings are literal when the lyric names them (a park is a park; Paris "
    "is recognizably Paris in the mood of the line). Who appears -- gender, "
    "age, kind, how many -- comes from the lyrics, each with a locked look you "
    "repeat verbatim. Content is as sexual as each scene's own lines and no "
    "more, and sexual content involves adults only. You know that H3 reads "
    "cinematography vocabulary directly (motion type + amplitude + speed), "
    "that every shot is its OWN standalone 5-26 second generation from one "
    "reference still, driven by a real slice of the song -- so motion reacts "
    "to music already playing. Every still is one unbroken 16:9 frame where "
    "each character appears once; never a triptych, collage or split screen. "
    "You never use em-dashes. You never use double spaces between sentences "
    "in JSON output."
)

def _select_model(use_grok: bool = False, use_large: bool = False) -> str:
    """Model tier: the creative planning calls request the large model;
    cheap mechanical calls keep the fast one."""
    if use_grok:
        return CONFIG["grok_model"]
    if use_large:
        return CONFIG.get("openai_model_large") or CONFIG["openai_model"]
    return CONFIG["openai_model"]

def _get_client(use_grok: bool = False):
    return _get_grok_client() if use_grok else _get_openai_client()


def _get_openai_client():
    global _openai_client
    if _openai_client is None:
        if not CONFIG["openai_api_key"]:
            raise RuntimeError(
                "OPENAI_API_KEY is not set. Export it in your shell "
                "(export OPENAI_API_KEY=sk-...) before running.")
        _openai_client = OpenAI(api_key=CONFIG["openai_api_key"],
                                timeout=_llm_timeout(), max_retries=0)
    return _openai_client


def _get_grok_client():
    global _grok_client
    if _grok_client is None:
        if not CONFIG["grok_api_key"]:
            raise RuntimeError(
                "XAI_API_KEY is not set but USE_GROK is True. Export it "
                "(export XAI_API_KEY=xai-...) or set USE_GROK=False.")
        _grok_client = OpenAI(api_key=CONFIG["grok_api_key"],
                              base_url="https://api.x.ai/v1",
                              timeout=_llm_timeout(), max_retries=0)
    return _grok_client


def _llm_timeout() -> float:
    # The SDK default is 600s plus 2 SILENT internal retries, so one printed
    # "Attempt" could hang ~30 min and a failed batch ~90 min. Our own retry
    # loop is the only retry layer (max_retries=0 on both clients).
    return float(CONFIG.get("llm_timeout_seconds", 900))


def _collect_stream(stream):
    """Join a streamed chat completion into (content, finish_reason, usage)."""
    parts, finish_reason, usage = [], None, None
    for chunk in stream:
        if getattr(chunk, "usage", None):
            usage = chunk.usage
        if not chunk.choices:
            continue
        choice = chunk.choices[0]
        if choice.delta and choice.delta.content:
            parts.append(choice.delta.content)
        if choice.finish_reason:
            finish_reason = choice.finish_reason
    return "".join(parts), finish_reason, usage


def _record_usage(usage) -> None:
    """Tally prompt vs cached prompt tokens so cache hits are visible."""
    if usage is None:
        return
    LLM_STATS["prompt_tokens"] += getattr(usage, "prompt_tokens", 0) or 0
    details = getattr(usage, "prompt_tokens_details", None)
    LLM_STATS["cached_tokens"] += getattr(details, "cached_tokens", 0) or 0


# Request params a (backend, model) has rejected once; never sent to it again.
# Grok 4.x rejects reasoning_effort="none" with a 400 on EVERY call, which used
# to empty every authoring stage and collapse the film into one fallback shot.
_UNSUPPORTED_LLM_PARAMS = {}
_DROPPABLE_LLM_PARAMS = ("reasoning_effort", "stream_options")
LLM_STATS = {"calls": 0, "failures": 0, "prompt_tokens": 0, "cached_tokens": 0}
# xAI caches by exact message prefix, per server; a stable x-grok-conv-id
# routes every call of this session to the same server so the prefix
# (system message + each loop's constant context) is actually reused.
_LLM_CONV_ID = secrets.token_hex(16)


def get_openai_prompt_response(prompt, max_completion_tokens=36000,
                               temperature=0.70, use_grok=None,
                               system_message=None, use_large=False,
                               context=None):
    if use_grok is None:
        use_grok = USE_GROK
    model = _select_model(use_grok, use_large=use_large)
    client = _get_client(use_grok)
    system_msg = system_message or H3_CINEMATOGRAPHER_SYSTEM_MESSAGE
    model_name = "Grok" if use_grok else "OpenAI"
    dropped = _UNSUPPORTED_LLM_PARAMS.setdefault((model_name, model), set())
    LLM_STATS["calls"] += 1
    attempt = 0
    while attempt < retry_limit:
        token_key = 'max_tokens' if use_grok else 'max_completion_tokens'
        # Most-stable first: cache hits stop at the first message that differs.
        messages = [{"role": "system", "content": system_msg}]
        if context:
            messages.append({"role": "user", "content": context})
        messages.append({"role": "user", "content": prompt})
        kwargs = dict(
            messages=messages,
            model=model,
            **{token_key: max_completion_tokens},
        )
        # Reasoning-tuned models reject any non-default temperature, so
        # `temperature` is never sent; reasoning_effort is requested instead,
        # unless this model has already rejected it. Grok can't turn reasoning
        # off ("none" is a 400), so it gets the lowest level it accepts.
        if "reasoning_effort" not in dropped:
            kwargs["reasoning_effort"] = (
                CONFIG.get("grok_reasoning_effort", "low") if use_grok else "none")
        # Grok reasoning models think for minutes on the big planning prompts
        # and can't turn reasoning off; a non-streamed call sits silent until
        # the read timeout kills it. Streaming keeps the connection alive.
        if use_grok:
            kwargs["stream"] = True
            kwargs["extra_headers"] = {"x-grok-conv-id": _LLM_CONV_ID}
            if "stream_options" not in dropped:
                kwargs["stream_options"] = {"include_usage": True}
        try:
            response = client.chat.completions.create(**kwargs)
            if use_grok:
                content, finish_reason, usage = _collect_stream(response)
            else:
                finish_reason = response.choices[0].finish_reason
                content = response.choices[0].message.content
                usage = response.usage
            _record_usage(usage)
        except Exception as e:
            msg = str(e)
            bad = next((p for p in _DROPPABLE_LLM_PARAMS
                        if p in kwargs and p in msg), None)
            if bad:
                dropped.add(bad)
                print(f"  [llm] {model_name} {model} rejects `{bad}`; "
                      f"retrying without it.")
                continue
            attempt += 1
            if getattr(e, "status_code", None) in (400, 401, 403, 404):
                print(f"Attempt {attempt} error ({model_name}): {e}. Not retryable.")
                break
            wait_time = min(2 ** attempt, 30)
            print(f"Attempt {attempt} error ({model_name}): {e}. "
                  f"Waiting {wait_time}s...")
            time.sleep(wait_time)
            continue
        attempt += 1
        if finish_reason == 'length':
            print(f"WARNING ({model_name}): Response truncated. "
                  f"Got {len(content) if content else 0} chars.")
        if content and len(content.strip()) > 0:
            return content
        print(f"Empty content on attempt {attempt} ({model_name}). "
              f"Finish: {finish_reason}")
    LLM_STATS["failures"] += 1
    print(f"ERROR: LLM call failed ({model_name} {model})")
    return ""


def llm_preflight() -> None:
    """One tiny call before any slow work. A dead backend would otherwise
    let every authoring stage fall back to the same template scene."""
    backend = "Grok" if USE_GROK else "OpenAI"
    model = _select_model(USE_GROK, use_large=True)
    reply = get_openai_prompt_response(
        'Return only this JSON object: {"ok": true}',
        max_completion_tokens=200, system_message="You return only valid JSON.",
        use_large=True)
    if not safe_json_dict(reply).get("ok"):
        raise RuntimeError(
            f"LLM preflight failed for {backend} model '{model}' (reply: "
            f"{reply[:200]!r}). Check the API key, model name and USE_GROK "
            f"before running; see the error printed above.")
    print(f"[llm] {backend} '{model}' is answering.")

# Style Bible -- the locked visual language for the whole video, generated per
# song by generate_style_bible(). DEFAULT_STYLE_BIBLE is only the fallback for
# fields the LLM leaves empty.
DEFAULT_STYLE_BIBLE = {
    "medium": "3D digital animation, cinematic lighting, hyperreal detail, intimate "
              "and beautiful with special effects",
    "palette": "deep indigo blues, molten gold accents, muted violet shadows",
    "lighting": "dramatic rim lighting, volumetric god-rays, high-contrast "
               "chiaroscuro",
    "recurring_symbol": "drifting golden embers",
    "visual_motif": "reflections and light crossing thresholds",
    "palette_arc": "cool restraint -> muted warmth -> luminous catharsis",
    "film_grammar": "shallow depth of field, cinematic rim lighting, "
                    "hyperreal detail, one clear focal point per frame",
    "cgi_magic_language": "physically integrated volumetric effects, rich material simulation, elegant particles and transformation",
    "cgi_effect_density": "moderate",
    "emotional_arc": ["longing", "temptation", "conflict", "despair",
                      "defiance", "catharsis"],
}

# =============================================================================
# LLM-prompt rendering helpers for the bibles (single source each).
# =============================================================================
def _as_style_str(value) -> str:
    """Coerce a style-bible field into a plain string, whatever shape the
    LLM returned it in (some models emit palette/medium/lighting as a list
    of phrases instead of one string)."""
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, (list, tuple)):
        return ", ".join(str(v).strip() for v in value if v)
    return str(value).strip()


def _image_theme() -> str:
    """This song's rendering medium: song_metadata image_theme, else the
    global IMAGE_THEME."""
    return str((CURRENT_BRIEF or {}).get("image_theme") or IMAGE_THEME or "").strip()


def _style_block(style_bible: dict) -> str:
    """Render a style bible into an LLM-prompt instruction block."""
    labels = {
        "medium": "Medium",
        "palette": "Color Palette",
        "lighting": "Lighting Style",
        "visual_motif": "Visual Motif",
        "palette_arc": "Palette Progression",
        "film_grammar": "Cinematography",
        "cgi_magic_language": "CGI Magic Language",
        "cgi_effect_density": "CGI Effect Density",
    }
    arc = style_bible.get("emotional_arc", [])
    arc_str = " -> ".join(arc) if isinstance(arc, list) else str(arc)
    lines = ["LOCKED VISUAL LANGUAGE (render every scene in this exact medium, "
             "palette, and lighting; do not deviate -- diversity comes from "
             "PLACE and SUBJECT, not from the look):"]
    for key, label in labels.items():
        val = _as_style_str(style_bible.get(key))
        if val:
            lines.append(f"- {label}: {val}")
    if arc_str:
        lines.append(f"- Emotional arc (progress through these across the "
                     f"song, in order): {arc_str}")
    return "\n".join(lines)


def _style_bible_image_prefix(style_bible: dict) -> str:
    """The style bible's medium, palette and lighting as one comma-joined
    phrase for the still prompt."""
    parts = [_as_style_str(style_bible.get("medium")),
             _as_style_str(style_bible.get("palette")),
             _as_style_str(style_bible.get("lighting"))]
    return ", ".join(p for p in parts if p)


def assign_cgi_magic_direction(scenes: list, config: dict,
                               families: list = None) -> None:
    """Assign a CGI effect family and intensity per scene. A scene's own
    story_cgi_hook wins; otherwise a family is drawn by energy band, never
    repeating recent families. Most scenes get restrained physical magic;
    only selected peaks become spectacular set pieces."""
    if not scenes or not config.get("cgi_magic_enabled", True):
        return
    rng=random.Random(17411 + len(scenes))
    last_families=[]
    for i,scene in enumerate(scenes):
        hook = str(scene.get("story_cgi_hook") or "").strip()
        if hook:
            scene["cgi_magic_enabled"] = True
            scene["cgi_effect_family"] = hook
            band = str(scene.get("energy_band", "mid"))
            spec_cap = max(2, int(len(scenes) * float(
                config.get("cgi_spectacular_cap_ratio", 0.28))))
            n_spec = sum(1 for s in scenes[:i]
                         if s.get("cgi_effect_intensity") == "spectacular")
            if band == "high" and n_spec < spec_cap and rng.random() < float(
                    config.get("cgi_high_spectacular_probability", 0.32)):
                scene["cgi_effect_intensity"] = "spectacular"
            elif band == "low":
                scene["cgi_effect_intensity"] = "subtle"
            else:
                scene["cgi_effect_intensity"] = "moderate"
            last_families.append(hook)
            continue
        band=str(scene.get("energy_band","mid"))
        if band=="high":
            enabled=(i==0) or (rng.random() < float(config.get("cgi_magic_peak_probability",0.95)))
            spectacular=(rng.random() < float(config.get("cgi_high_spectacular_probability",0.32)))
        elif band=="mid":
            enabled=(rng.random() < float(config.get("cgi_magic_probability",0.78)))
            spectacular=(rng.random() < float(config.get("cgi_mid_spectacular_probability",0.12)))
        else:
            enabled=(rng.random() < float(config.get("cgi_low_enabled_probability",0.30)))
            spectacular=False
        if not enabled:
            scene["cgi_magic_enabled"]=False
            scene["cgi_effect_family"]=""
            scene["cgi_effect_intensity"]="none"
            continue
        pool=list(families or MAGIC_EFFECT_FAMILIES)
        recent=set(last_families[-int(config.get("cgi_max_same_family_run",2)):])
        choices=[x for x in pool if x not in recent] or pool
        family=rng.choice(choices)
        scene["cgi_magic_enabled"]=True
        scene["cgi_effect_family"]=family
        if band=="low":
            intensity="subtle"
        elif spectacular:
            intensity="spectacular"
        elif band=="high":
            intensity="moderate"
        else:
            intensity="moderate" if rng.random()<0.65 else "subtle"
        scene["cgi_effect_intensity"]=intensity
        last_families.append(family)


def generate_style_bible(text: str, config: dict) -> dict:
    """LLM-generated, per-song locked visual language, grounded in the Creative
    Brief's genre/world so the PALETTE and LIGHTING match the MUSIC's visual
    tradition. Output is filtered to the known keys.

    "medium" is anchored to the global IMAGE_THEME regardless of genre: the
    prompt states it as a hard override, and a post-parse guard forces
    IMAGE_THEME into "medium" if the LLM ignored it, so every consumer of the
    bible (still-image style prefix, every scene's style clause) agrees.
    """
    print("Generating Style Bible (locked visual language)...")
    image_theme = _image_theme()

    medium_rule = (
        f'HARD OVERRIDE (non-negotiable, ignore genre convention for this '
        f'one field): the "medium" below MUST be anchored to "{image_theme}" '
        f'-- this is a global setting and applies REGARDLESS of the song\'s '
        f'genre. Do NOT substitute naturalistic live-action realism, '
        f'photographic realism, documentary style, or any other medium even '
        f'if the genre would normally call for it. You may add a few '
        f'compatible descriptive words (rendering technique, texture, '
        f'shading style) but the core medium phrase must start with '
        f'"{image_theme}".'
        if image_theme else
        'Chosen to fit the GENRE\'s visual tradition above. A country/'
        'folk/rock song wants naturalistic cinematic live-action realism '
        'with film texture and natural light -- NOT stylized 3D render or '
        'anime -- unless the genre genuinely calls for stylization.'
    )

    prompt = f'''
You are the art director for a music video. Cohesion comes from a consistent
LOOK, a recurring symbol, and an emotional arc.

### {_brief_block()}

Theme to amplify lyrics: {THEME}
Lyrics: {text}

### CGI MOVIE-MAGIC DIRECTION
{CGI_MAGIC_DIRECTIVE}

Return ONLY raw JSON (no markdown) with EXACTLY these keys, values short and
concrete:
- "medium": ONE rendering medium used for EVERY shot. {medium_rule}
  Pick one and commit.
- "palette": 3-4 signature colors plus a grade that belongs to this genre's
  world (e.g. iron-red dirt, denim blue, diesel-smoke gray, wheat gold for
  heartland country), applied to every shot.
- "lighting": one recurring lighting style native to this world.
- "recurring_symbol": ONE small physical motif from the METAPHOR MAP or the
  world itself, woven through roughly half the shots. Can be none.
- "film_grammar": shared lens/camera language (e.g. "anamorphic lens,
  shallow depth of field, subtle film grain, rule-of-thirds"). Keep this
  compatible with the "medium" above -- do not imply photographic/live-action
  capture (e.g. "film grain", "35mm") when the medium is a non-photographic
  rendering style.
- "emotional_arc": array of 4-6 short emotional beats the video moves
  through in order.
- "cgi_magic_language": 1-2 short sentences defining the film's signature CGI
  effects language, materials and light behavior.
- "cgi_effect_density": "subtle|moderate|spectacular".
'''
    bible = _llm_dict(prompt, temperature=0.6)

    # Filter to the known schema FIRST (drops stray keys like "shots").
    bible = {k: bible.get(k) for k in DEFAULT_STYLE_BIBLE}

    for k, v in DEFAULT_STYLE_BIBLE.items():
        if not bible.get(k):
            bible[k] = v
    for k in ("medium", "palette", "lighting", "recurring_symbol",
              "visual_motif", "palette_arc", "film_grammar",
              "cgi_magic_language", "cgi_effect_density"):
        bible[k] = _as_style_str(bible.get(k)) or _as_style_str(DEFAULT_STYLE_BIBLE.get(k, ""))
    if isinstance(bible.get("emotional_arc"), str):
        bible["emotional_arc"] = [b.strip() for b in
                                  re.split(r"[,;>|]", bible["emotional_arc"]) if b.strip()]
    elif isinstance(bible.get("emotional_arc"), list):
        bible["emotional_arc"] = [_as_style_str(b) for b in bible["emotional_arc"] if b]

    # DETERMINISTIC GUARD: never trust the LLM alone to honor the medium
    # override -- force image_theme into "medium" if it isn't already there,
    # so every downstream consumer of this bible (style_prefix in
    # generate_image(), and every scene's own "Style Details:" clause via
    # _style_block()) is guaranteed consistent with the intended rendering
    # style, regardless of genre or LLM compliance.
    if image_theme:
        current_medium = bible.get("medium", "")
        if image_theme.lower() not in current_medium.lower():
            print(f"  [style] LLM's medium ('{current_medium}') did not "
                  f"anchor to image_theme ('{image_theme}'); forcing it in.")
            bible["medium"] = (f"{image_theme}, {current_medium}"
                               if current_medium else image_theme)

    print("Style Bible:\n" + json.dumps(bible, indent=2))
    return bible

# =============================================================================
# Scene duration normalization
# =============================================================================
def _merge_scene_pair(earlier: dict, later: dict) -> dict:
    merged = dict(earlier)
    merged["text"] = " ".join(t for t in [(earlier.get("text") or "").strip(),
                                          (later.get("text") or "").strip()] if t)
    return merged


def normalize_scene_durations(scenes: list, last_end_value: float,
                              config: dict) -> list:
    """
    Enforce duration bounds on the timing scenes before any content is
    planned. Also stamps each scene's final duration into "_duration". Scenes longer than h3_max_scene_seconds are
    split into consecutive pieces here so that each piece stays within a
    single H3 call and gets its own reference image and animation downstream.

    COVERAGE GUARD: the first scene is forced to start at 0.0. Without this,
    if beat-driven cutting is disabled or fails and falls back to lyric
    timestamps, the first scene starts at the first LYRIC's timestamp --
    which can be many seconds (an intro) after the song actually begins.
    Nothing downstream would ever render that leading span, silently
    shortening the final video versus the audio. Forcing start=0.0 here
    (instead of only in build_scene_jobs) keeps scenes.json itself an
    accurate, self-contained record of the intended timeline.
    """
    if not scenes:
        return scenes
    min_sec = float(config["h3_min_scene_seconds"])
    max_sec = float(config["h3_max_scene_seconds"])

    for i, s in enumerate(scenes):
        if s.get("start") is None:
            s["start"] = 0.0 if i == 0 else scenes[i - 1].get("start", 0.0)
    scenes = sorted(scenes, key=lambda s: s["start"])

    # COVERAGE GUARD: the song always starts at 0.0. If the first scene's
    # start is meaningfully later, something upstream (beat-driven cutting
    # disabled/failed, or an intro with no lyrics) left the song's beginning
    # uncovered. Rather than silently lose that span, pull the first scene
    # back to 0.0 so the whole song is accounted for.
    first_start = float(scenes[0].get("start", 0.0))
    if first_start > 0.05:
        print(f"  [normalize] WARNING: first scene started at {first_start:.2f}s, "
              f"not 0.0s -- this would leave the first {first_start:.2f}s of the "
              f"song unrendered. Forcing the first scene to start at 0.0s. "
              f"(Likely cause: beat-driven cutting failed or is disabled and "
              f"lyric timestamps were used instead -- check earlier logs for "
              f"'beat-driven cutting failed'.)")
        scenes[0]["start"] = 0.0

    merged = []
    idx = 0
    n = len(scenes)
    while idx < n:
        current = dict(scenes[idx])
        nxt = scenes[idx + 1]["start"] if idx + 1 < n else last_end_value
        while (nxt - current["start"]) < min_sec and idx + 1 < n:
            current = _merge_scene_pair(current, scenes[idx + 1])
            idx += 1
            nxt = scenes[idx + 1]["start"] if idx + 1 < n else last_end_value
        merged.append(current)
        idx += 1
    if len(merged) >= 2 and (last_end_value - merged[-1]["start"]) < min_sec:
        tail = merged.pop()
        merged[-1] = _merge_scene_pair(merged[-1], tail)

    final = []
    for i, scene in enumerate(merged):
        start = scene["start"]
        end = merged[i + 1]["start"] if i + 1 < len(merged) else last_end_value
        duration = max(min_sec, end - start)
        if duration <= max_sec:
            scene["_duration"] = duration
            final.append(scene)
            continue
        pieces = int(-(-duration // max_sec))  # ceil
        piece_len = duration / pieces
        for p in range(pieces):
            copy = dict(scene)
            copy["start"] = start + p * piece_len
            copy["_duration"] = piece_len
            final.append(copy)

    print(f"  [normalize] {len(scenes)} -> {len(final)} scene(s) "
          f"(min={min_sec}s, max={max_sec}s).")
    return final


# =============================================================================
# SCENE PIPELINE -- every song.
#
#   film bible  -> who sings, who the story is about, where it happens (named
#                  places filmed literally), chapters, motifs, payoff
#   scene plan  -> per scene: performance | story | performance_in_story,
#                  cast in frame, literal location, event, dance, heat
#   place guard -> a lyric that names Paris / the park lands there, in code
#   heat gate   -> scene heat never exceeds what its own lines earn
#   writer      -> reference-still prompt + H3 motion prompt per scene
#   audit       -> LLM judge; weak scenes are rewritten once with a fix note
# =============================================================================
_VOICE_TEXT = {
    "male": "a male lead vocalist",
    "female": "a female lead vocalist",
    "duet": "a male-female duet (two performers)",
    "group": "a group or choir (a lead singer plus backing singers)",
    "male_rap": "a male rapper",
    "female_rap": "a female rapper",
    "instrumental": ("no lead vocal (instrumental): the performer is a dancer "
                     "or musician who embodies the track"),
}
_MINOR_RE = re.compile(r"\b(child|children|kid|kids|baby|infant|toddler|teen|"
                       r"teenager|boy|girl|schoolboy|schoolgirl|newborn|little)\b",
                       re.IGNORECASE)


def _looks_minor(entry: dict) -> bool:
    """True when the age reads under 18 or the description names a child."""
    age = str(entry.get("age") or "")
    nums = [int(n) for n in re.findall(r"\d+", age)]
    if nums and min(nums) < 18:
        return True
    return bool(_MINOR_RE.search(f"{age} {entry.get('role', '')} {entry.get('name', '')}"))


def _performer_override(meta: dict) -> str:
    return str((meta or {}).get("performer") or (meta or {}).get("protagonist")
               or globals().get("PROTAGONIST") or "").strip()


def _str_list(value, max_items: int = 8) -> list:
    if value is None:
        return []
    if isinstance(value, str):
        value = [p for p in re.split(r"\s*[,;]\s*", value)]
    if not isinstance(value, (list, tuple)):
        value = [value]
    return [str(v).strip() for v in value if str(v).strip()][:max_items]


def _normalize_film_bible(bible: dict, meta: dict, vocals: dict) -> dict:
    """Make the LLM's film bible safe to index: stable keys (singer,
    singer_2, ...; char_01, char_02, ...), at least one performer, adult
    flags, lowercase place triggers."""
    b = dict(bible or {})
    vocal = (vocals or {}).get("vocal", "")
    performers = [p for p in (b.get("performers") or []) if isinstance(p, dict)]
    if not performers:
        performers = [{"name": "the singer", "voice": vocal or "unknown",
                       "look": _performer_override(meta) or
                       "a charismatic adult lead singer whose look fits the genre",
                       "wardrobe": "a performance outfit native to the genre",
                       "dance_style": "grooves, steps and spins on the beat"}]
    override = _performer_override(meta)
    if override:
        # The producer's performer notes name the lead (e.g. "Dr. Frankenstein,
        # a ..."). If the LLM filed that person as a character and invented a
        # separate narrator, promote the character to lead performer.
        head = override.lower()[:80]
        twin = next((c for c in (b.get("characters") or []) if isinstance(c, dict)
                     and len(str(c.get("name") or "").strip()) > 2
                     and str(c.get("name")).strip().lower() in head), None)
        lead = performers[0]
        if twin is not None:
            b["characters"] = [c for c in b["characters"] if c is not twin]
            for k in ("name", "look", "wardrobe", "age", "gender"):
                if twin.get(k):
                    lead[k] = twin[k]
            print(f"  [film] lead performer is {lead['name']} (from performer notes).")
        elif str(lead.get("name") or "").strip().lower() not in head:
            lead["look"] = override
    for n, p in enumerate(performers):
        p["key"] = "singer" if n == 0 else f"singer_{n + 1}"
        p["role"] = p.get("role") or ("lead singer" if n == 0 else "singer")
        p["is_adult"] = True
        p["performance_settings"] = _str_list(p.get("performance_settings"), 3)
        p["wardrobe_by_chapter"] = _str_list(p.get("wardrobe_by_chapter"), 8)
    b["performers"] = performers

    chars = []
    for n, c in enumerate([c for c in (b.get("characters") or [])
                           if isinstance(c, dict)], 1):
        c["key"] = f"char_{n:02d}"
        # Either signal of a minor wins: the LLM's flag or the age/role text.
        flagged = ("is_adult" in c and (not c["is_adult"]
                                        or str(c["is_adult"]).lower() == "false"))
        c["is_adult"] = not (flagged or _looks_minor(c))
        chars.append(c)
    b["characters"] = chars

    places = []
    for p in [p for p in (b.get("places") or []) if isinstance(p, dict)]:
        name = str(p.get("name") or "").strip()
        if not name:
            continue
        trig = [t.lower() for t in _str_list(p.get("lyric_triggers"), 6)]
        p["lyric_triggers"] = trig or [name.lower()]
        p["name"] = name
        places.append(p)
    b["places"] = places
    def _noun(name):
        words = re.findall(r"[a-z]+", str(name or "").lower())
        return words[-1] if words else ""
    people = {_noun(e.get("name")) for e in performers + chars} - {""}
    props = []
    for p in [p for p in (b.get("hero_props") or []) if isinstance(p, dict)]:
        if not str(p.get("name") or "").strip():
            continue
        if _noun(p.get("name")) in people:
            # Same entity already cast as a performer/character (a living
            # instrument that also sings): keep one lock, not two.
            continue
        p["key"] = f"prop_{len(props) + 1:02d}"
        p["role"] = p.get("role") or "hero prop"
        p["is_adult"] = True
        p["stages"] = _str_list(p.get("stages"), 6)
        props.append(p)
    b["hero_props"] = props
    pieces = []
    for p in [p for p in (b.get("set_pieces") or []) if isinstance(p, dict)]:
        if not str(p.get("what") or p.get("hero_visual") or "").strip():
            continue
        p["id"] = f"sp{len(pieces) + 1:02d}"
        p["cast"] = _str_list(p.get("cast"), 8)
        pieces.append(p)
    b["set_pieces"] = pieces
    b["effect_palette"] = _str_list(b.get("effect_palette"), 12)
    b["chapters"] = [c for c in (b.get("chapters") or []) if isinstance(c, dict)]
    b["motifs"] = _str_list(b.get("motifs"), 3)
    if not isinstance(b.get("metaphor_map"), list):
        b["metaphor_map"] = []
    resolved = str((meta or {}).get("genre") or "").strip()
    if resolved:
        b["genre"] = resolved
    b["vocals"] = vocals or {}
    return b


def generate_film_bible(text: str, meta: dict, content: dict,
                              vocals: dict, config: dict) -> dict:
    """The film's single source of truth: logline, performer(s), the cast the
    lyrics call for, named places, chapters, motifs."""
    print("Generating Film Bible (performers, cast, places, chapters)...")
    genre = (meta or {}).get("genre") or ("infer the most likely genre from the "
                                          "lyrical voice and vocabulary")
    world = str((meta or {}).get("world") or "").strip()
    override = _performer_override(meta)
    vocal = (vocals or {}).get("vocal", "")
    voice_text = _VOICE_TEXT.get(vocal, "unknown -- infer from the lyrics' "
                                        "voice, pronouns and genre")
    conf = (vocals or {}).get("confidence")
    prompt = f"""
You are the director developing an original, amazing, creative music video
for THIS song. Invent a film people would want to rewatch.

Musical genre (audio analysis / producer): {genre}
If the genre is a hybrid, blend both styles.
{f"Producer's world notes (authoritative): {world}" if world else ""}
Detected lead voice: {voice_text}{f" (confidence {conf:.0%})" if isinstance(conf, (int, float)) else ""}
{f"Producer's performer notes (authoritative -- use this look): {override}" if override else ""}
{_direction_block(meta)}
What the song is about: {(content or {}).get("summary", "")}
Song shape: {(content or {}).get("story_shape", "narrative")}; song heat: {(content or {}).get("song_heat", "none")}

Theme:
{THEME}

{_film_mandate_block()}

Lyrics (timestamped transcript):
{text[:10000]}

Design the film:
1. STORY: find the story the lyrics tell. If they narrate events, those
   events ARE the plot. If they are mostly feeling, invent a concrete,
   emotionally specific story that dramatizes that feeling (who, where,
   what happens, what changes). Surprising and specific, never generic.
2. PERFORMER(S): one per voice we hear -- a solo voice is one performer, a
   duet two, a group as many as the sound suggests. When the producer's
   performer notes name a character, the LEAD performer IS that character,
   in story and performance shots alike; never invent a separate narrator.
   Other voices may be story characters too (a creature, a ghost). Match the detected voice
   (gender, and an age that fits the timbre, genre and lyrics). Give each a
   dance style native to the genre. If the lyrics are first person, the performer may
   also be the story's protagonist ("performer_is_protagonist": true) --
   then story scenes show them living the story without singing.
3. CHARACTERS: everyone and everything else the story needs -- whatever
   gender, age, kind (human, angel, animal, machine) and number the lyrics
   call for -- each with a locked look specific enough to recognise in
   every shot.
4. PLACES: every place the lyrics NAME (cities, parks, streets, rivers,
   bars, churches, highways, eras), filmed recognizably in the mood of
   the line that names it; plus the few invented places the story needs.
5. CHAPTERS: 3-7 chapters in song order, each opened by a lyric line.
6. MOTIFS: 1-3 recurring images planted early and paid off at the end.
7. HERO PROPS: any object the story is built around (an instrument being
   made, a car, a letter) is a locked prop with a precise look, and
   "stages" if it changes over the film (unfinished -> finished -> alive).
8. SET PIECES: 18-30 DISTINCT filmable set pieces in rough story order --
   the shot menu the whole video is cut from. Each is a different subject,
   place or phenomenon with one striking hero image; together they cover
   every chapter, every MUST SHOW item, the cast, the props and the
   spectacle. Two set pieces must never describe the same picture.
9. EFFECT PALETTE: 6-10 CGI effect families native to THIS song's world
   (not generic sparkles), each a short phrase.

Return ONLY raw JSON:
{{
  "genre": "copy the genre string verbatim",
  "title_concept": "one line high concept",
  "logline": "1-2 sentences",
  "story_shape": "narrative|anthology|performance|symbolic",
  "world": "2-3 sentences: the places and era(s) of THIS song",
  "era": "one line",
  "tone": "one line",
  "performer_is_protagonist": false,
  "performers": [{{"name": "...", "voice": "male|female|...", "age": "...",
     "look": "20-35 words: face, hair, skin, build, age, one distinguishing mark",
     "wardrobe": "default performance outfit",
     "wardrobe_by_chapter": ["outfit for chapter 1", "..."],
     "dance_style": "how they move from low to high energy, native to the genre",
     "performance_settings": ["1-3 recurring performance places that belong to this song's world"]}}],
  "characters": [{{"name": "...", "role": "...", "kind": "human|angel|animal|...",
     "gender": "...", "age": "...", "is_adult": true,
     "look": "15-30 words", "wardrobe": "8-15 words"}}],
  "places": [{{"name": "...", "lyric_triggers": ["lowercase words in the lyrics that call this place"],
     "named_in_lyrics": true, "look": "how it is filmed: landmarks, streets, materials, light",
     "mood": "..."}}],
  "chapters": [{{"name": "...", "lyric_cue": "first words of the opening line",
     "setting": "...", "what_happens": "..."}}],
  "motifs": ["..."],
  "hero_props": [{{"name": "...", "look": "15-30 words", "stages": ["look at stage 1", "..."]}}],
  "set_pieces": [{{"category": "short label, e.g. workshop / build / ensemble dance / phenomenon / performance",
     "what": "one sentence: who or what does what", "where": "specific place",
     "hero_visual": "the single striking image of the shot", "cast": ["names from performers/characters/props"]}}],
  "effect_palette": ["..."],
  "ending_payoff": "one sentence",
  "metaphor_map": [{{"lyric_phrase": "...", "grounded_visual": "..."}}],
  "surrealism_budget": "one line"
}}
No characters the story does not need; none it needs left out.
"""
    for _attempt in range(2):
        raw = _llm_dict(prompt, temperature=0.85, use_large=True)
        if raw.get("logline") or raw.get("set_pieces") or raw.get("characters"):
            break
        print("  [film] the LLM returned an empty film bible; retrying once...")
    else:
        raise RuntimeError(
            "Film bible came back empty twice: every scene would collapse into "
            "the same fallback shot. See the LLM errors above.")
    bible = _normalize_film_bible(raw, meta, vocals)
    perf = bible["performers"][0]
    print(f"  [film] {bible.get('title_concept') or bible.get('logline', '')}"[:200])
    print(f"  [film] {len(bible['performers'])} performer(s); lead: "
          f"{perf.get('name', '')} -- {str(perf.get('look', ''))[:110]}")
    print(f"  [film] {len(bible['characters'])} character(s): "
          + ", ".join(f"{c['key']}={c.get('name', '?')} ({c.get('role', '')})"
                      for c in bible["characters"]))
    if bible["places"]:
        print("  [film] places: " + ", ".join(p["name"] for p in bible["places"]))
    if bible["hero_props"]:
        print("  [film] props: " + ", ".join(p["name"] for p in bible["hero_props"]))
    print(f"  [film] {len(bible['set_pieces'])} set piece(s); "
          f"{len(bible['effect_palette'])} effect families.")
    return bible


def _cast_registry(bible: dict) -> dict:
    """key -> {name, role, is_adult, lock}; lock is the verbatim look text."""
    reg = {}
    for p in (bible or {}).get("performers") or []:
        reg[p["key"]] = p
    for c in (bible or {}).get("characters") or []:
        reg[c["key"]] = c
    for p in (bible or {}).get("hero_props") or []:
        reg[p["key"]] = p
    for key, e in reg.items():
        # Name + look + wardrobe only: the lock is copied into image prompts,
        # where a role sentence is noise. Planners get the role separately.
        bits = [str(e.get("name") or key)]
        if e.get("look"):
            bits.append(str(e["look"]).strip())
        wardrobe = str(e.get("wardrobe") or "").strip()
        if wardrobe and not key.startswith("prop_") and not re.match(
                r"(?i)(no |none|n/a|nothing)", wardrobe):
            bits.append(f"wearing {wardrobe}")
        e["lock"] = "; ".join(bits)
    return reg


def _cast_block(bible: dict) -> str:
    lines = ["LOCKED CAST (repeat each look verbatim whenever that character is in frame):"]
    for key, e in _cast_registry(bible).items():
        extra = ""
        if key.startswith("singer") and e.get("dance_style"):
            extra = f" | dance style: {e['dance_style']}"
        role = f" [{e['role']}]" if e.get("role") else ""
        lines.append(f"- {key}{role}: {e['lock']}{extra}")
    return "\n".join(lines)


def _bible_for_prompt(bible: dict) -> str:
    keep = ("title_concept", "logline", "story_shape", "world", "era", "tone",
            "performer_is_protagonist", "places", "chapters", "motifs",
            "ending_payoff", "metaphor_map")
    slim = {k: bible.get(k) for k in keep if bible.get(k)}
    slim["performers"] = [{k: p.get(k) for k in ("key", "name", "wardrobe_by_chapter",
                                                  "dance_style", "performance_settings")}
                          for p in bible.get("performers") or []]
    slim["hero_props"] = [{k: p.get(k) for k in ("key", "name", "stages")}
                          for p in bible.get("hero_props") or []]
    return json.dumps(slim, indent=1, ensure_ascii=False)


def _set_piece_menu(bible: dict) -> str:
    return json.dumps([{k: p.get(k) for k in ("id", "category", "what", "where")}
                       for p in bible.get("set_pieces") or []],
                      indent=0, ensure_ascii=False)


def _brief_from_film_bible(bible: dict, meta: dict) -> dict:
    """CURRENT_BRIEF in the shape _brief_block() and generate_style_bible() read."""
    performers = "; ".join(f"{p.get('name', '')}: {p.get('look', '')}"
                           for p in bible.get("performers") or [])
    return {
        "genre": bible.get("genre") or (meta or {}).get("genre", ""),
        "world": bible.get("world", ""),
        "era_and_props": bible.get("era", ""),
        "protagonist_note": (f"Performers -- {performers}. "
                             f"Story: {bible.get('logline', '')}"),
        "metaphor_map": bible.get("metaphor_map") or [],
        "surrealism_budget": bible.get("surrealism_budget") or "grounded, with earned spectacle",
        "director_notes": str((meta or {}).get("director_notes") or "").strip(),
        "must_show": _str_list((meta or {}).get("must_show"), 24),
        "image_theme": str((meta or {}).get("image_theme") or "").strip(),
        "performance_target_ratio": (meta or {}).get("performance_target_ratio"),
    }


def _timed_scenes(segments: list, audio_file: str, config: dict,
                  audio_images_dir: str, last_end_value: float) -> list:
    """The film's cut list: beat-aligned cuts seeded by the lyric starts
    (when beat_driven_cutting is on and analysis succeeds), otherwise one
    scene per merged lyric segment; then duration-normalized."""
    if segments:
        scenes = [{"start": float(t), "text": ""} for t, _ in segments]
    else:
        step = float(config.get("cut_target_mid_seconds", 8.0))
        scenes = [{"start": float(t), "text": ""}
                  for t in np.arange(0.0, max(step, float(last_end_value)), step)]
    if config.get("beat_driven_cutting", True):
        try:
            analysis = analyze_music(audio_file, config)
            with open(os.path.join(audio_images_dir, "music_analysis.json"), "w") as f:
                json.dump(analysis, f)
            cut_times = build_beat_aligned_boundaries(
                analysis, [s["start"] for s in scenes], config)
            scenes = scenes_from_cuts(cut_times, analysis, last_end_value)
        except Exception as e:
            print(f"WARNING: beat-driven cutting failed ({e}); "
                  f"keeping lyric-timed scenes.")
    pre_count = len(scenes)
    scenes = normalize_scene_durations(scenes, last_end_value, config)
    print(f"Duration normalization: {pre_count} -> {len(scenes)} scene(s).")
    return scenes


def _apply_plan(s: dict, it: dict, reg: dict, singer: str) -> None:
    stype = str(it.get("scene_type") or "").strip().lower()
    if stype not in SCENE_TYPES:
        stype = ("story" if s.get("lyric_fidelity_mode") == "literal"
                 else "performance")
    chars = [c for c in _str_list(it.get("characters"), len(reg)) if c in reg]
    if stype != "story" and not any(c.startswith("singer") for c in chars):
        chars.insert(0, singer)
    extras = str(it.get("extras") or "").strip()
    cap = int(CONFIG.get("max_cast_in_frame", 4))
    if len(chars) > cap:
        keep = ([c for c in chars if c.startswith("singer")] +
                [c for c in chars if not c.startswith("singer")])[:cap]
        overflow = [str(reg[c].get("name") or c) for c in chars if c not in keep]
        chars = [c for c in chars if c in keep]
        extras = ", ".join(x for x in [extras, "in the background: " + ", ".join(overflow)] if x)
    s["scene_type"] = stype
    s["characters"] = chars
    s["chapter"] = str(it.get("chapter") or "").strip()
    s["extras"] = extras
    s["location"] = str(it.get("location") or s.get("location") or "").strip()
    s["story_event"] = str(it.get("story_event") or s.get("lyric_meaning") or "").strip()
    s["performer_action"] = (str(it.get("performer_action") or "").strip()
                             if stype != "story" else "")
    s["wardrobe_note"] = str(it.get("wardrobe_note") or "").strip()
    s["shot_scale"] = str(it.get("shot_scale") or "medium-wide").strip()
    s["camera_angle"] = str(it.get("camera_angle") or "eye-level").strip()
    s["time_of_day"] = str(it.get("time_of_day") or "").strip()
    s["weather"] = str(it.get("weather") or "").strip()
    s["scene_heat"] = _norm_heat(it.get("scene_heat"), "none")
    s["story_cgi_hook"] = str(it.get("story_cgi_hook") or "").strip()
    s["continuity"] = str(it.get("continuity") or "").strip()
    s["set_piece"] = str(it.get("set_piece") or "").strip().lower()
    s["prop_stage"] = str(it.get("prop_stage") or "").strip()


def plan_scenes(scenes: list, bible: dict, text: str, config: dict) -> None:
    """Batched planning pass: what kind of shot each scene is, who is in it,
    where it is, and what happens. Mutates scenes in place."""
    if not scenes:
        return
    print(f"Planning scenes ({len(scenes)} shots)...")
    reg = _cast_registry(bible)
    singer = bible["performers"][0]["key"]
    batch_size = max(4, int(config.get("plan_batch_size", 16)))
    target = _performance_target(config)
    menu = _set_piece_menu(bible)
    overview = json.dumps([{
        "i": i, "t": round(float(s.get("start", 0.0)), 1),
        "energy": s.get("energy_band", "mid"),
        "lyrics": (s.get("text") or "(instrumental)")[:90],
    } for i, s in enumerate(scenes)], ensure_ascii=False)
    bible_json = _bible_for_prompt(bible)
    cast_block = _cast_block(bible)

    # Constant across batches: sent as its own message so it is cached.
    context = f"""
You are the director planning every shot of this music video, in order.

FILM BIBLE:
{bible_json}

{cast_block}

### {_brief_block()}

{_film_mandate_block()}

SET-PIECE MENU (the shots this film is cut from):
{menu}

Whole song at a glance (index, start sec, energy, lyrics):
{overview}

Decide per lyric, aiming for roughly {target:.0%} of the whole film as
"performance" or "performance_in_story":
- Lines that describe people, beings or events doing things -> "story":
  show those characters doing exactly that (the angel pulls the man from
  the wreck; the couple dances in the kitchen). Nobody sings in it.
- Choruses, hooks, first-person feeling lines with no concrete event, and
  instrumental breaks that need lift -> "performance": the performer(s)
  sing and dance, sized to energy_band (low: sway and small steps; mid:
  groove; high: the big dance break).
- When the performer's feeling and the story's event share a moment ->
  "performance_in_story".
- Never more than 3 performance shots in a row; story runs may be longer
  only while one continuous event is unfolding.
- Location: if lyric_places is not empty, the shot is THERE (Paris =
  recognizable Paris in the mood of the line; a park = a park). Otherwise
  use the chapter's setting or one of the performance settings.
  Stay put while the lyric stays put.
- Vary shot scale across neighbours; plant the motifs early and pay them
  off near the end.
- SET PIECES: every scene is built on ONE set piece from the menu. Use each
  at most twice in the film, never in two adjacent scenes, and work through
  the menu so every category appears; every MUST SHOW item in the brief is
  covered. Performance shots take a set piece too: the performer performs
  INSIDE it, never on a generic stage.
- VARIETY: neighbouring shots differ in at least two of location, subject,
  shot scale and lighting. A viewer must never see the same picture twice.
- Hero props evolve in story order: give each prop in frame its stage.

For EACH scene return:
- "index" (copied)
- "scene_type": "performance" | "story" | "performance_in_story"
- "chapter": chapter name from the bible
- "characters": the PRINCIPAL cast keys in frame (from {list(reg)}), at most
  {int(config.get("max_cast_in_frame", 4))}; crowds and ensembles beyond that go in "extras". Performance
  types include at least one performer
- "extras": "" or a short description of background people the place needs
- "location": a specific, filmable place
- "story_event": one sentence: what happens in this shot
- "performer_action": performance types only: how the performer(s) sing and
  dance here; "" for story
- "wardrobe_note": what each principal wears in this shot
- "shot_scale": "wide" | "medium-wide" | "medium" | "close-up" | "extreme close-up"
- "camera_angle": e.g. "eye-level", "low-angle", "high-angle", "overhead", "dutch tilt"
- "time_of_day", "weather"
- "scene_heat": "none" | "romantic" | "sensual" | "explicit" -- never above
  this scene's lyric_heat
- "story_cgi_hook": one sentence of story-driven VFX, or ""
- "continuity": what carries over from the previous shot
- "set_piece": the menu id this shot is built on (e.g. "sp07")
- "prop_stage": "" or, for each hero prop in frame, its stage this shot

Return ONLY a raw JSON array, one object per scene, in order.
"""

    for start in range(0, len(scenes), batch_size):
        batch = scenes[start:start + batch_size]
        planned = [{
            "i": j, "type": scenes[j].get("scene_type"),
            "set_piece": scenes[j].get("set_piece", ""),
            "where": (scenes[j].get("location") or "")[:80],
            "what": (scenes[j].get("story_event") or "")[:120],
        } for j in range(max(0, start - 4), start)]
        rows = [{
            "index": start + k,
            "start_sec": round(float(s.get("start", 0.0)), 1),
            "duration_sec": round(float(s.get("_duration", 0.0)), 1),
            "energy_band": s.get("energy_band", "mid"),
            "lyrics": (s.get("text") or "").strip(),
            "lyric_before": (s.get("lyric_before") or "")[:120],
            "lyric_after": (s.get("lyric_after") or "")[:120],
            "lyric_meaning": s.get("lyric_meaning", ""),
            "lyric_visual_cues": (s.get("lyric_visual_cues") or [])[:4],
            "lyric_atmosphere": s.get("lyric_atmosphere", ""),
            "lyric_places": s.get("lyric_places") or [],
            "lyric_heat": s.get("lyric_heat", "none"),
            "mode": s.get("lyric_fidelity_mode", ""),
        } for k, s in enumerate(batch)]
        prompt = f"""
Shots already planned just before this batch (keep continuity):
{json.dumps(planned, ensure_ascii=False)}

Scenes to plan:
{json.dumps(rows, indent=1, ensure_ascii=False)}
"""
        got = _by_index(_llm_items(prompt, temperature=0.75, use_large=True,
                                   context=context))
        missing = 0
        for k, s in enumerate(batch):
            it = got.get(start + k)
            if not it:
                missing += 1
                it = {}
            _apply_plan(s, it, reg, singer)
        if missing:
            print(f"  [plan] {missing} scene(s) in batch @{start} got a default plan.")

    types = [s.get("scene_type") for s in scenes]
    perf = sum(1 for t in types if t != "story") / max(1, len(types))
    print(f"  [plan] story={types.count('story')} performance={types.count('performance')} "
          f"performance_in_story={types.count('performance_in_story')} "
          f"(performers on screen {perf:.0%}, target ~{target:.0%})")


def _performance_target(config: dict) -> float:
    """Share of shots with the performer singing: song_metadata wins."""
    t = (CURRENT_BRIEF or {}).get("performance_target_ratio")
    if t in (None, ""):
        t = config.get("performance_target_ratio", 0.35)
    return float(t)


def _adopt_set_piece(s: dict, sp: dict, name_to_key: dict) -> None:
    """Rebuild a scene around set piece sp, keeping its type and performer."""
    s["set_piece"] = sp["id"]
    s["location"] = str(sp.get("where") or s.get("location") or "").strip()
    s["story_event"] = str(sp.get("what") or s.get("story_event") or "").strip()
    keys = []
    for name in sp.get("cast") or []:
        n = name.lower()
        key = next((k for nm, k in name_to_key.items()
                    if nm and (nm in n or n in nm)), None)
        if key and key not in keys:
            keys.append(key)
    performers = [c for c in s.get("characters") or [] if c.startswith("singer")]
    if s.get("scene_type") == "story":
        s["characters"] = keys or s.get("characters") or []
    else:
        s["characters"] = performers + [k for k in keys if k not in performers]
    s["continuity"] = ""


def enforce_visual_variety(scenes: list, bible: dict, config: dict) -> None:
    """Hard guard against the every-shot-is-the-same-picture failure: each
    scene stands on a set piece, none is used more than set_piece_max_uses
    times or twice in a row, performers stay near their target share, no
    long runs of plain performance shots, and shot scale keeps changing."""
    pieces = {p["id"]: p for p in bible.get("set_pieces") or []}
    order = list(pieces)
    reg = _cast_registry(bible)
    name_to_key = {str(e.get("name") or "").lower(): k for k, e in reg.items()}
    n = len(scenes)
    reassigned = converted = rescaled = 0

    # 1. Performer share: demote the excess, instrumental shots first.
    target = _performance_target(config)
    perf_idx = [i for i, s in enumerate(scenes) if s.get("scene_type") != "story"]
    excess = len(perf_idx) - int(round((target + 0.1) * n))
    for i in sorted(perf_idx, key=lambda i: (bool(scenes[i].get("has_lyrics")),
                                             scenes[i].get("energy_band") == "low")):
        if excess <= 0:
            break
        scenes[i]["scene_type"] = "story"
        scenes[i]["performer_action"] = ""
        excess -= 1
        converted += 1

    # 2. Set pieces: valid, not overused, never the same twice in a row.
    if pieces:
        max_uses = int(config.get("set_piece_max_uses", 2))
        uses = {p: 0 for p in pieces}
        prev = None
        for i, s in enumerate(scenes):
            pid = s.get("set_piece")
            if pid not in pieces or uses[pid] >= max_uses or pid == prev:
                want = i / max(1, n - 1) * (len(order) - 1)
                prev_cat = pieces[prev].get("category") if prev in pieces else None
                cands = sorted((p for p in order if p != prev),
                               key=lambda p: (uses[p],
                                              pieces[p].get("category") == prev_cat,
                                              abs(order.index(p) - want)))
                if cands:
                    _adopt_set_piece(s, pieces[cands[0]], name_to_key)
                    pid = cands[0]
                    reassigned += 1
            if pid in uses:
                uses[pid] += 1
            prev = pid
        unused = [p for p, u in uses.items() if not u]
        if unused:
            print(f"  [variety] {len(unused)} set piece(s) never used: {', '.join(unused)}")
    else:
        print("  [variety] WARNING: film bible has no set pieces; variety rests on the planner.")

    # 3. No long runs of plain performance shots.
    run = 0
    for s in scenes:
        run = run + 1 if s.get("scene_type") == "performance" else 0
        if run > int(config.get("max_performance_run", 2)):
            s["scene_type"] = "performance_in_story"
            converted += 1
            run = 0

    # 4. Shot scale never holds for three shots.
    scales = ["wide", "medium", "close-up", "medium-wide", "extreme close-up"]
    for i in range(2, n):
        a, b, cur = (scenes[i - 2].get("shot_scale"), scenes[i - 1].get("shot_scale"),
                     scenes[i].get("shot_scale"))
        if a == b == cur:
            scenes[i]["shot_scale"] = next(x for x in scales if x != cur)
            rescaled += 1

    types = [s.get("scene_type") for s in scenes]
    print(f"  [variety] {reassigned} set piece(s) reassigned, {converted} shot "
          f"type(s) changed, {rescaled} shot scale(s) changed; story="
          f"{types.count('story')} performance={types.count('performance')} "
          f"performance_in_story={types.count('performance_in_story')}")


def enforce_lyric_places(scenes: list, bible: dict) -> None:
    """Hard guard: a lyric that names a place puts the shot in that place."""
    fixed = 0
    places = bible.get("places") or []
    for s in scenes:
        loc = s.get("location") or ""
        loc_l = loc.lower()
        lyric = " ".join([s.get("text") or ""] + list(s.get("lyric_places") or [])).lower()
        if not lyric.strip():
            continue
        target = None
        matched = False
        for p in places:
            if any(re.search(r"\b" + re.escape(t) + r"\b", lyric)
                   for t in p["lyric_triggers"] if t):
                matched = True
                if p["name"].lower() not in loc_l and not any(
                        t in loc_l for t in p["lyric_triggers"]):
                    target = f"{p['name']} -- {p.get('look') or loc}".strip(" -")
                break
        if not matched:
            # A place the lyric names that the bible did not list.
            for lp in s.get("lyric_places") or []:
                head = lp.lower().replace("the ", "").strip()
                if head and head not in loc_l and lp.lower() not in loc_l:
                    target = f"{lp}, {loc}" if loc else lp
                break
        if target:
            s["location"] = target[:220]
            fixed += 1
    if fixed:
        print(f"  [places] moved {fixed} scene(s) into the place their lyric names.")


def gate_scene_heat(scenes: list, bible: dict, config: dict) -> None:
    """scene_heat <= min(the scene's own lyric_heat, CONFIG max_scene_heat);
    anyone under 18 in frame -> none."""
    reg = _cast_registry(bible)
    ceiling = _norm_heat(config.get("max_scene_heat"), "explicit")
    lowered = 0
    for s in scenes:
        planned = _norm_heat(s.get("scene_heat"), "none")
        cap = _norm_heat(s.get("lyric_heat"), "none")
        if _heat_rank(cap) > _heat_rank(ceiling):
            cap = ceiling
        if any(not reg.get(c, {}).get("is_adult", True) for c in s.get("characters") or []):
            cap = "none"
        heat = planned if _heat_rank(planned) <= _heat_rank(cap) else cap
        if heat != planned:
            lowered += 1
        s["scene_heat"] = heat
    hot = sum(1 for s in scenes if _heat_rank(s.get("scene_heat")) >= _heat_rank("sensual"))
    print(f"  [heat] {hot} sensual/explicit scene(s); {lowered} lowered to "
          f"match their lyrics.")


def _fallback_prompts(s: dict, reg: dict, bible: dict = None) -> None:
    who = "; ".join(reg[c]["lock"] for c in s.get("characters") or [] if c in reg)
    stype = s.get("scene_type", "story")
    act = s.get("performer_action") if stype != "story" else s.get("story_event")
    sp = next((p for p in (bible or {}).get("set_pieces") or []
               if p.get("id") == s.get("set_piece")), {})
    if not s.get("scene_description"):
        when = ", ".join(x for x in (s.get("time_of_day"), s.get("weather")) if x)
        cues = "; ".join(s.get("lyric_visual_cues") or [])
        parts = [f"{s.get('shot_scale') or 'Medium-wide'} shot, "
                 f"{s.get('camera_angle') or 'eye-level'}.",
                 str(sp.get("hero_visual") or "").strip(), f"{who}." if who else "",
                 str(act or s.get("lyric_meaning") or "").strip(),
                 f"{cues}." if cues else "",
                 f"Setting: {s.get('location') or sp.get('where') or (bible or {}).get('world') or 'the story world'}"
                 + (f", {when}." if when else ".")]
        s["scene_description"] = " ".join(p for p in parts if p)
    if not s.get("motion"):
        if stype == "story":
            s["motion"] = (f"{s.get('story_event') or 'The characters act out the moment.'} "
                           "The camera tracks the action with medium amplitude. Nobody sings.")
        else:
            s["motion"] = (f"The performer sings to camera and dances on the beat: "
                           f"{s.get('performer_action') or 'stepping and swaying with the groove'}. "
                           "The camera pushes in slowly.")


def write_scene_prompts(scenes: list, bible: dict, style_bible: dict,
                              config: dict, indices: list = None,
                              notes: dict = None) -> None:
    """Batched writer: the reference-still prompt and the H3 motion prompt."""
    reg = _cast_registry(bible)
    indices = list(range(len(scenes))) if indices is None else list(indices)
    notes = notes or {}
    if not indices:
        return
    print(f"Writing scene prompts ({len(indices)} scene(s))...")
    batch_size = max(2, int(config.get("write_batch_size", 8)))
    grammar = "\n".join(_camera_grammar_block(b) for b in ("low", "mid", "high"))
    pieces = {p["id"]: p for p in bible.get("set_pieces") or []}
    fallbacks = 0

    # Constant across batches (and the audit rewrite pass): sent as its own
    # message so it is cached.
    context = f"""
You are the cinematographer writing the final generation prompts for a
music video. Each scene becomes ONE reference still (text-to-image, no
memory of other shots) and ONE animation of that still driven by the song.

FILM: {bible.get("logline", "")}
Tone: {bible.get("tone", "")}. Motifs: {", ".join(bible.get("motifs") or [])}.

### {_style_block(style_bible)}

{_direction_block(CURRENT_BRIEF)}

{FILM_MANDATE}

CAMERA REGISTERS BY ENERGY:
{grammar}

{FORMAT_EXAMPLES}

For EACH scene write:
- "scene_description": the reference STILL, 90-160 words, never more. Open with shot
  scale and angle. Name every principal in frame and describe each with
  their locked look and this shot's wardrobe (copy the lock phrases -- the
  image model has never seen them). Put the location's recognizable proof
  in frame (landmark, architecture, landscape, vehicle, weather). Freeze
  the story_event at its most telling instant. Performance shots: the
  performer(s) mid-song (mouth open on a note) and mid-dance-move. Include time
  of day, weather and lighting, and the scene's visual_effect (if any) as
  part of the picture. State what each character is wearing --
  or how undressed they are -- exactly as the story and scene_heat call
  for. Describe only what is in the picture; no readable text, signs or
  captions.
  The scene's set_piece.hero_visual IS the subject of the frame; build the
  picture around it. Hero props in frame get their locked look at this
  shot's prop_stage. Never repeat the subject or composition of
  previous_description; never fall back to a performer standing on a stage.
- "motion": the animation prompt, 40-110 words. Exactly ONE dominant
  camera move from this scene's energy register, written as a natural
  clause. Describe what changes from the start to the end of the shot,
  with movement landing on the beat. "performance": the performer(s) sing
  the line to camera and dance (name the moves). "performance_in_story":
  the performer sings while the story action happens around them. "story":
  the characters act it out; end with "Nobody sings."
- "emotion": 1-3 words.
Heat: write each scene at exactly its scene_heat as defined in the FILM
MANDATE -- no hotter, and no tamer when the lyric is explicit.
If a scene has a "fix_note", the previous version failed review: fix exactly that.

Return ONLY a raw JSON array:
[{{"index": 0, "scene_description": "...", "motion": "...", "emotion": "..."}}]
"""

    for k in range(0, len(indices), batch_size):
        idxs = indices[k:k + batch_size]
        rows = []
        for i in idxs:
            s = scenes[i]
            prev = scenes[i - 1] if i > 0 else {}
            rows.append({
                "index": i,
                "scene_type": s.get("scene_type"),
                "lyrics": (s.get("text") or "").strip(),
                "lyric_meaning": s.get("lyric_meaning", ""),
                "energy_band": s.get("energy_band", "mid"),
                "scene_heat": s.get("scene_heat", "none"),
                "characters": [{"key": c, "lock": reg[c]["lock"]}
                               for c in s.get("characters") or [] if c in reg],
                "extras": s.get("extras", ""),
                "location": s.get("location", ""),
                "chapter": s.get("chapter", ""),
                "story_event": s.get("story_event", ""),
                "performer_action": s.get("performer_action", ""),
                "wardrobe_note": s.get("wardrobe_note", ""),
                "shot_scale": s.get("shot_scale", ""),
                "camera_angle": s.get("camera_angle", ""),
                "time_of_day": s.get("time_of_day", ""),
                "weather": s.get("weather", ""),
                "visual_effect": (f"{s.get('cgi_effect_family')} "
                                  f"({s.get('cgi_effect_intensity') or 'moderate'})"
                                  if s.get("cgi_magic_enabled") and s.get("cgi_effect_family")
                                  else ""),
                "continuity": s.get("continuity", ""),
                "set_piece": {k: pieces[s["set_piece"]].get(k)
                              for k in ("category", "what", "where", "hero_visual")}
                             if s.get("set_piece") in pieces else {},
                "prop_stage": s.get("prop_stage", ""),
                "previous_shot": (prev.get("story_event") or "")[:140],
                "previous_description": (prev.get("scene_description") or "")[:200],
                "fix_note": notes.get(i, ""),
            })
        prompt = f"""
Scenes:
{json.dumps(rows, indent=1, ensure_ascii=False)}
"""
        got = _by_index(_llm_items(prompt, temperature=0.75, use_large=True,
                                   context=context))
        for i in idxs:
            s = scenes[i]
            had_desc = bool(s.get("scene_description"))
            it = got.get(i) or {}
            desc = str(it.get("scene_description") or "").strip()
            motion = str(it.get("motion") or "").strip()
            if desc:
                s["scene_description"] = desc
            if motion:
                s["motion"] = motion
            if it.get("emotion"):
                s["emotion"] = str(it["emotion"]).strip()
            if not (desc and motion):
                print(f"  [write] scene {i + 1}: using fallback prompt text.")
                if not had_desc:
                    fallbacks += 1
            _fallback_prompts(s, reg, bible)
    ratio = fallbacks / max(1, len(indices))
    if ratio > float(config.get("max_fallback_ratio", 0.25)):
        raise RuntimeError(
            f"{fallbacks}/{len(indices)} scene prompts fell back to template "
            f"text -- the LLM is failing, and rendering now would produce "
            f"near-identical stills. See the LLM errors above.")


def audit_scenes(scenes: list, bible: dict, style_bible: dict,
                       config: dict) -> None:
    """LLM judge: lyric fidelity, literal places, cast locks, sing-vs-act,
    heat. Scenes under audit_min_score are rewritten once."""
    if not scenes or not config.get("audit_enabled", True):
        return
    print(f"Auditing scenes ({len(scenes)})...")
    min_score = int(config.get("audit_min_score", 3))
    batch_size = max(4, int(config.get("audit_batch_size", 12)))
    reg = _cast_registry(bible)
    notes = {}
    scores = []
    # Constant across batches: sent as its own message so it is cached.
    context = f"""
You are the music-video editor reviewing shots before they are rendered.
FILM: {bible.get("logline", "")}

{_direction_block(CURRENT_BRIEF)}

Score each shot 0-5 against ALL of these:
1. Lyric fidelity: a story shot SHOWS what its lyric describes; a
   performance shot fits the feeling of the line.
2. Place: if lyric_places names a place, the still is recognizably there.
3. Cast: every character in "characters" is described with their locked
   look; performance shots show the performer(s) singing AND dancing;
   story shots show characters acting and nobody singing.
4. Heat: content never sexier than scene_heat; scene_heat never above
   lyric_heat.
5. Craft: specific, vivid, filmable; one camera move; one unbroken frame.
6. Variety and brief: the shot does not repeat the subject or composition
   of previous_description, and serves the director's brief below. A
   generic performer-on-a-stage picture scores at most 2.
For any score below {min_score}, "fix" must be one concrete instruction.

Return ONLY a raw JSON array: [{{"index": 0, "score": 4, "fix": ""}}]
"""

    for start in range(0, len(scenes), batch_size):
        batch = scenes[start:start + batch_size]
        rows = [{
            "index": start + k,
            "scene_type": s.get("scene_type"),
            "lyrics": (s.get("text") or "")[:200],
            "lyric_meaning": (s.get("lyric_meaning") or "")[:200],
            "lyric_places": s.get("lyric_places") or [],
            "lyric_heat": s.get("lyric_heat", "none"),
            "scene_heat": s.get("scene_heat", "none"),
            "location": s.get("location", ""),
            "characters": {c: reg[c]["lock"][:160] for c in s.get("characters") or [] if c in reg},
            "scene_description": (s.get("scene_description") or "")[:900],
            "motion": (s.get("motion") or "")[:500],
            "previous_description": ((scenes[start + k - 1].get("scene_description") or "")[:250]
                                     if start + k > 0 else ""),
        } for k, s in enumerate(batch)]
        prompt = f"""
Shots:
{json.dumps(rows, indent=1, ensure_ascii=False)}
"""
        for it in _llm_items(prompt, temperature=0.2, use_large=True,
                             context=context):
            if not isinstance(it.get("index"), int):
                continue
            try:
                score = int(it.get("score"))
            except (TypeError, ValueError):
                continue
            scores.append(score)
            if score < min_score and 0 <= it["index"] < len(scenes):
                notes[it["index"]] = str(it.get("fix") or "Show the lyric more clearly.").strip()
    if scores:
        print(f"  [audit] mean score {sum(scores) / len(scores):.2f}; "
              f"{len(notes)} scene(s) below {min_score} -> rewriting.")
    if notes:
        write_scene_prompts(scenes, bible, style_bible, config,
                                  indices=sorted(notes), notes=notes)


def author_scenes(audio_file: str, config: dict, ctx: dict):
    """Author every scene of the song. ctx carries the transcription and the
    per-song paths computed in process_audio_scenes(); returns the same
    tuple as process_audio_scenes()."""
    global CURRENT_BRIEF
    text_for_llm = ctx["text_for_llm"]
    audio_images_dir = ctx["audio_images_dir"]
    audio_videos_dir = ctx["audio_videos_dir"]
    last_end_value = ctx["last_end_value"]
    timestamp = ctx["timestamp"]
    meta = ctx.get("meta") or {}
    content = ctx.get("content") or {}
    vocals = ctx.get("vocals") or {}
    scenes_file_path = os.path.join(audio_images_dir, "scenes.json")

    film_bible = generate_film_bible(text_for_llm, meta, content, vocals, config)
    brief = _brief_from_film_bible(film_bible, meta)
    CURRENT_BRIEF = brief
    style_bible = generate_style_bible(text_for_llm, config)
    with open(os.path.join(audio_images_dir, "entity_style_bible.json"), "w") as f:
        json.dump({"style_bible": style_bible,
                   "film_bible": film_bible, "creative_brief": brief,
                   "content": {**content, "vocals": vocals}},
                  f, indent=2)

    scenes = _timed_scenes(ctx.get("segments") or [], audio_file, config,
                           audio_images_dir, last_end_value)
    restamp_scene_lyrics(scenes, ctx.get("raw_segments") or [], last_end_value, config)
    generate_lyric_interpretation(scenes, text_for_llm, config)
    for s in scenes:
        s.setdefault("energy_band", "mid")

    plan_scenes(scenes, film_bible, text_for_llm, config)
    enforce_visual_variety(scenes, film_bible, config)
    enforce_lyric_places(scenes, film_bible)
    gate_scene_heat(scenes, film_bible, config)
    assign_cgi_magic_direction(scenes, config, film_bible.get("effect_palette"))
    write_scene_prompts(scenes, film_bible, style_bible, config)
    audit_scenes(scenes, film_bible, style_bible, config)
    print(f"  [llm] {LLM_STATS['calls']} call(s), {LLM_STATS['failures']} failed; "
          f"{LLM_STATS['cached_tokens']:,} of {LLM_STATS['prompt_tokens']:,} "
          f"prompt tokens served from cache.")

    with open(os.path.join(audio_images_dir, "lyric_interpretation.json"), "w") as f:
        json.dump([{"index": i, "start": s.get("start"), **_lyric_fields(s)}
                   for i, s in enumerate(scenes)], f, indent=2)
    with open(scenes_file_path, "w") as f:
        json.dump(scenes, f, indent=2)
    print(f"Successfully processed {len(scenes)} scenes.")
    return scenes, audio_images_dir, audio_videos_dir, last_end_value, timestamp


def transcribe_song(audio_file: str, config: dict) -> dict:
    """Whisper sliding-window transcription plus this song's output folders.
    Returns the context dict both scene pipelines consume."""
    audio_basename = os.path.splitext(os.path.basename(audio_file))[0]
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    unique_id = f"{audio_basename}_{timestamp}"

    audio_images_dir = os.path.join(config["base_working_dir"], unique_id)
    audio_videos_dir = os.path.join(config["base_video_dir"], unique_id)
    os.makedirs(audio_images_dir, exist_ok=True)
    os.makedirs(audio_videos_dir, exist_ok=True)

    print("Transcribing audio using Whisper (sliding window)...")
    processor, whisper_model = get_audio_pipe()
    full_audio, sampling_rate = load_audio_mono_16k(audio_file)
    total_duration = len(full_audio) / sampling_rate
    print(f"Total audio duration: {total_duration:.2f}s")

    segments = []
    chunk_length = 30
    overlap = 5
    step = chunk_length - overlap

    for start_sec in range(0, int(total_duration), step):
        end_sec = min(start_sec + chunk_length, total_duration)
        print(f" > Processing chunk: {start_sec}s to {end_sec}s")
        chunk_audio = full_audio[int(start_sec * sampling_rate): int(end_sec * sampling_rate)]
        inputs = processor(chunk_audio, sampling_rate=16000,
                           return_tensors="pt", return_attention_mask=True)
        input_features = inputs.input_features.to("cuda").to(dtype)
        attention_mask = inputs.attention_mask.to("cuda")
        with torch.no_grad():
            predicted_ids = whisper_model.generate(
                input_features, attention_mask=attention_mask,
                return_timestamps=True, task="transcribe",
                max_new_tokens=444, no_speech_threshold=0.6,
                logprob_threshold=-1.0, compression_ratio_threshold=2.4,
                condition_on_prev_tokens=False, temperature=0.0,
            )
        chunk_text = processor.decode(predicted_ids[0], skip_special_tokens=False,
                                      decode_with_timestamps=True)
        raw_chunks = re.findall(r"<\|(\d+\.\d+)\|>(.*?)(?=<\|\d+\.\d+\|>|$)", chunk_text)
        for ts_str, txt in raw_chunks:
            clean_txt = txt.strip()
            relative_ts = float(ts_str)
            if start_sec + chunk_length < total_duration and relative_ts >= step:
                continue
            words = clean_txt.split()
            if len(words) > 6:
                trigram = " ".join(words[:3]).lower()
                if clean_txt.lower().count(trigram) > 2:
                    print(f"   > Skipping hallucinated segment at {relative_ts + start_sec:.1f}s")
                    continue
            if clean_txt:
                segments.append((relative_ts + start_sec, clean_txt))

    del processor, whisper_model, full_audio
    reset_memory(device)

    segments.sort(key=lambda x: x[0])
    raw_segments = segments.copy()
    segments = merge_transcript_segments(raw_segments, total_duration, config)
    ctx = {
        "segments": segments, "raw_segments": raw_segments,
        "text_for_llm": _text_for_llm(segments),
        "last_end_value": float(total_duration), "timestamp": timestamp,
        "audio_images_dir": audio_images_dir, "audio_videos_dir": audio_videos_dir,
    }
    _save_transcript(ctx)
    return ctx


def merge_transcript_segments(raw_segments: list, total_duration: float,
                              config: dict) -> list:
    """Merge Whisper lines until each spans at least h3_min_scene_seconds."""
    min_seconds = float(config["h3_min_scene_seconds"])
    merged = []
    if raw_segments:
        current_text = []
        current_start = raw_segments[0][0]
        for idx, (ts, txt) in enumerate(raw_segments):
            current_text.append(txt)
            is_last = (idx == len(raw_segments) - 1)
            next_ts = raw_segments[idx + 1][0] if not is_last else total_duration
            if next_ts - current_start >= min_seconds or is_last:
                merged.append((current_start, " ".join(current_text)))
                current_start = next_ts
                current_text = []
    return merged


def _text_for_llm(segments: list) -> str:
    return "".join(f"Start: {s:.2f}, Text: {t}\n" for s, t in segments)


def _save_transcript(ctx: dict) -> None:
    """transcript.json lets reauthor_video() skip Whisper next time."""
    with open(os.path.join(ctx["audio_images_dir"], "transcript.json"), "w") as f:
        json.dump({"raw_segments": ctx["raw_segments"],
                   "duration": ctx["last_end_value"]}, f, indent=1)


def clean_transcript(ctx: dict, meta: dict, config: dict) -> None:
    """Fix Whisper mishearings before anything reads the lyrics. With the
    real lyrics (song_metadata "lyrics") each line is snapped to the lyric it
    is; without them only clear mishearings are fixed and stock Whisper
    hallucinations are dropped. Timestamps never change."""
    raw = ctx.get("raw_segments") or []
    if not raw or not config.get("transcript_cleanup", True):
        return
    lyrics = str((meta or {}).get("lyrics") or "").strip()
    rows = [{"index": i, "t": round(float(ts), 1), "heard": txt}
            for i, (ts, txt) in enumerate(raw)]
    reference = (f"The REAL lyrics (authoritative): replace each heard line "
                 f"with the lyric line(s) actually sung there.\n{lyrics[:8000]}"
                 if lyrics else
                 "No reference lyrics: fix only CLEAR mishearings -- words that "
                 "make no sense where a near-homophone fits the song's subject, "
                 "rhyme and meter. Otherwise keep the heard text exactly.")
    prompt = f"""
You are correcting a Whisper transcript of a song before a music video is
built from it.
Song subject hints: {(meta or {}).get("director_notes", "")[:600]}

{reference}

Set "drop": true for lines that are Whisper hallucinations rather than
singing (e.g. "Thank you for watching", "I'm sorry.", "Subtitles by...",
a lone stray word over an instrumental), and for exact repeats caused by
overlapping windows.

Return ONLY a raw JSON array, one object per line, in order:
[{{"index": 0, "text": "corrected line", "drop": false}}]

Lines:
{json.dumps(rows, indent=0, ensure_ascii=False)}
"""
    got = _by_index(_llm_items(prompt, temperature=0.1, use_large=True,
                               system_message="You are a meticulous lyric "
                               "transcriber. You return only valid JSON."))
    if not got:
        print("  [transcript] cleanup returned nothing; keeping Whisper text.")
        return
    fixed, changed, dropped = [], 0, 0
    for i, (ts, txt) in enumerate(raw):
        it = got.get(i)
        if it is None:
            fixed.append((ts, txt))
            continue
        new = str(it.get("text") or "").strip()
        if it.get("drop") is True or not new:
            dropped += 1
            continue
        if new != txt:
            changed += 1
            print(f"    [transcript] {ts:6.1f}s  {txt!r} -> {new!r}")
        fixed.append((ts, new))
    ctx["raw_segments"] = fixed
    ctx["segments"] = merge_transcript_segments(fixed, ctx["last_end_value"], config)
    ctx["text_for_llm"] = _text_for_llm(ctx["segments"])
    print(f"  [transcript] {changed} line(s) corrected, {dropped} dropped.")
    if ctx.get("audio_images_dir"):
        with open(os.path.join(ctx["audio_images_dir"], "transcript_clean.json"), "w") as f:
            json.dump({"raw_segments": fixed, "duration": ctx["last_end_value"]}, f, indent=1)


def _resolve_vocals(audio_file: str, config: dict, meta: dict,
                    prior: dict = None) -> dict:
    """song_metadata "voice" wins, then a prior run's detection, then CLAP."""
    voice = str((meta or {}).get("voice") or "").strip().lower()
    if voice:
        print(f"[vocals] using song_metadata voice '{voice}' (CLAP skipped)")
        return {"vocal": voice, "confidence": 1.0, "source": "song_metadata"}
    if prior:
        return prior
    return detect_vocals_clap(audio_file, config)


def process_audio_scenes(audio_file: str, config: dict):
    """Transcribe the song, read its lyrics, and author every scene.
    Returns (scenes, audio_images_dir, audio_videos_dir, last_end_value,
    timestamp)."""
    llm_preflight()
    ctx = transcribe_song(audio_file, config)
    # Genre: CONFIG/song_metadata override wins; otherwise CLAP fills it
    # (hybrid when the runner-up is close).
    ctx["meta"] = resolve_song_genre(audio_file, config)
    clean_transcript(ctx, ctx["meta"], config)
    ctx["content"] = classify_song_content(ctx["text_for_llm"], config)
    ctx["vocals"] = _resolve_vocals(audio_file, config, ctx["meta"])
    return author_scenes(audio_file, config, ctx)


def load_style_bible(audio_images_dir: str) -> dict:
    """Read the persisted style bible (DEFAULT_STYLE_BIBLE if missing)."""
    try:
        with open(os.path.join(audio_images_dir, "entity_style_bible.json")) as f:
            return json.load(f).get("style_bible") or DEFAULT_STYLE_BIBLE
    except Exception:
        return DEFAULT_STYLE_BIBLE


# =============================================================================
# Z-Image still generation — exactly ONE reference image per SCENE
# =============================================================================
def process_audio_images(config: dict, scenes, audio_images_dir):
    """
    Generate one reference still per scene, pure txt2img. Character
    consistency comes from the locked look text the scene writer repeats
    verbatim in every scene_description.
    """
    if not scenes:
        print("No scenes produced; aborting still generation.")
        return audio_images_dir
    print("Loading Krea-2 pipeline and generating scene reference images...")
    print(f"  Krea stills: single {WIDTH}x{HEIGHT} pass, anti-duplicate="
          f"{config.get('krea_anti_duplicate', True)}")
    pipe = load_KREA2_model()
    resolution = (HEIGHT, WIDTH)

    style_prefix_base = _style_bible_image_prefix(load_style_bible(audio_images_dir))

    for scene_num, scene in enumerate(scenes, 1):
        out_path = os.path.join(audio_images_dir, f"image_{str(scene_num).zfill(2)}.jpg")
        if config.get("krea_skip_existing", False) and os.path.exists(out_path):
            print(f"Scene {scene_num}/{len(scenes)}: reuse {out_path}")
            continue
        scene_description = scene.get("scene_description") or "an animated scene in a music video"
        when = ", ".join(x for x in (scene.get("time_of_day"), scene.get("weather")) if x)
        style_prefix = ", ".join(x for x in (style_prefix_base, when) if x)
        # The writer already put place, attire and any visual effect into
        # scene_description; only the film's visual style is added.
        image_prompt = scene_description
        print(f"Scene {scene_num}/{len(scenes)}: "
              f"[{scene.get('energy_band') or 'mid'}|{scene.get('scene_type') or 'story'}] "
              f"{scene_description[:80]}...")
        image, _ = generate_image(
            pipe, image_prompt, resolution, -1, style_prefix,
            config=config, scene=scene,
        )
        image.save(out_path, dpi=(300, 300))

    del pipe
    reset_memory(device)
    print(f"VRAM after Z-Image release: "
          f"{torch.cuda.memory_allocated() / 1e9:.1f} GB alloc, "
          f"{torch.cuda.memory_reserved() / 1e9:.1f} GB reserved")


# =============================================================================
# H3 prompt assembly -- pure text. Each scene renders in exactly ONE Wan2GP
# call from ONE reference image (<Picture 1>) and ONE audio reference
# (<Audio 1>). ref2va's video_length is NOT limited by the audio reference's
# length -- the audio reference just sets tone/rhythm; H3 generates video
# (and its own audio, which we discard) for the full requested video_length.
# =============================================================================
def assemble_h3_scene_prompt(scene: dict, config: dict) -> str:
    """The single-scene H3 prompt: keep the cast and place of <Picture 1>,
    sing-and-dance only in performance shots, then the scene's motion."""
    stype = str(scene.get("scene_type") or "story")
    body = (scene.get("motion") or "").strip()
    if not body:
        body = ("The characters shown in the reference image carry the scene's "
                "action forward as the camera slowly pushes in.")
    if stype == "performance":
        role = ("The performer sings this part of the song to camera with "
                "natural lip movement and dances on the beat; the movement "
                "grows with the music.")
    elif stype == "performance_in_story":
        role = ("The performer sings with natural lip movement while the story "
                "action happens around them; only the performer sings.")
    else:
        role = ("This is a story shot: nobody sings or lip-syncs; the "
                "characters act, react and move with purpose.")
    fx = ""
    if config.get("cgi_magic_enabled", True) and scene.get("cgi_magic_enabled", True):
        fx = (f" CGI movie-magic: {scene.get('cgi_effect_family','')} at "
              f"{scene.get('cgi_effect_intensity','moderate')} intensity, a "
              "physical consequence of the action with coherent depth and "
              "light interaction; keep faces, hands and the story action readable.")
    anchor = (
        "<Picture 1> establishes the characters, their faces, bodies and "
        "wardrobe, the setting, and the visual style for this entire scene; "
        "preserve every identity throughout, keep each character appearing "
        "once, wardrobe changing only as the action describes, no new "
        "principal characters, no jump to another place. <Audio 1> is the "
        "actual music under this shot -- movement lands on its pulse, swells "
        f"and drops. {role}"
    ) + fx
    prompt = f"{anchor} {body}".strip()
    return _truncate_prompt(prompt, max_words=config.get("wgp_prompt_max_words", 350))


# =============================================================================
# Scene job assembly -- computes everything a render needs (prompt, still
# image path, audio reference path, frame count) WITHOUT rendering anything.
# One job per SCENE: a single ref2va call's video_length covers the whole
# scene, chosen from the scene's real span on the song timeline and
# independent of the (<=15s) audio reference clip's own length.
# =============================================================================
def build_scene_jobs(config: dict, scenes: list, audio_file: str,
                     audio_images_dir: str, audio_videos_dir: str,
                     timestamp: str, last_end_value: float,
                     start_scene: int = 0,
                     skip_existing_output: bool = True) -> list:
    """
    Returns a flat list of job dicts, one per renderable scene, each with:
      scene_index, prompt, image_ref, audio_ref (<=15s tone reference),
      audio_full (the scene's ENTIRE span of song audio, for gapless manual
      re-sync), num_frames (grid-valid length actually rendered by H3),
      trim_frames (exact length the clip is cut back to after render), seed,
      output_path, and (on reuse) "_rendered".

    FRAME-COUNT ALGORITHM (per scene, driven by the audio):
      target_frames = round(span_seconds * fps)   -- the exact length that
        matches the audio this scene covers, with NO padding. This is what
        the clip is trimmed back to after rendering.
      render_frames = smallest H3-grid-valid length >= target_frames
        (H3 can only emit lengths on its latent grid, frames_offset +
        frames_steps * k -- it cannot render an arbitrary length).
      trim = render_frames - target_frames frames are cut off the TAIL
        after H3 produces the clip, so the kept
        clip is exactly target_frames long -- e.g. target=300, H3 needs a
        grid length of 324, so we render 324 and trim 24 frames (1.0s at
        24fps) off the end.
    This guarantees each KEPT clip matches its audio span exactly. NOTE: we
    deliberately do NOT add an extra "+1 initial frame" to target_frames --
    doing so would make every kept clip 1 frame (~0.04s at 24fps) longer
    than its audio, which conflicts with matching the audio exactly. If you
    want a literal one-frame pad reserved before motion starts, that's a
    prompt/stylistic concern for H3, not a timing one -- flag it and this
    can be reinstated for just the very first scene.

    TIMELINE COVERAGE:
    A running frame CURSOR advances by each scene's EXACT span (trim_frames),
    never by the grid-inflated render length, so per-scene grid padding can
    never shift a later boundary or accumulate into drift. The cursor STARTS
    at the first renderable scene's start time -- which MUST be 0.0 when
    rendering the whole song (start_scene == 0). If it isn't (beat-driven
    cutting failed/disabled and fell back to lyric timestamps leaving an
    intro uncovered, or the first scene's reference image is missing so it
    drops out of `renderable`), that leading span would never be rendered,
    so coverage is forced back to 0.0 with a loud warning. This repeats
    normalize_scene_durations()'s guard because resume_video() can run this
    against an older scenes.json.

    The audio REFERENCE handed to H3 is still capped to
    wgp_audio_ref_max_seconds (WanGP hard-limits it to 2-15s) -- it only sets
    tone/rhythm and does NOT affect video_length.

    start_scene: scenes with index < start_scene (0-based) are skipped.
    skip_existing_output: reuse an output clip already on disk (assumed already
    trimmed from the run that produced it).
    """
    print("Loading full song into memory for per-scene audio slicing...")
    full_audio, sr = load_full_audio(audio_file)
    song_seconds = full_audio.shape[1] / sr
    print(f"  Loaded: {song_seconds:.2f}s @ {sr}Hz, "
          f"{full_audio.shape[0]} channel(s).")

    # The song's true end. Prefer the decoded length; last_end_value (from
    # transcription) should match, but the actual audio wins for alignment.
    song_end = max(float(last_end_value or 0.0), song_seconds)

    audio_slices_dir = os.path.join(audio_videos_dir, "audio_slices")
    audio_full_dir = os.path.join(audio_videos_dir, "audio_full")
    clips_dir = os.path.join(audio_videos_dir, "scene_clips")
    for d in (audio_slices_dir, audio_full_dir, clips_dir):
        os.makedirs(d, exist_ok=True)

    fps = config["h3_fps"]

    # --- H3 latent grid: valid render lengths are frames_offset +
    # frames_steps * k, clamped to [k_min, k_max]. For MiniMax H3 that's
    # 5 + 17k. We render the CEIL of each scene's exact target onto this
    # grid (so the clip is never shorter than its audio) and trim back after.
    frames_offset = config["wgp_frames_offset"]
    frames_steps = config["wgp_frames_steps"]
    k_min = max(1, -(-(config["wgp_frames_minimum"] - frames_offset) // frames_steps))  # ceil
    _max_total_frames = int(config.get("wgp_video_length_max_frames",
                                       config["wgp_frames_maximum"]))
    k_max = (_max_total_frames - frames_offset) // frames_steps                         # floor
    grid_min = frames_offset + frames_steps * k_min
    grid_max = frames_offset + frames_steps * k_max

    def _ceil_to_grid(frame_count: int) -> int:
        """Smallest valid grid length >= frame_count (clamped)."""
        k = -(-(frame_count - frames_offset) // frames_steps)  # ceil-div
        k = max(k_min, min(k_max, k))
        return frames_offset + frames_steps * k

    # The AUDIO REFERENCE clip is capped at wgp_audio_ref_max_seconds
    # (H3/WanGP enforce 2-15s) -- tone/rhythm only, independent of video_length.
    audio_ref_max = config["wgp_audio_ref_max_seconds"]
    audio_ref_min = config["wgp_audio_ref_min_seconds"]

    # ------------------------------------------------------------------
    # PASS 1: which scenes are renderable (have a reference image), in order,
    # with each one's real start time. A scene's END boundary is the NEXT
    # renderable scene's start (last runs to song end), so a skipped scene's
    # span is absorbed by its predecessor instead of becoming a silent gap
    # -- EXCEPT for the very first scene, whose START isn't absorbed by
    # anything, which is exactly the leading-gap bug guarded against below.
    # ------------------------------------------------------------------
    renderable = []  # list of (scene_index, scene, start_sec)
    skipped_starts = []
    for i, scene in enumerate(scenes):
        if i < start_scene:
            continue
        image_path = os.path.join(audio_images_dir, f"image_{str(i + 1).zfill(2)}.jpg")
        if not os.path.exists(image_path):
            print(f"Scene {i+1:02d}: SKIPPING \u2014 missing reference image: {image_path}")
            skipped_starts.append(float(scene.get("start", 0.0)))
            continue
        renderable.append((i, scene, float(scene.get("start", 0.0))))

    if not renderable:
        print("No renderable scenes.")
        return []

    # COVERAGE GUARD: when rendering from the true start of the song
    # (start_scene == 0), the first renderable scene MUST start at 0.0.
    # If it doesn't, force it back and warn loudly -- this is what silently
    # shortened the final video versus the audio before this fix.
    if start_scene == 0 and renderable[0][2] > 0.05:
        gap = renderable[0][2]
        i0, scene0, _ = renderable[0]
        print(f"  [coverage] WARNING: first renderable scene (index {i0}) "
              f"starts at {gap:.2f}s, not 0.0s -- {gap:.2f}s of the song's "
              f"beginning would be missing from the final video. Extending "
              f"this scene's span back to 0.0s to cover it. Likely causes: "
              f"beat-driven cutting failed/was disabled and fell back to "
              f"lyric timestamps (check earlier logs), or scene(s) covering "
              f"the intro were skipped for a missing reference image "
              f"(skipped scene starts: {[round(t, 2) for t in skipped_starts if t < gap]}).")
        renderable[0] = (i0, scene0, 0.0)

    # Boundaries: each renderable scene spans [start_k, start_{k+1}); the last
    # runs to song_end. Guard against non-monotonic starts.
    boundaries = []
    for pos, (i, scene, start_sec) in enumerate(renderable):
        if pos + 1 < len(renderable):
            end_sec = renderable[pos + 1][2]
        else:
            end_sec = song_end
        if end_sec < start_sec:
            end_sec = start_sec
        boundaries.append((start_sec, end_sec))

    # ------------------------------------------------------------------
    # PASS 2: assign frames with a running cursor locked to song time.
    # trim_frames = exact audio span (defines the timeline); num_frames =
    # that span ceiled onto H3's grid (what actually gets rendered).
    # ------------------------------------------------------------------
    jobs = []
    n_reused = 0
    cursor_frame = int(round(boundaries[0][0] * fps))  # 0 after the guard above
    song_end_frame = int(round(song_end * fps))

    for pos, (i, scene, start_sec) in enumerate(renderable):
        _start_sec, end_sec = boundaries[pos]
        is_last = (pos == len(renderable) - 1)

        # Target the TRUE boundary in frames so the cursor never drifts. The
        # last scene targets the song end exactly.
        target_end_frame = song_end_frame if is_last else int(round(end_sec * fps))

        # Exact audio span, in frames -- this is what the clip is trimmed to.
        trim_frames = max(1, target_end_frame - cursor_frame)

        # A span shorter than the grid minimum cannot be rendered below
        # grid_min frames, and the trim can only shrink DOWN to that span --
        # so such a clip inevitably runs slightly long. Warn (rare: only if
        # h3_min_scene_seconds * fps < grid_min).
        if trim_frames < grid_min:
            print(f"Scene {i+1:02d}: NOTE \u2014 exact span {trim_frames} "
                  f"frames is below the H3 grid minimum {grid_min}; the clip "
                  f"will render and stay near {grid_min} frames, running "
                  f"slightly long.")

        # Render the next grid length >= the exact target (audio as a floor).
        num_frames = _ceil_to_grid(trim_frames)

        # If the grid clamp forced the render BELOW the exact span (only
        # possible for a scene longer than grid_max), trimming can't recover
        # the missing tail -- cap the trim to what was rendered and warn so the
        # (rare) drift is visible rather than silent.
        if num_frames < trim_frames:
            print(f"Scene {i+1:02d}: WARNING \u2014 exact span {trim_frames} "
                  f"frames exceeds H3 grid max {grid_max}; clipping to "
                  f"{num_frames}. Consider raising wgp_video_length_max_frames "
                  f"or lowering h3_max_scene_seconds.")
            trim_frames = num_frames

        # Advance the cursor by the EXACT span (trim length), never by the
        # grid-inflated render length -- that's what keeps boundaries locked.
        actual_start_sec = cursor_frame / fps
        cursor_frame += trim_frames
        actual_end_sec = cursor_frame / fps
        span_seconds = trim_frames / fps

        image_path = os.path.join(audio_images_dir, f"image_{str(i + 1).zfill(2)}.jpg")
        out_name = f"scene_{str(i + 1).zfill(2)}_{timestamp}.mp4"
        out_path = os.path.abspath(os.path.join(clips_dir, out_name))

        if skip_existing_output and os.path.exists(out_path):
            print(f"Scene {i+1:02d}: already rendered at {out_path} -- reusing "
                  f"(assumed already trimmed to {trim_frames} frames / "
                  f"{span_seconds:.3f}s).")
            jobs.append({
                "scene_index": i,
                "prompt": None,
                "image_ref": os.path.abspath(image_path),
                "audio_ref": None,
                "audio_full": None,
                "num_frames": num_frames,
                "trim_frames": trim_frames,
                "seed": None,
                "output_path": out_path,
                "_rendered": True,
                "timeline_start_sec": round(actual_start_sec, 3),
                "timeline_end_sec": round(actual_end_sec, 3),
                "clip_seconds": round(span_seconds, 3),
            })
            n_reused += 1
            continue

        # FULL-span audio for gapless manual re-sync: exactly the audio under
        # this (trimmed) clip on the assembled timeline.
        full_chunk, _ = slice_audio_tensor(full_audio, sr, actual_start_sec, span_seconds)
        audio_full_path = os.path.join(
            audio_full_dir, f"scene_{str(i + 1).zfill(2)}.wav")
        if full_chunk.shape[1] > 0:
            sf.write(audio_full_path, full_chunk.T, sr)
        else:
            audio_full_path = None

        # AUDIO REFERENCE (tone/rhythm only, 2-15s): from the START of the span.
        ref_duration = min(span_seconds, audio_ref_max)
        ref_duration = max(ref_duration, audio_ref_min)
        audio_chunk, actual_ref_duration = slice_audio_tensor(
            full_audio, sr, actual_start_sec, ref_duration)
        if actual_ref_duration < audio_ref_min:
            fallback_start = max(0.0, actual_end_sec - audio_ref_min)
            audio_chunk, actual_ref_duration = slice_audio_tensor(
                full_audio, sr, fallback_start, audio_ref_min)
            if actual_ref_duration < 1.0:
                print(f"Scene {i+1:02d}: WARNING \u2014 only {actual_ref_duration:.2f}s "
                      f"of audio available for the reference near song end; "
                      f"rendering anyway so the timeline stays gapless.")

        slice_path = os.path.join(audio_slices_dir, f"scene_{str(i + 1).zfill(2)}.wav")
        sf.write(slice_path, audio_chunk.T, sr)

        prompt_text = assemble_h3_scene_prompt(scene, config)

        jobs.append({
            "scene_index": i,
            "prompt": prompt_text,
            "image_ref": os.path.abspath(image_path),
            "audio_ref": os.path.abspath(slice_path),
            "audio_full": os.path.abspath(audio_full_path) if audio_full_path else None,
            "num_frames": num_frames,      # grid-valid length H3 renders
            "trim_frames": trim_frames,    # exact length clip is cut back to
            "seed": secrets.randbits(31),
            "output_path": out_path,
            "timeline_start_sec": round(actual_start_sec, 3),
            "timeline_end_sec": round(actual_end_sec, 3),
            "clip_seconds": round(span_seconds, 3),
        })

        print(f"  Scene {i+1:02d}: render {num_frames} frames -> trim to "
              f"{trim_frames} = {span_seconds:.3f}s  "
              f"[{actual_start_sec:.2f}s -> {actual_end_sec:.2f}s]  "
              f"(song span {end_sec - start_sec:.2f}s)")

    n_to_render = len(jobs) - n_reused
    assembled_seconds = cursor_frame / fps
    drift = assembled_seconds - song_end
    print(f"Planned {len(jobs)} scene job(s) out of {len(scenes)} scene(s): "
          f"{n_reused} reused, {n_to_render} to render.")
    print(f"  Timeline: trimmed clips span {assembled_seconds:.2f}s vs song "
          f"{song_end:.2f}s (drift {drift:+.2f}s).")
    if drift < -0.5:
        print(f"  WARNING: assembled video is {-drift:.2f}s SHORTER than the "
              f"song -- there may be a gap or a grid-clamped over-long scene.")
    # Sanity: verify the timeline is gapless to within one frame.
    prev_end = None
    for job in jobs:
        s = job.get("timeline_start_sec")
        if s is None:
            continue
        if prev_end is not None and abs(s - prev_end) > (1.5 / fps):
            print(f"  WARNING: timeline gap of {s - prev_end:+.3f}s before "
                  f"scene {job['scene_index']+1:02d} -- clips will not be "
                  f"contiguous.")
        prev_end = job.get("timeline_end_sec")
    return jobs


# =============================================================================
# Wan2GP CLI adapter -- field names confirmed by reading
# https://github.com/fastflair/Wan2GP/blob/main/wgp.py and
# models/minimax_h3/minimax_h3_handler.py directly:
#   - model_type: "minimax_h3_ref2va" (or "minimax_h3_ref2va_pruned")
#   - image_prompt_type "I" + image_refs: [path]  -> use reference image(s)
#   - audio_prompt_type "A" + audio_guide: path    -> use one audio reference
#   - resolution "WxH", video_length (frames), seed, num_inference_steps
#   - guidance_phases: 1 = generate at resolution; 2 = H3 two-phase
#     latent upscale (draft, learned 3D resize, refine at resolution).
#     "2~" enables phase-2 spatial tiling.
#   - switch_threshold: phase-2 noise-level start (default 0.9035)
#   - NO negative_prompt -- the model_def sets "no_negative_prompt": True,
#     so this key is omitted entirely rather than sent and ignored.
#   - _parse_settings_json() accepts a JSON list of these param-dicts
#     directly (no "id"/"params" wrapper required) and runs them as one
#     queue, in order -- so the whole song is ONE wgp.py call.
# =============================================================================
H3_LATENT_UPSCALE_SCALES = (1.0, 1.5, 2.0, 2.5, 3.0)
WGP_PHASE_2_TILING_GUIDANCE = "2~"
WGP_PHASE_2_TILING_FLAG = "~"


def _parse_h3_latent_scale(raw) -> float:
    """Map CONFIG values like 2, "2x", "1.5", "none" onto a legal factor."""
    if raw is None or raw is False:
        return 1.0
    if isinstance(raw, str):
        text = raw.strip().lower().replace("×", "x")
        if text in ("", "none", "off", "false", "no", "1", "1.0", "1x"):
            return 1.0
        if text.endswith("x"):
            text = text[:-1]
        raw = text
    try:
        scale = float(raw)
    except (TypeError, ValueError):
        return 1.0
    return min(H3_LATENT_UPSCALE_SCALES, key=lambda s: abs(s - scale))


def h3_latent_upscale_enabled(config: dict) -> bool:
    backend = str(config.get("upscale_backend") or "").strip().lower()
    if backend not in ("h3_latent", "latent", "minimax_h3_latent"):
        return False
    return _parse_h3_latent_scale(config.get("h3_latent_upscale_scale", 2.0)) > 1.0 + 1e-6


def _snap_h3_pixels(value: float, align: int = 32) -> int:
    align = max(16, int(align or 32))
    return max(align, int(round(float(value) / align) * align))


def h3_output_resolution(config: dict) -> tuple:
    """Final WxH handed to wgp.py. Native canvas, or native * latent scale."""
    width = int(config["h3_width"])
    height = int(config["h3_height"])
    if not h3_latent_upscale_enabled(config):
        return width, height
    scale = _parse_h3_latent_scale(config.get("h3_latent_upscale_scale", 2.0))
    align = int(config.get("h3_latent_upscale_align", 32) or 32)
    return (_snap_h3_pixels(width * scale, align),
            _snap_h3_pixels(height * scale, align))


def _wgp_video_prompt_type_for_task(config: dict) -> str:
    """KI locks output size to the still. Two-phase needs the task resolution."""
    video_prompt_type = str(config.get("wgp_video_prompt_type") or "KI")
    if h3_latent_upscale_enabled(config) and "K" in video_prompt_type:
        video_prompt_type = video_prompt_type.replace("K", "")
        if "I" not in video_prompt_type:
            video_prompt_type = "I" + video_prompt_type
    if (h3_latent_upscale_enabled(config)
            and config.get("h3_latent_upscale_tiling")
            and WGP_PHASE_2_TILING_FLAG not in video_prompt_type):
        video_prompt_type += WGP_PHASE_2_TILING_FLAG
    return video_prompt_type


def apply_h3_latent_upscale_runtime(config: dict) -> None:
    """Export H3_TWO_PHASE_SCALE into the wgp.py subprocess env."""
    overrides = dict(config.get("wgp_env_overrides") or {})
    if h3_latent_upscale_enabled(config):
        scale = _parse_h3_latent_scale(config.get("h3_latent_upscale_scale", 2.0))
        overrides["H3_TWO_PHASE_SCALE"] = str(scale)
        overrides["H3_UPSCALER_REPO"] = overrides.get(
            "H3_UPSCALER_REPO", "LBH-123-AI/Minimax_h3_latent_Upscaler")
    else:
        overrides.pop("H3_TWO_PHASE_SCALE", None)
    config["wgp_env_overrides"] = overrides


def _wgp_task_from_scene(job: dict, config: dict) -> dict:
    out_w, out_h = h3_output_resolution(config)
    task = {
        "model_type": config["wgp_model_type"],
        "prompt": job["prompt"],
        "resolution": f"{out_w}x{out_h}",
        "video_length": job["num_frames"],
        "seed": job["seed"],
        "num_inference_steps": config["wgp_num_inference_steps"],
        # CRUCIAL: for ref2va, the flag that ENABLES reference images lives
        # in video_prompt_type, NOT image_prompt_type. wgp.py's
        # validate_settings() keeps image_refs only when "I" is in
        # video_prompt_type (else it nulls them to zero visual refs -- the
        # "found 0 visual and 1 audio" error). The handler's
        # image_ref_choices letters_filter is "KI"; "K" marks the first
        # reference image as the main subject that also defines output
        # dimensions, which is exactly our per-scene still. image_prompt_type
        # is a separate thing (start/end/continuation frames: S/E/T/V/L),
        # left empty here. Audio still goes through audio_prompt_type "A".
        # When H3 latent upscaling is on, K is stripped so the task
        # resolution (native * scale) wins instead of the still's size.
        "image_prompt_type": "",
        "video_prompt_type": _wgp_video_prompt_type_for_task(config),
        "image_refs": [job["image_ref"]],
        "audio_prompt_type": "A",
        "audio_guide": job["audio_ref"],
    }
    if h3_latent_upscale_enabled(config):
        tiling = bool(config.get("h3_latent_upscale_tiling"))
        task["guidance_phases"] = WGP_PHASE_2_TILING_GUIDANCE if tiling else 2
        task["switch_threshold"] = float(
            config.get("h3_latent_upscale_switch_threshold", 0.9035))
    else:
        task["guidance_phases"] = 1
    if config.get("wgp_use_turbo_lora"):
        # get_lora_local_path() in wgp.py joins this straight onto lora_dir
        # and checks os.path.isfile() -- it wants the FULL filename with
        # extension (matching exactly what ensure_turbo_lora_in_wgp() saved
        # to disk), not the basename. Stripping the extension here was the
        # bug behind "Loras files are missing or invalid" -- every task got
        # skipped before rendering even started.
        task["activated_loras"] = [config["wgp_turbo_lora_filename"]]
        task["loras_multipliers"] = config.get("wgp_lora_multiplier", "1.0")
    return task


def ensure_turbo_lora_in_wgp(config: dict) -> bool:
    """
    Make sure the Turbo EMA ckpt850 LoRA is somewhere wgp.py's own
    get_lora_dir() will actually find it, then download it if it isn't
    anywhere at all. Same caveat as before: this LoRA was published against
    the fl2va checkpoint partition, not ref2va -- watch the first few
    outputs closely.

    wgp.py resolves the MiniMax H3 lora directory via
    minimax_h3_handler.get_lora_dir(): `<lora_root>/minimax_h3` by default
    (lora_root defaults to "loras"), UNLESS your fork is run with
    --lora-dir-minimax-h3 pointing somewhere else -- e.g. the flat
    "loras/" folder some setups use directly. Rather than assume which one
    your setup expects, this checks every directory in
    config["wgp_lora_search_dirs"] for an existing copy and, if the file is
    found somewhere other than the canonical target
    (wgp_repo_dir/wgp_lora_subdir), copies it into the target too so
    wgp.py's default resolver finds it regardless of where you originally
    put it.
    """
    if not config.get("wgp_use_turbo_lora"):
        return False

    filename = config["wgp_turbo_lora_filename"]
    target_dir = os.path.join(config["wgp_repo_dir"], config["wgp_lora_subdir"])
    target_path = os.path.join(target_dir, filename)

    if os.path.exists(target_path):
        return True

    # Look for an existing copy anywhere plausible before downloading a new one.
    search_dirs = [target_dir] + [
        os.path.join(config["wgp_repo_dir"], d)
        for d in (config.get("wgp_lora_search_dirs") or [])
    ]
    found_path = None
    for d in search_dirs:
        candidate = os.path.join(d, filename)
        if os.path.exists(candidate):
            found_path = candidate
            break

    os.makedirs(target_dir, exist_ok=True)
    if found_path is not None:
        print(f"[wgp] Found Turbo LoRA at {found_path}; copying into "
              f"{target_path} so wgp.py's default resolver finds it too.")
        try:
            shutil.copy(found_path, target_path)
            return True
        except Exception as e:
            print(f"[wgp] WARNING: couldn't copy {found_path} -> {target_path} "
                  f"({e}). If that target directory is wrong for your setup, "
                  f"set wgp_lora_subdir to match wherever --lora-dir-minimax-h3 "
                  f"(or your fork's default) actually points, and re-run.")
            config["wgp_use_turbo_lora"] = False
            return False

    try:
        print(f"[wgp] Downloading Turbo LoRA into {target_path} ...")
        downloaded = hf_hub_download(
            repo_id=config["wgp_turbo_lora_repo"],
            filename=filename,
            token=config.get("hf_token") or None,
        )
        shutil.copy(downloaded, target_path)
        print("[wgp] Turbo LoRA ready. NOTE: published against fl2va's "
              "checkpoint, not ref2va's -- watch early outputs for quality "
              "regressions even though the file loaded fine.")
        return True
    except Exception as e:
        print(f"[wgp] WARNING: Turbo LoRA download failed ({e}). "
              f"Continuing without it -- set wgp_use_turbo_lora=False to "
              f"silence this and consider bumping wgp_num_inference_steps "
              f"back up to ~20 (the model's own default).")
        config["wgp_use_turbo_lora"] = False
        return False


def _build_conda_wrapped_command(cmd: list, config: dict) -> str:
    """
    Wrap `cmd` (a list, e.g. ["python", "wgp.py", "--process", ...]) as a
    single bash command string that:
      1. activates the conda env Wan2GP needs (config["wgp_conda_env"]),
      2. runs cmd inside it,
      3. switches back to config["wgp_conda_base_env"] afterward regardless
         of whether cmd succeeded or failed,
      4. exits with cmd's own exit code, so failures still propagate.

    Uses `conda info --base` to locate conda.sh rather than assuming the
    calling shell's ~/.bashrc already has `conda init` in it, so this works
    the same whether or not this script itself is run from an activated
    conda env.
    """
    quoted_cmd = " ".join(shlex.quote(part) for part in cmd)
    wgp_env = config["wgp_conda_env"]
    base_env = config["wgp_conda_base_env"]
    return (
        f'CONDA_BASE="$(conda info --base 2>/dev/null)"; '
        f'if [ -n "$CONDA_BASE" ] && [ -f "$CONDA_BASE/etc/profile.d/conda.sh" ]; then '
        f'  source "$CONDA_BASE/etc/profile.d/conda.sh"; '
        f'fi; '
        f'conda activate {shlex.quote(wgp_env)}; '
        f'{quoted_cmd}; '
        f'status=$?; '
        f'conda activate {shlex.quote(base_env)} 2>/dev/null || conda deactivate 2>/dev/null || true; '
        f'exit $status'
    )

def _clean_raw_seg_name(raw_path: str, scene_index: int) -> str:
    """
    wgp.py names its outputs "{datetime}_seed{seed}_{full prompt text}.mp4",
    which makes for absurdly long filenames. Rebuild a short name that keeps
    the useful "{datetime}_seed{seed}" prefix and replaces the prompt tail
    with a stable "seg_NN" segment tag, e.g.
      2026-08-15-16h06m42s_seed190678693_seg_01.mp4
    Falls back gracefully if the datetime/seed prefix can't be parsed.

    Used only at the very end, when a task's real output is moved into
    scene_clips/ -- "seg_NN" here is the FINAL SCENE index (1-based), the
    authoritative numbering for the finished video. Contrast with
    _live_seg_name() below, which numbers by TASK position within the
    current batch, purely so wgp_render/raw/ is readable while a run is
    still in progress.
    """
    base = os.path.basename(raw_path)
    stem, ext = os.path.splitext(base)
    if not ext:
        ext = ".mp4"
    m = re.match(r'^(.*?_seed\d+)', stem)
    prefix = m.group(1) if m else stem.split('_')[0]
    return f"{prefix}_seg_{scene_index + 1:02d}{ext}"


def _rename_in_place(path: str, new_name: str) -> str:
    """Rename `path` within its directory. Returns the new path, or the
    original if the rename fails."""
    new_path = os.path.join(os.path.dirname(path), new_name)
    if os.path.abspath(new_path) == os.path.abspath(path):
        return path
    try:
        os.replace(path, new_path)
        return new_path
    except OSError as e:
        print(f"[wgp] WARNING: couldn't rename {os.path.basename(path)} "
              f"-> {new_name} ({e}); using the original name.")
        return path


def _rename_raw_clip(raw_path: str, scene_index: int) -> str:
    """Rename a wgp.py raw output to the clean seg_NN convention, in place."""
    return _rename_in_place(raw_path, _clean_raw_seg_name(raw_path, scene_index))


def _live_seg_name(task_no: int, total_tasks: int, window_no: int,
                   window_total: int, seed) -> str:
    """
    Name for a raw clip the MOMENT it's detected during a live run -- purely
    for making wgp_render/raw/ readable while a batch is in progress (task
    number instead of an opaque timestamp, and an explicit window marker so
    multiple files for one multi-window scene are obviously related rather
    than looking like unrelated clips). e.g.
      02_w1of2_seed387287610.mp4
      02_w2of2_seed387287610.mp4
      03_seed2129198441.mp4          (single window: no window marker)

    This is DISTINCT from the final scene-index-based "seg_NN" rename that
    happens when a clip is moved to scene_clips/ (_clean_raw_seg_name) --
    task number here is position within THIS batch (to_render), which can
    differ from the true scene index on a resume where earlier scenes were
    already reused and skipped.
    """
    width = max(2, len(str(total_tasks)))
    prefix = f"{task_no:0{width}d}"
    window_part = f"_w{window_no}of{window_total}" if window_total and window_total > 1 else ""
    seed_part = f"_seed{seed}" if seed is not None else ""
    return f"{prefix}{window_part}{seed_part}.mp4"


def _rename_live(path: str, task_no: int, total_tasks: int,
                 window_no: int, window_total: int, seed) -> str:
    """Rename a just-saved raw clip to _live_seg_name(), in place."""
    return _rename_in_place(path, _live_seg_name(
        task_no, total_tasks, window_no, window_total, seed))


def _finalize_task_output(job: dict, produced_path: str, config: dict, label: str) -> None:
    """Rename a task's raw output to seg_NN, move it to the job's output_path,
    and conform it to trim_frames. Sets job["_rendered"] to the verified result."""
    produced_path = _rename_raw_clip(produced_path, job["scene_index"])
    os.makedirs(os.path.dirname(job["output_path"]), exist_ok=True)
    shutil.move(produced_path, job["output_path"])
    ok = conform_clip_to_frames(job["output_path"], job.get("trim_frames"), config["h3_fps"])
    job["_rendered"] = ok
    if ok:
        print(f"  Scene {job['scene_index']+1:02d} ({label}): "
              f"-> {os.path.basename(job['output_path'])}")
    else:
        print(f"  Scene {job['scene_index']+1:02d} ({label}): rendered but "
              f"failed to conform to the exact frame count -- will retry.")


def render_all_scenes_via_wgp(jobs: list, audio_videos_dir: str, config: dict) -> list:
    """ONE wgp.py invocation for the whole song: writes every scene's task
    into a single JSON list (the batch manifest), calls
    `wgp.py --process batch_manifest.json --output-dir <dir>` once, then
    maps each task's REAL output file from wgp.py's own stdout.

    OUTPUT MAPPING: wgp.py prints "[Task N/M]" at the start of each task and
    "New video saved to Path: <path>" every time it writes a video -- once
    per sliding window, so one task can produce several files. The LAST save
    seen inside a task's block is its complete, final-window result; earlier
    windows are deleted afterwards (wgp_cleanup_intermediate_window_files).
    If no such lines can be parsed (a different wgp.py version), files are
    matched by mtime order as an unreliable fallback, with a warning.

    Each save is renamed immediately to "{task}[_w{k}of{n}]_seed{seed}.mp4"
    so wgp_render/raw/ is readable mid-run; each task's final output is then
    renamed to seg_NN, moved to job["output_path"] and conformed to
    trim_frames. job["_rendered"] is True only when that conform verified
    the exact frame count, so a failed conform is retried.

    Jobs already marked _rendered on entry (reused clips) are never
    submitted. A wgp.py STARTUP crash (nonzero exit before any task
    completes) stamps job["_startup_crash"] on every pending job so the
    retry wrapper stops instead of re-invoking a process that will crash
    identically.
    """
    ensure_turbo_lora_in_wgp(config)
    apply_h3_latent_upscale_runtime(config)

    if not jobs:
        return jobs

    to_render = [job for job in jobs if not job.get("_rendered")]
    n_reused = len(jobs) - len(to_render)
    for job in to_render:
        job.pop("_startup_crash", None)

    if n_reused:
        print(f"\n{n_reused} scene(s) already had an output clip and are "
              f"being reused without calling wgp.py.")
    if not to_render:
        print("Nothing left to render -- every scene already has an output clip.")
        return jobs

    render_dir = os.path.abspath(os.path.join(audio_videos_dir, "wgp_render"))
    os.makedirs(render_dir, exist_ok=True)

    manifest_path = os.path.join(render_dir, "batch_manifest.json")
    tasks = [_wgp_task_from_scene(job, config) for job in to_render]
    with open(manifest_path, "w") as f:
        json.dump(tasks, f, indent=2)
    print(f"\nWrote batch manifest with {len(tasks)} scene task(s): {manifest_path}")

    raw_out_dir = os.path.join(render_dir, "raw")
    os.makedirs(raw_out_dir, exist_ok=True)

    cmd = [
        config["wgp_python_executable"], config["wgp_script"],
        "--process", manifest_path,
        "--output-dir", raw_out_dir,
    ] + list(config.get("wgp_extra_cli_args") or [])
    conda_cmd = _build_conda_wrapped_command(cmd, config)
    print(f"$ {' '.join(cmd)}  (cwd={config['wgp_repo_dir']}, "
          f"conda env: {config['wgp_conda_env']} -> {config['wgp_conda_base_env']})")

    if config.get("wgp_dry_run"):
        print("[dry-run] not executing wgp.py.")
        print(f"[dry-run] wrapped shell command:\n{conda_cmd}")
        for job in to_render:
            job["_rendered"] = False
        return jobs

    # --- stdout parsing state ---
    current_task_no = None
    current_window = {}      # task_no -> (window_no, window_total) last seen
    task_output_files = {}   # task_no -> LAST "New video saved to Path" seen for it
    all_seen_files = set()   # every save seen, for orphan cleanup accounting
    completed_task_nos = set()
    skipped_task_nos = set()
    stdout_lines = []
    returncode = None
    total_tasks = len(to_render)

    task_start_re = re.compile(r"\[Task (\d+)/\d+\]")
    window_re = re.compile(r"Sliding Window (\d+)/(\d+)")
    saved_path_re = re.compile(r"New video saved to Path:\s*(.+\.mp4)\s*$")
    completed_re = re.compile(r"Task (\d+) completed")
    skipped_re = re.compile(r"\[SKIP\] Task (\d+) failed")

    try:
        wgp_env = os.environ.copy()
        wgp_env.update(config.get("wgp_env_overrides") or {})
        for key in (config.get("wgp_env_unset") or []):
            wgp_env.pop(key, None)

        proc = subprocess.Popen(
            ["bash", "-c", conda_cmd], cwd=config["wgp_repo_dir"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env=wgp_env,
        )
        for line in proc.stdout:
            print(line, end="")
            stdout_lines.append(line)

            m = task_start_re.search(line)
            if m:
                current_task_no = int(m.group(1))

            m = window_re.search(line)
            if m and current_task_no is not None:
                current_window[current_task_no] = (int(m.group(1)), int(m.group(2)))

            m = saved_path_re.search(line)
            if m and current_task_no is not None:
                path = m.group(1).strip()
                win_no, win_total = current_window.get(current_task_no, (1, 1))
                seed = None
                if 1 <= current_task_no <= len(to_render):
                    seed = to_render[current_task_no - 1].get("seed")
                # Rename IMMEDIATELY so wgp_render/raw/ stays readable while
                # the batch is still running, not only after it finishes.
                path = _rename_live(path, current_task_no, total_tasks,
                                    win_no, win_total, seed)
                # Overwrite: the LAST save seen before the next [Task ...]
                # marker is the complete result for this task (the final
                # sliding window, or the only window if there's just one).
                task_output_files[current_task_no] = path
                all_seen_files.add(path)

            m = completed_re.search(line)
            if m:
                completed_task_nos.add(int(m.group(1)))

            m = skipped_re.search(line)
            if m:
                skipped_task_nos.add(int(m.group(1)))

        proc.wait(timeout=config.get("wgp_call_timeout_seconds", 3600))
        returncode = proc.returncode
    except Exception as e:
        print(f"[wgp] ERROR running wgp.py: {e}")

    if returncode not in (0, None) and not completed_task_nos:
        print(f"\n{'!'*70}")
        print(f"[wgp] wgp.py exited with code {returncode} before completing "
              f"any task -- this is a startup/crash failure, not a "
              f"per-scene generation issue. Last output:")
        print("".join(stdout_lines[-25:]))
        print(f"{'!'*70}\n")
        for job in to_render:
            job["_rendered"] = False
            job["_startup_crash"] = True
        return jobs

    n_orphans = len(all_seen_files) - len(task_output_files)
    if n_orphans > 0:
        print(f"[wgp] {n_orphans} intermediate sliding-window file(s) were "
              f"produced and correctly ignored (only each task's FINAL "
              f"window is used as its real output).")

    if not task_output_files:
        # Safety net: our stdout format assumptions didn't match anything
        # (e.g. a different wgp.py fork/version). Fall back to the old
        # count/mtime heuristic rather than failing every scene outright,
        # but warn loudly since this path is known to misalign under
        # sliding windows.
        print("[wgp] WARNING: could not parse any '[Task N/M]' / 'New video "
              "saved to Path' lines from stdout -- falling back to mtime/"
              "positional file matching. This is UNRELIABLE if any scene "
              "used sliding windows; verify results carefully.")
        produced = sorted(
            (os.path.join(raw_out_dir, f)
             for f in os.listdir(raw_out_dir) if f.lower().endswith(".mp4")),
            key=os.path.getmtime,
        )
        for pos, (job, produced_path) in enumerate(zip(to_render, produced), 1):
            _finalize_task_output(job, produced_path, config, f"fallback position {pos}")
        return jobs

    for idx, job in enumerate(to_render):
        task_no = idx + 1
        produced_path = task_output_files.get(task_no)
        if produced_path is None or not os.path.exists(produced_path):
            reason = ("validation failed" if task_no in skipped_task_nos
                      else "not completed" if produced_path is None
                      else "file missing on disk")
            print(f"  Scene {job['scene_index']+1:02d} (task {task_no}): "
                  f"no usable output -- {reason}.")
            job["_rendered"] = False
            continue
        _finalize_task_output(job, produced_path, config, f"task {task_no}")

    # Clean up leftover intermediate window files that were never selected
    # as any task's final output -- they're pure debris (each can be a
    # multi-GB video) and were only ever useful as inputs to the next
    # window's generation.
    if config.get("wgp_cleanup_intermediate_window_files", True):
        selected = set(task_output_files.values())
        removed = 0
        for f in all_seen_files - selected:
            try:
                if os.path.exists(f):
                    os.remove(f)
                    removed += 1
            except OSError as e:
                print(f"[wgp] NOTE: couldn't remove intermediate file {f}: {e}")
        if removed:
            print(f"[wgp] Cleaned up {removed} intermediate sliding-window "
                  f"file(s) from {raw_out_dir}.")

    return jobs

def reconcile_scene_clip_durations(clip_paths: list, audio_file: str,
                                   config: dict) -> dict:
    """
    Verifies (and if needed, corrects) that the sum of the given clips'
    durations matches the input audio file's duration exactly. Probes real
    on-disk frame counts (doesn't trust job["trim_frames"] bookkeeping, in
    case an earlier conform silently failed), and if the total is off,
    adjusts ONLY the last clip -- trims it if the total is long, pads it by
    holding its final frame if the total is short -- so the corrected sum
    matches the audio to within one frame (<0.1s at any fps this pipeline
    uses).

    Takes an explicit ordered list of clip paths (normally
    [job["output_path"] for job in jobs if job["_rendered"]]) rather than
    scanning scene_clips/ itself, so it can never pick up a stale leftover
    file from an earlier attempt in the same run directory and always
    matches exactly what the current render actually produced.

    NOTE: this corrects TIMING only. If a whole scene failed to render, this
    stretches/shrinks the tail clip to compensate rather than recreating the
    missing content -- rerun resume_video() to properly backfill a missing
    scene instead of relying on this for that case.
    """
    fps = config["h3_fps"]
    existing = [p for p in clip_paths if p and os.path.exists(p)]
    if not existing:
        print("[reconcile] no clips provided (or none exist on disk); nothing to do.")
        return {}

    info = sf.info(audio_file)
    song_seconds = info.frames / info.samplerate
    target_frames = int(round(song_seconds * fps))

    frame_counts = [max(0, _probe_video_frame_count(p)) for p in existing]
    total_frames = sum(frame_counts)
    delta = target_frames - total_frames

    print(f"[reconcile] {len(existing)} clip(s). Audio target: {target_frames}f "
          f"({song_seconds:.3f}s). Clips sum: {total_frames}f "
          f"({total_frames/fps:.3f}s). Delta: {delta:+d}f ({delta/fps:+.3f}s).")

    if abs(delta) < 1:
        print("[reconcile] already exact within one frame; nothing to do.")
        return {"target_frames": target_frames, "total_frames": total_frames}

    last_path = existing[-1]
    new_last_frames = max(1, frame_counts[-1] + delta)
    print(f"[reconcile] adjusting last clip ({os.path.basename(last_path)}) "
          f"from {frame_counts[-1]}f to {new_last_frames}f to close the gap "
          f"exactly.")
    ok = conform_clip_to_frames(last_path, new_last_frames, fps)
    if not ok:
        print(f"[reconcile] WARNING: conform of the final clip did not "
              f"verify cleanly; see the [conform] log line above for the "
              f"actual resulting frame count.")

    final_last = _probe_video_frame_count(last_path)
    final_total = total_frames - frame_counts[-1] + max(0, final_last)
    remaining_drift = (target_frames - final_total) / fps
    print(f"[reconcile] new total: {final_total}f ({final_total/fps:.3f}s) "
          f"vs audio {song_seconds:.3f}s (remaining drift "
          f"{remaining_drift:+.3f}s).")
    return {"target_frames": target_frames, "total_frames_before": total_frames,
           "total_frames_after": final_total, "adjusted_clip": last_path}

def _probe_video_dims(path: str) -> tuple:
    """Returns (width, height, pix_fmt) for a video file's first stream."""
    cmd = [
        "ffprobe", "-v", "error", "-select_streams", "v:0",
        "-show_entries", "stream=width,height,pix_fmt",
        "-of", "csv=p=0", path,
    ]
    out = subprocess.run(cmd, check=True, capture_output=True, text=True)
    parts = out.stdout.strip().split(",")
    return (int(parts[0]), int(parts[1]), parts[2] if len(parts) > 2 else "")


def _clip_dims(clip_paths: list, tag: str) -> tuple:
    """({path: (w, h, pix_fmt)}, {(w, h, pix_fmt): count}) for probe-able clips."""
    dims = {}
    for p in clip_paths:
        try:
            dims[p] = _probe_video_dims(p)
        except Exception as e:
            print(f"[{tag}] WARNING: couldn't probe {p} ({e})")
    counts = {}
    for d in dims.values():
        counts[d] = counts.get(d, 0) + 1
    return dims, counts


def check_clip_dimension_consistency(clip_paths: list) -> bool:
    """Prints every clip's (w, h, pix_fmt) and flags mismatches/odd dims --
    run this before concatenate_videos() whenever an upscale step is in the
    pipeline, since the concat demuxer feeds frames straight to the encoder
    with no scaling, and NVENC hard-fails on any resolution change mid-run."""
    dims, counts = _clip_dims(clip_paths, "dims")
    if not dims:
        return True
    if len(counts) == 1:
        (w, h, pf) = next(iter(counts))
        odd = (w % 2) or (h % 2)
        print(f"[dims] all {len(dims)} clips are {w}x{h} {pf}"
              f"{' -- ODD DIMENSION, NVENC will refuse this' if odd else ''}")
        return not odd
    print(f"[dims] MISMATCH across {len(dims)} clips -- {len(counts)} distinct "
          f"(w,h,pix_fmt) combos found (this alone explains an NVENC concat failure):")
    for d, n in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f"    {d}: {n} clip(s)")
    for p, d in dims.items():
        if counts[d] == 1:
            print(f"    outlier: {os.path.basename(p)} -> {d}")
    return False

def _ffmpeg_supports_nvenc() -> bool:
    """One-time probe: actually attempt a trivial h264_nvenc encode rather
    than just grepping `ffmpeg -encoders` (which can list an encoder that's
    compiled in but still fails to open, e.g. no GPU driver present). Result
    is cached for the life of the process so this only runs once per run."""
    global _NVENC_SUPPORTED
    if _NVENC_SUPPORTED is not None:
        return _NVENC_SUPPORTED
    cmd = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-f", "lavfi",
           "-i", "nullsrc=s=64x64:d=0.1", "-c:v", "h264_nvenc",
           "-f", "null", "-"]
    try:
        subprocess.run(cmd, check=True, capture_output=True, timeout=30)
        _NVENC_SUPPORTED = True
    except Exception as e:
        _NVENC_SUPPORTED = False
        reason = ""
        if isinstance(e, subprocess.CalledProcessError) and e.stderr:
            lines = e.stderr.decode("utf-8", "ignore").strip().splitlines()
            reason = lines[-1] if lines else ""
        print(f"[ffmpeg] h264_nvenc is not usable with this ffmpeg build/GPU "
              f"driver ({reason or e}); falling back to libx264 (CPU) for "
              f"all video encodes in this run. This is commonly because the "
              f"system ffmpeg package (e.g. Ubuntu's default 'ffmpeg' repo "
              f"package) was built without NVENC support -- a build that "
              f"includes it (conda-forge's ffmpeg, a BtbN static build, or "
              f"the jellyfin-ffmpeg PPA) would restore GPU encoding speed.")
    return _NVENC_SUPPORTED


def _resolve_video_codec(requested_codec: str) -> str:
    """Transparent fallback: if the caller asked for h264_nvenc but this
    ffmpeg/GPU combination can't actually use it, silently use libx264
    instead rather than crashing the pipeline at the final assembly step.
    Any other requested codec is passed through unchanged."""
    if requested_codec == "h264_nvenc" and not _ffmpeg_supports_nvenc():
        return "libx264"
    return requested_codec

def _h264_args(codec: str, nvenc_preset: str, nvenc_cq) -> list:
    """Encoder flags: NVENC VBR at constant quality, or the CPU fallback."""
    if codec == "h264_nvenc":
        return ["-c:v", "h264_nvenc", "-preset", nvenc_preset, "-tune", "hq",
                "-rc", "vbr", "-cq", str(nvenc_cq), "-b:v", "0"]
    return ["-c:v", codec, "-preset", "veryfast", "-crf", "16"]


def concatenate_videos(video_paths: list, out_path: str, config: dict = None) -> str:
    """Video-only concat (there is no real song audio track in these clips
    -- H3's own generated audio is discarded; you resync the real song
    afterward in your editor, same as before)."""
    if len(video_paths) == 1:
        shutil.copy(video_paths[0], out_path)
        return out_path
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as f:
        listfile = f.name
        for p in video_paths:
            f.write(f"file '{os.path.abspath(p)}'\n")
    try:
        cfg = config or {}
        codec = _resolve_video_codec(cfg.get("final_video_codec", "h264_nvenc"))
        cmd = (["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", listfile]
               + _h264_args(codec, cfg.get("final_nvenc_preset", "p5"),
                            cfg.get("final_nvenc_cq", 16))
               + ["-pix_fmt", "yuv420p", out_path])
        try:
            subprocess.run(cmd, check=True, capture_output=True)
        except subprocess.CalledProcessError as e:
            stderr_tail = e.stderr.decode("utf-8", "ignore")[-2000:] if e.stderr else ""
            raise RuntimeError(
                f"ffmpeg concat failed (exit {e.returncode}) using codec "
                f"'{codec}'. If this mentions an unrecognized nvenc option, "
                f"this ffmpeg build lacks NVENC support (_ffmpeg_supports_nvenc "
                f"should have caught that and fallen back to libx264 -- if "
                f"you're still seeing this, the probe itself may need "
                f"attention). If it's about mismatched inputs, check "
                f"normalize_clip_resolutions()'s printed output above. "
                f"ffmpeg stderr:\n{stderr_tail}"
            ) from e
    finally:
        os.unlink(listfile)
    return out_path

def _probe_video_frame_count(path: str) -> int:
    """Exact decoded frame count via ffprobe. Uses -count_frames because
    container-level nb_frames metadata is often missing/wrong on freshly
    muxed mp4s -- this forces a real decode-and-count."""
    cmd = [
        "ffprobe", "-v", "error", "-count_frames",
        "-select_streams", "v:0",
        "-show_entries", "stream=nb_read_frames",
        "-of", "default=nokey=1:noprint_wrappers=1",
        path,
    ]
    try:
        out = subprocess.run(cmd, check=True, capture_output=True, text=True)
        return int(out.stdout.strip())
    except Exception as e:
        print(f"[probe] WARNING: couldn't count frames in {path} ({e})")
        return -1


def conform_clip_to_frames(path: str, n_frames, fps) -> bool:
    """Force `path` to contain EXACTLY n_frames frames at EXACTLY `fps`
    (trimming, or holding the last frame when short). Returns True only if
    the on-disk result was verified to match; callers treat False as "this
    clip's duration is not trustworthy" and retry it.
    """
    if not n_frames or int(n_frames) <= 0:
        return True
    if not os.path.exists(path):
        print(f"[conform] WARNING: clip not found, cannot conform: {path}")
        return False

    n_frames = int(n_frames)
    actual = _probe_video_frame_count(path)

    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False,
                                     dir=os.path.dirname(path)) as tmp:
        tmp_path = tmp.name

    try:
        if actual >= 0 and actual < n_frames:
            deficit = n_frames - actual
            print(f"[conform] NOTE: {os.path.basename(path)} rendered only "
                  f"{actual}/{n_frames} requested frames ({deficit} short, "
                  f"{deficit / fps:.2f}s). Padding by holding the final "
                  f"frame so this clip still matches its audio span exactly.")
            pad = ["-vf", f"tpad=stop_mode=clone:stop_duration={deficit / fps:.6f}"]
        else:
            pad = []
        cmd = (["ffmpeg", "-y", "-i", path] + pad
               + ["-frames:v", str(n_frames), "-r", str(fps), "-an"]
               + _h264_args("libx264", "", 0) + [tmp_path])

        subprocess.run(cmd, check=True, capture_output=True)
        shutil.move(tmp_path, path)

        final_count = _probe_video_frame_count(path)
        if final_count < 0:
            print(f"[conform] WARNING: could not verify frame count after "
                  f"conforming {path}; treating as failed.")
            return False
        if final_count != n_frames:
            print(f"[conform] WARNING: {os.path.basename(path)} has "
                  f"{final_count} frames after conforming, expected "
                  f"{n_frames} ({final_count - n_frames:+d} frames, "
                  f"{(final_count - n_frames) / fps:+.3f}s).")
            return False
        return True
    except subprocess.CalledProcessError as e:
        stderr_tail = e.stderr.decode("utf-8", "ignore")[-300:] if e.stderr else ""
        print(f"[conform] WARNING: could not conform {os.path.basename(path)} "
              f"to {n_frames} frames ({stderr_tail}).")
        return False
    except Exception as e:
        print(f"[conform] WARNING: unexpected error conforming "
              f"{os.path.basename(path)} ({e}).")
        return False
    finally:
        if os.path.exists(tmp_path):
            os.unlink(tmp_path)


def render_scenes_with_retries(jobs: list, audio_videos_dir: str,
                               config: dict) -> list:
    """
    Wraps render_all_scenes_via_wgp() with automatic retry passes for scenes
    that failed to produce an output. This exists because errors like
    "CUDA driver error: device not ready" are transient GPU/driver hiccups
    (common on VRAM-oversubscribed WSL2 setups) rather than problems with
    the scene's configuration -- a scene that dies with one of those will
    usually succeed on a fresh attempt, once the GPU has settled.

    Between attempts, jobs that already have an output on disk are left
    marked _rendered=True; render_all_scenes_via_wgp() automatically skips
    those (it only submits jobs without _rendered), so each retry pass only
    re-attempts the scenes still missing an output. This reuses the exact
    same skip-existing machinery as resume_video(), so nothing already done
    is ever redone.

    A wgp.py STARTUP crash (flagged via job["_startup_crash"]) aborts the
    retry loop immediately -- re-invoking a process that crashes during its
    own imports would just waste every remaining attempt (and the sleep
    between them) on an identical, unrecoverable failure.

    Controlled by CONFIG:
      wgp_max_render_attempts : total attempts for the batch (default 3).
      wgp_retry_sleep_seconds : pause between attempts to let the GPU settle.
    """
    max_attempts = max(1, int(config.get("wgp_max_render_attempts", 3)))
    sleep_between = float(config.get("wgp_retry_sleep_seconds", 20))

    for attempt in range(1, max_attempts + 1):
        pending = [j for j in jobs if not j.get("_rendered")]
        if not pending:
            break

        if attempt > 1:
            print(f"\n{'#'*70}")
            print(f"# Render retry pass {attempt}/{max_attempts}: "
                  f"{len(pending)} scene(s) still need an output.")
            print("# (Transient GPU errors like 'device not ready' usually "
                  "clear on a retry.)")
            print("#" * 70)
            if sleep_between > 0:
                print(f"Waiting {sleep_between:.0f}s for the GPU to settle "
                      f"before retrying...")
                time.sleep(sleep_between)

        jobs = render_all_scenes_via_wgp(jobs, audio_videos_dir, config)

        # If wgp.py crashed at startup, retrying is pointless -- stop now.
        if any(j.get("_startup_crash") for j in jobs if not j.get("_rendered")):
            print("[render] wgp.py failed during startup; aborting retries "
                  "(this won't clear on its own -- fix the environment/deps "
                  "and rerun resume_video() with the same timestamp).")
            break

    still_missing = [j for j in jobs if not j.get("_rendered")]
    if still_missing:
        idxs = ", ".join(str(j["scene_index"] + 1) for j in still_missing)
        print(f"\n[render] {len(still_missing)} scene(s) still failed to "
              f"render: scenes {idxs}. They'll be omitted from the final cut; "
              f"rerun resume_video() with the same timestamp to try just those "
              f"again later (already-rendered scenes are skipped automatically).")
    return jobs


def _rendered_clips(jobs: list, audio_file: str = None, config: dict = None) -> list:
    """Rendered clip paths in scene order, after reporting missing scenes and
    reconciling the clips' total duration against the real audio file."""
    clips = [job["output_path"] for job in jobs
             if job.get("_rendered") and os.path.exists(job["output_path"])]
    missing_jobs = [job for job in jobs if not job.get("_rendered")]
    if missing_jobs:
        missing_seconds = sum(job.get("clip_seconds") or 0.0 for job in missing_jobs)
        idxs = ", ".join(str(job["scene_index"] + 1) for job in missing_jobs)
        print(f"NOTE: {len(missing_jobs)} scene(s) did not render "
              f"({missing_seconds:.2f}s of song coverage missing; scenes: "
              f"{idxs}). Rerun resume_video() with the same timestamp to "
              f"backfill just these scenes.")
    if clips and audio_file and config and config.get("reconcile_clip_durations", True):
        try:
            reconcile_scene_clip_durations(clips, audio_file, config)
        except Exception as e:
            print(f"[reconcile] WARNING: duration reconciliation failed "
                  f"({e}); continuing with clips as-is.")
    return clips


def assemble_final_video(jobs: list, audio_videos_dir: str, timestamp: str,
                         config: dict, audio_file: str = None) -> str:
    clips = _rendered_clips(jobs, audio_file, config)
    if not clips:
        print("No scenes rendered \u2014 nothing to assemble.")
        return ""

    # RESOLUTION GUARD: the concat demuxer feeds decoded frames straight to
    # the encoder with no scaling, so a resolution change mid-stream (most
    # likely a partially-failed 4x AI upscale pass -- upscale_rendered_
    # scene_clips() falls back to a clip's native, un-upscaled resolution on
    # a per-clip failure -- but this guard is source-agnostic) hard-crashes
    # the encoder. Check and, if needed, normalize BEFORE concatenation
    # rather than discovering the mismatch via a cryptic ffmpeg exit code.
    is_uniform = check_clip_dimension_consistency(clips)
    if not is_uniform:
        clips = normalize_clip_resolutions(clips, config)

    final_path = os.path.join(audio_videos_dir, f"final_{timestamp}.mp4")
    concatenate_videos(clips, final_path, config)

    if not config.get("h3_keep_scene_clips", True):
        for p in clips:
            os.remove(p)

    print(f"\nVideo phase complete. Final (video-only; no audio track): {final_path}")
    print(f"Next: bring {final_path} plus the original song into your "
          f"editor and sync/replace audio there.")
    return final_path

def summarize_rendered_scenes(jobs: list, audio_videos_dir: str,
                              config: dict = None, audio_file: str = None) -> list:
    """
    Used instead of assemble_final_video() when config["concat_final_video"]
    is False -- no ffmpeg concat happens, individual scene clips are simply
    left as-is under audio_videos_dir/scene_clips/. Returns the list of
    successfully rendered clip paths, in scene order.

    If audio_file and config are given, reconciles the clips' total
    duration against the real audio file (see reconcile_scene_clip_durations)
    before returning -- this is the path your own post-process (which sums
    scene_clips/ and lays it under the audio) actually depends on.
    """
    clips = _rendered_clips(jobs, audio_file, config)
    if clips:
        print(f"\n{len(clips)} scene clip(s) ready (no final concat, per "
              f"concat_final_video=False):")
        for p in clips:
            print(f"  {p}")
    else:
        print("No scenes rendered.")
    return clips

# =============================================================================
# 4x AI VIDEO UPSCALING / DETAIL RESTORATION
# =============================================================================
REALESRGAN_X4PLUS_URL = (
    "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/"
    "RealESRGAN_x4plus.pth"
)

def _ensure_realesrgan_weights(config: dict) -> str:
    weights_dir=os.path.abspath(config.get("upscale_weights_dir","./models/realesrgan"))
    os.makedirs(weights_dir,exist_ok=True)
    path=os.path.join(weights_dir,"RealESRGAN_x4plus.pth")
    if os.path.exists(path): return path
    from urllib.request import urlretrieve
    print(f"[upscale] downloading Real-ESRGAN weights -> {path}")
    tmp=path+".part"
    try:
        urlretrieve(REALESRGAN_X4PLUS_URL,tmp)
        os.replace(tmp,path)
    finally:
        if os.path.exists(tmp): os.remove(tmp)
    return path

def _patch_torchvision_functional_tensor_compat():
    """basicsr (a realesrgan dependency) imports the private module
    torchvision.transforms.functional_tensor, which torchvision removed in
    v0.17+ in favor of torchvision.transforms.functional. This is the single
    most common reason 'pip install realesrgan basicsr opencv-python' still
    raises an ImportError on a modern torch/torchvision install. This aliases
    the missing module to torchvision.transforms.functional before basicsr
    imports it -- a harmless no-op on older torchvision where the module
    still exists."""
    import sys
    try:
        import torchvision.transforms.functional_tensor  # noqa: F401
    except ImportError:
        try:
            import torchvision.transforms.functional as _F
            sys.modules["torchvision.transforms.functional_tensor"] = _F
        except ImportError:
            pass  # torchvision itself missing/broken; let the real import fail normally

def _load_realesrgan_upscaler(config: dict, tile_override: int = None):
    """Build a RealESRGANer for RealESRGAN_x4plus. RRDBNet comes from basicsr
    (realesrgan.archs.rrdbnet_arch does not exist) and RealESRGANer from
    realesrgan.utils, matching upstream's own inference scripts.
    """
    _patch_torchvision_functional_tensor_compat()
    try:
        from basicsr.archs.rrdbnet_arch import RRDBNet
        from realesrgan.utils import RealESRGANer
    except Exception as e:
        raise RuntimeError(
            f"Real-ESRGAN import failed ({type(e).__name__}: {e}). If you've "
            f"already run 'pip install realesrgan basicsr opencv-python', a "
            f"bare reinstall won't fix this twice -- the message above is the "
            f"real cause (commonly a basicsr/torchvision incompatibility, or "
            f"a numpy version mismatch)."
        ) from e
    model=RRDBNet(num_in_ch=3,num_out_ch=3,num_feat=64,num_block=23,num_grow_ch=32,scale=4)
    tile = tile_override if tile_override is not None else int(config.get("upscale_tile",192))
    return RealESRGANer(scale=4,model_path=_ensure_realesrgan_weights(config),model=model,
                        tile=tile,tile_pad=int(config.get("upscale_tile_pad",16)),
                        pre_pad=0,half=bool(config.get("upscale_use_half",True)))

def _check_realesrgan_available() -> tuple:
    """One-time import check with the real diagnostic message, cached for
    the run. Replaces re-discovering the identical import failure on every
    single scene (and every tile-size retry within each scene) with one
    clear diagnosis up front. Uses the same import paths as
    _load_realesrgan_upscaler() -- RRDBNet from basicsr, RealESRGANer from
    realesrgan.utils -- so this check and the actual load can never disagree
    about what's importable."""
    global _REALESRGAN_AVAILABLE, _REALESRGAN_IMPORT_ERROR
    if _REALESRGAN_AVAILABLE is not None:
        return _REALESRGAN_AVAILABLE, _REALESRGAN_IMPORT_ERROR
    _patch_torchvision_functional_tensor_compat()
    try:
        from basicsr.archs.rrdbnet_arch import RRDBNet  # noqa: F401
        from realesrgan.utils import RealESRGANer  # noqa: F401
        _REALESRGAN_AVAILABLE, _REALESRGAN_IMPORT_ERROR = True, None
    except Exception as e:
        _REALESRGAN_AVAILABLE = False
        _REALESRGAN_IMPORT_ERROR = f"{type(e).__name__}: {e}"
    return _REALESRGAN_AVAILABLE, _REALESRGAN_IMPORT_ERROR

def _ffmpeg_video_info(path: str) -> tuple:
    cmd=["ffprobe","-v","error","-select_streams","v:0","-show_entries","stream=r_frame_rate,nb_frames",
         "-of","default=noprint_wrappers=1:nokey=1",path]
    vals=subprocess.check_output(cmd,text=True).strip().splitlines()
    raw=vals[0] if vals else "24/1"
    if "/" in raw:
        a,b=raw.split("/",1); fps=float(a)/max(float(b),1e-9)
    else: fps=float(raw)
    try: frames=int(vals[1]) if len(vals)>1 else 0
    except ValueError: frames=0
    return fps,frames


def _encode_upscaled_frames(pattern: str,out_path: str,fps: float,config: dict,audio_source: str=None):
    codec = _resolve_video_codec(config.get("upscale_codec","h264_nvenc"))
    cmd=["ffmpeg","-y","-framerate",f"{fps:.08f}","-i",pattern]
    if audio_source and config.get("upscale_audio_copy",True):
        cmd += ["-i",audio_source,"-map","0:v:0","-map","1:a:0?"]
    cmd += _h264_args(codec, config.get("upscale_nvenc_preset","p5"),
                      config.get("upscale_nvenc_cq",16))
    cmd += ["-pix_fmt","yuv420p","-movflags","+faststart"]
    if audio_source and config.get("upscale_audio_copy",True): cmd += ["-c:a","copy"]
    cmd += [out_path]
    try:
        subprocess.run(cmd,check=True,capture_output=True)
    except subprocess.CalledProcessError as e:
        stderr_tail = e.stderr.decode("utf-8","ignore")[-400:] if e.stderr else ""
        raise RuntimeError(f"ffmpeg encode of upscaled frames failed: {stderr_tail}") from e


def _ffmpeg_rescale(src: str, dst: str, target_w: int, target_h: int, config: dict = None) -> str:
    """Re-encode `src` to exactly (target_w, target_h) via ffmpeg's scale
    filter. Used to bring a clip that fell back to native resolution up to
    whatever resolution the rest of the batch ended up at."""
    cfg = config or {}
    vf = (f"scale={target_w}:{target_h}:force_original_aspect_ratio=decrease,"
          f"pad={target_w}:{target_h}:(ow-iw)/2:(oh-ih)/2,setsar=1")
    codec = _resolve_video_codec(cfg.get("upscale_codec", "h264_nvenc"))
    cmd = ["ffmpeg", "-y", "-i", src, "-vf", vf]
    cmd += _h264_args(codec, cfg.get("upscale_nvenc_preset", "p5"),
                      cfg.get("upscale_nvenc_cq", 16))
    cmd += ["-pix_fmt", "yuv420p", "-an", dst]
    try:
        subprocess.run(cmd, check=True, capture_output=True)
    except subprocess.CalledProcessError as e:
        stderr_tail = e.stderr.decode("utf-8", "ignore")[-400:] if e.stderr else ""
        raise RuntimeError(f"ffmpeg rescale to {target_w}x{target_h} failed: {stderr_tail}") from e
    return dst

def normalize_clip_resolutions(clip_paths: list, config: dict) -> list:
    dims, counts = _clip_dims(clip_paths, "normalize")
    if not dims:
        return clip_paths
    target_w, target_h, _ = max(counts, key=lambda d: counts[d])
    target_w -= target_w % 2  # NVENC/yuv420p require even dimensions
    target_h -= target_h % 2

    if len(counts) == 1 and target_w == next(iter(counts))[0] and target_h == next(iter(counts))[1]:
        return clip_paths  # already uniform and even, nothing to do

    print(f"[normalize] target resolution {target_w}x{target_h} "
          f"(majority of {len(clip_paths)} clips); rescaling outliers...")

    out_paths = []
    for p in clip_paths:
        w, h, _ = dims.get(p, (target_w, target_h, ""))
        if (w, h) == (target_w, target_h):
            out_paths.append(p)
            continue
        base, ext = os.path.splitext(p)
        dst = base + "_norm" + ext
        try:
            _ffmpeg_rescale(p, dst, target_w, target_h, config)
            print(f"[normalize] rescaled {os.path.basename(p)} "
                  f"({w}x{h} -> {target_w}x{target_h})")
            out_paths.append(dst)
        except Exception as e:
            print(f"[normalize] WARNING: could not rescale {p} ({e}); "
                  f"keeping original -- concat may still fail on this clip.")
            out_paths.append(p)
    return out_paths        
    
def upscale_video_realesrgan(input_path: str,output_path: str,config: dict) -> str:
    """4x learned super-resolution of one clip (no frame interpolation).

    A CUDA out-of-memory error partway through a clip is retried with a
    smaller tile (halved each attempt, down to upscale_min_tile) before
    giving up, because a clip that falls back to native resolution would
    break the concat of an otherwise-upscaled batch (see
    normalize_clip_resolutions(), the last-resort safety net).
    """
    if config.get("upscale_skip_existing",True) and os.path.exists(output_path): return output_path
    try: import cv2
    except ImportError as e: raise RuntimeError("opencv-python is required for integrated video upscaling.") from e

    fps,frame_count=_ffmpeg_video_info(input_path)
    min_tile = int(config.get("upscale_min_tile", 64))
    tile = int(config.get("upscale_tile", 192))
    max_attempts = int(config.get("upscale_max_attempts", 3)) if config.get("upscale_retry_smaller_tile", True) else 1

    last_exc = None
    for attempt in range(1, max_attempts + 1):
        temp=tempfile.mkdtemp(prefix="realesrgan_video_")
        outdir=os.path.join(temp,"frames"); os.makedirs(outdir,exist_ok=True)
        cap=None; upsampler=None
        try:
            upsampler=_load_realesrgan_upscaler(config, tile_override=tile)
            cap=cv2.VideoCapture(input_path)
            if not cap.isOpened(): raise RuntimeError(f"Could not open scene clip: {input_path}")
            i=0
            while True:
                ok,frame=cap.read()
                if not ok: break
                enhanced,_=upsampler.enhance(frame,outscale=float(config.get("upscale_scale",4)))
                fn=os.path.join(outdir,f"frame_{i:08d}.png")
                if not cv2.imwrite(fn,enhanced): raise RuntimeError(f"Could not write {fn}")
                i+=1
                if i==1 or i%24==0 or (frame_count and i==frame_count):
                    print(f"  [upscale] {os.path.basename(input_path)}: {i}/{frame_count or '?'} (tile={tile})")
            if i==0: raise RuntimeError("No frames decoded.")
            _encode_upscaled_frames(os.path.join(outdir,"frame_%08d.png"),output_path,fps,config,
                                    input_path if config.get("upscale_audio_copy",True) else None)
            print(f"[upscale] complete: {output_path}")
            return output_path
        except Exception as e:
            last_exc = e
            is_last = (attempt == max_attempts) or (tile <= min_tile)
            print(f"[upscale] attempt {attempt}/{max_attempts} failed for "
                  f"{os.path.basename(input_path)} at tile={tile} ({e}).")
            if not is_last:
                tile = max(min_tile, tile // 2)
                print(f"[upscale] retrying with a smaller tile ({tile})...")
        finally:
            try:
                if cap is not None: cap.release()
            except Exception: pass
            try: del upsampler
            except Exception: pass
            try: reset_memory(device)
            except Exception: pass
            shutil.rmtree(temp,ignore_errors=True)

    raise RuntimeError(f"Real-ESRGAN upscale failed after {max_attempts} attempt(s): {last_exc}") from last_exc

def upscale_rendered_scene_clips(jobs: list,audio_videos_dir: str,config: dict) -> list:
    """Run the 4x AI upscale pass on every rendered scene clip.

    Upscaled clips go to a sibling "upscaled/" directory, so scene_clips/
    always holds exactly the native-resolution H3 outputs. job["output_path"]
    is repointed at the upscaled file, so reconciliation, dimension checks
    and concatenation work unchanged.
    """
    backend = str(config.get("upscale_backend") or "realesrgan").strip().lower()
    if backend in ("none", "off", "false"):
        print("[upscale] backend=none -- leaving native H3 clips as-is.")
        return jobs
    if backend in ("h3_latent", "latent", "minimax_h3_latent"):
        out_w, out_h = h3_output_resolution(config)
        scale = _parse_h3_latent_scale(config.get("h3_latent_upscale_scale", 2.0))
        print(f"[upscale] H3 latent two-phase already ran during generation "
              f"({scale:g}x -> {out_w}x{out_h}); skipping post-process ESRGAN.")
        return jobs
    if not config.get("upscale_scene_clips", False):
        return jobs
    if backend != "realesrgan":
        print(f"[upscale] backend '{config.get('upscale_backend')}' not integrated; using originals.")
        return jobs

    # Check ONCE, up front, instead of discovering the same import failure
    # independently for every scene (and every tile-size retry within it).
    available, err = _check_realesrgan_available()
    if not available:
        n_pending = sum(1 for j in jobs if j.get("_rendered"))
        print(f"[upscale] Real-ESRGAN is not usable in this environment -- "
              f"skipping the upscale pass entirely for all {n_pending} "
              f"rendered scene(s); keeping native-resolution H3 clips.")
        print(f"[upscale]   root cause: {err}")
        return jobs

    upscaled_dir = os.path.join(audio_videos_dir,
                                config.get("upscale_output_dir_name", "upscaled"))
    os.makedirs(upscaled_dir, exist_ok=True)

    print("\n"+"="*70+"\n4x AI SUPER-RESOLUTION / DETAIL PASS\n"+"="*70)
    print(f"[upscale] writing upscaled clips to: {upscaled_dir}")
    for job in jobs:
        src=job.get("output_path")
        if not job.get("_rendered") or not src or not os.path.exists(src): continue
        stem,_=os.path.splitext(os.path.basename(src))
        dst=os.path.join(upscaled_dir, stem+str(config.get("upscale_output_suffix","_4x"))+".mp4")
        try:
            out=upscale_video_realesrgan(src,dst,config)
            job["source_output_path"]=src; job["output_path"]=out; job["upscaled"]=True
        except Exception as e:
            print(f"[upscale] WARNING scene {job.get('scene_index',0)+1} failed: {e}")
            print("[upscale] Falling back to original H3 clip.")
            job["upscaled"]=False
    return jobs

# =============================================================================
# Orchestration
# =============================================================================
def _startup_banner(config: dict):
    beat = "on" if config.get("beat_driven_cutting", True) else "off"
    print("=" * 70)
    print("Music video generator v11 — lyric-driven scenes + Wan2GP H3")
    print("  * Local work         : Whisper transcription, music analysis, "
          "LLM scene authoring,")
    print("                         Z-Image reference stills, per-scene audio "
          "slicing.")
    print("  * Render (H3)        : ONE scene = ONE image = ONE Wan2GP "
          "ref2va call; H3 generates")
    print("                         the scene's motion from the starting "
          "image + prompt + audio.")
    print("                         Every scene is batched into a single "
          "wgp.py --process call.")
    print(f"  * Scene length       : {config['h3_min_scene_seconds']:.0f}"
          f"\u2013{config['h3_max_scene_seconds']:.0f}s "
          f"(beat-driven cutting: {beat}, energy-aware camera grammar).")
    print(f"  * Conda env          : activates '{config['wgp_conda_env']}' for "
          f"that call, switches back")
    print(f"                         to '{config['wgp_conda_base_env']}' "
          f"afterward regardless of outcome.")
    print(f"  * Model type         : {config['wgp_model_type']}")
    print(f"  * dry_run            : {config.get('wgp_dry_run')}")
    print(f"  * CGI movie magic    : {'on' if config.get('cgi_magic_enabled', True) else 'off'}")
    print(f"  * Scene heat         : follows each scene's own lines "
          f"(ceiling: {config.get('max_scene_heat', 'explicit')}; adults only)")
    backend = str(config.get("upscale_backend") or "realesrgan").strip().lower()
    if backend in ("h3_latent", "latent", "minimax_h3_latent") and h3_latent_upscale_enabled(config):
        scale = _parse_h3_latent_scale(config.get("h3_latent_upscale_scale", 2.0))
        out_w, out_h = h3_output_resolution(config)
        tiled = "tiled" if config.get("h3_latent_upscale_tiling") else "full-frame"
        print(f"  * Upscale            : H3 latent two-phase {scale:g}x during "
              f"generation ({config['h3_width']}x{config['h3_height']} -> "
              f"{out_w}x{out_h}, {tiled} refine)")
    elif backend == "realesrgan" and config.get("upscale_scene_clips", False):
        print(f"  * Upscale            : Real-ESRGAN "
              f"{config.get('upscale_scale', 4)}x after generation")
    else:
        print("  * Upscale            : off")
    print("=" * 70)


def _render_and_finish(config: dict, scenes: list, audio_file: str,
                       audio_images_dir: str, audio_videos_dir: str,
                       timestamp: str, last_end_value: float,
                       start_scene: int = 0, skip_existing_output: bool = True):
    """Shared tail of create_video() / resume_video(): build jobs, render via
    wgp.py with retries, optional upscale, then concat or summarize."""
    jobs = build_scene_jobs(
        config, scenes, audio_file, audio_images_dir, audio_videos_dir,
        timestamp, last_end_value,
        start_scene=start_scene, skip_existing_output=skip_existing_output,
    )
    jobs = render_scenes_with_retries(jobs, audio_videos_dir, config)
    jobs = upscale_rendered_scene_clips(jobs, audio_videos_dir, config)
    if config.get("concat_final_video", True):
        assemble_final_video(jobs, audio_videos_dir, timestamp, config,
                             audio_file=audio_file)
    else:
        summarize_rendered_scenes(jobs, audio_videos_dir, config=config,
                                  audio_file=audio_file)


def create_video(config, audio_file):
    """
    Full per-song pipeline:
      1. transcription -> scene authoring -> Z-Image stills (all local)
      2. build every scene's job spec + audio reference slice (local, no model)
      3. render every scene through ONE batched wgp.py call (external, Wan2GP)
      4. reconcile total clip duration against the real audio file
      5. concatenate the rendered scenes into one final video (local, ffmpeg)
    """
    _startup_banner(config)
    print(f"Processing audio file: {audio_file}")
    scenes, audio_images_dir, audio_videos_dir, last_end_value, timestamp = \
        process_audio_scenes(audio_file, config)
    if not scenes:
        print("No scenes produced; aborting this song.")
        return
    print(f"{len(scenes)} scenes:\n{json.dumps(scenes, indent=4)}")
    print(f"last_end_value: {last_end_value}  timestamp: {timestamp}")
    process_audio_images(config, scenes, audio_images_dir)
    _render_and_finish(config, scenes, audio_file, audio_images_dir,
                       audio_videos_dir, timestamp, last_end_value)


def resume_video(config, title: str, timestamp: str, audio_file: str,
                 start_scene: int = 0, skip_existing_output: bool = True,
                 make_stills: bool = False):
    """
    Resume a previous run WITHOUT repeating Whisper transcription, LLM
    scene authoring, or Z-Image stills -- reuses the scenes.json and
    reference images a prior create_video() call already wrote to disk for
    the given (title, timestamp), and only (re)does the render step.

    title/timestamp must match the folder names under base_working_dir /
    base_video_dir from the run you're resuming, e.g. for
    ./images/IgniteLight_20260808_175541/scenes.json that's
    title="IgniteLight", timestamp="20260808_175541".

    make_stills: True generates the reference stills from scenes.json first
    -- use it after reauthor_video(..., render=False) once the plan reads well.

    start_scene: 0-based scene index to resume from -- scenes before this
    are skipped entirely (matches the old script's start_image param).
    Usually you don't need this: skip_existing_output (default True)
    already reuses any scene that already has an output clip on disk from
    a prior attempt with this same timestamp, and only re-submits the rest
    to wgp.py -- so a simple rerun after a crash naturally continues where
    it left off without needing to know which scene number failed.
    """
    unique_id = f"{title}_{timestamp}"
    audio_images_dir = os.path.join(config["base_working_dir"], unique_id)
    audio_videos_dir = os.path.join(config["base_video_dir"], unique_id)
    scenes_file_path = os.path.join(audio_images_dir, "scenes.json")

    if not os.path.exists(scenes_file_path):
        print(f"ERROR: scenes file not found: {scenes_file_path}")
        return
    with open(scenes_file_path, "r") as f:
        scenes = json.load(f)
    print(f"Resuming '{title}' ({timestamp}): loaded {len(scenes)} scene(s) "
          f"from {scenes_file_path}")

    os.makedirs(audio_videos_dir, exist_ok=True)
    if make_stills:
        process_audio_images(config, scenes, audio_images_dir)

    # The song's total length is needed so the last scene runs exactly to the
    # song end. process_audio_scenes() computed it originally as last_end_value;
    # on resume we recover it straight from the audio file.
    try:
        last_end_value = get_audio_duration(audio_file)
    except Exception as e:
        print(f"WARNING: couldn't read audio duration ({e}); "
              f"falling back to the last scene's start + a default span.")
        last_end_value = ((float(scenes[-1].get("start", 0.0)) +
                           float(config.get("h3_max_scene_seconds", 30.0)))
                          if scenes else 0.0)

    _render_and_finish(config, scenes, audio_file, audio_images_dir,
                       audio_videos_dir, timestamp, last_end_value,
                       start_scene=start_scene,
                       skip_existing_output=skip_existing_output)


def reauthor_video(config, title: str, timestamp: str, audio_file: str,
                   render: bool = True):
    """
    Re-run the LLM authoring for a song that was already transcribed, into a
    FRESH timestamped folder (the old run is left untouched), then make the
    stills and render unless render=False. Reads the old run's
    transcript.json (or rebuilds the lines from its scenes.json) and reuses
    its detected genre and vocals, so Whisper and CLAP are skipped. Use it
    after changing SONG_DIRECTION, THEME or the prompts.
    """
    llm_preflight()
    old_dir = os.path.join(config["base_working_dir"], f"{title}_{timestamp}")
    t_path = os.path.join(old_dir, "transcript.json")
    if os.path.exists(t_path):
        with open(t_path) as f:
            t = json.load(f)
        raw = [(float(ts), str(txt)) for ts, txt in t["raw_segments"]]
        duration = float(t["duration"])
    else:
        with open(os.path.join(old_dir, "scenes.json")) as f:
            old = json.load(f)
        raw = [(float(s["start"]), s["text"].strip()) for s in old
               if (s.get("text") or "").strip()]
        duration = get_audio_duration(audio_file)
        print(f"[reauthor] no transcript.json; rebuilt {len(raw)} line(s) from scenes.json")
    prior = {}
    try:
        with open(os.path.join(old_dir, "entity_style_bible.json")) as f:
            prior = json.load(f)
    except Exception:
        pass

    new_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    audio_images_dir = os.path.join(config["base_working_dir"], f"{title}_{new_ts}")
    audio_videos_dir = os.path.join(config["base_video_dir"], f"{title}_{new_ts}")
    os.makedirs(audio_images_dir, exist_ok=True)
    os.makedirs(audio_videos_dir, exist_ok=True)
    ctx = {"raw_segments": raw, "last_end_value": duration, "timestamp": new_ts,
           "audio_images_dir": audio_images_dir, "audio_videos_dir": audio_videos_dir,
           "segments": merge_transcript_segments(raw, duration, config)}
    ctx["text_for_llm"] = _text_for_llm(ctx["segments"])
    _save_transcript(ctx)

    meta = get_song_metadata(audio_file, config)
    if not meta.get("genre"):
        prior_genre = str((prior.get("film_bible") or {}).get("genre") or "").strip()
        if prior_genre:
            meta["genre"], meta["genre_source"] = prior_genre, "prior run"
        else:
            meta = resolve_song_genre(audio_file, config, meta)
    ctx["meta"] = meta
    clean_transcript(ctx, meta, config)
    ctx["content"] = classify_song_content(ctx["text_for_llm"], config)
    ctx["vocals"] = _resolve_vocals(audio_file, config, meta,
                                    prior=(prior.get("content") or {}).get("vocals"))
    scenes, audio_images_dir, audio_videos_dir, last_end_value, new_ts = \
        author_scenes(audio_file, config, ctx)
    print(f"[reauthor] {len(scenes)} scene(s) -> {audio_images_dir}")
    if render and scenes:
        process_audio_images(config, scenes, audio_images_dir)
        _render_and_finish(config, scenes, audio_file, audio_images_dir,
                           audio_videos_dir, new_ts, last_end_value)
    return scenes, audio_images_dir


In [4]:
#_startup_banner(CONFIG)
# run new systems
USE_GROK = False
for audio_file in CONFIG["audio_files"]:
    create_video(CONFIG, audio_file)
    reset_memory(device)


Music video generator v11 — lyric-driven scenes + Wan2GP H3
  * Local work         : Whisper transcription, music analysis, LLM scene authoring,
                         Z-Image reference stills, per-scene audio slicing.
  * Render (H3)        : ONE scene = ONE image = ONE Wan2GP ref2va call; H3 generates
                         the scene's motion from the starting image + prompt + audio.
                         Every scene is batched into a single wgp.py --process call.
  * Scene length       : 3–26s (beat-driven cutting: on, energy-aware camera grammar).
  * Conda env          : activates 'wan2gp' for that call, switches back
                         to 'base' afterward regardless of outcome.
  * Model type         : minimax_h3_ref2va
  * dry_run            : False
  * CGI movie magic    : on
  * Scene heat         : follows each scene's own lines (ceiling: explicit; adults only)
  * Upscale            : Real-ESRGAN 4x after generation
Processing audio file: /mnt/d/Share/Audio/Fran

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Total audio duration: 233.56s
 > Processing chunk: 0s to 30s


[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_ne

 > Processing chunk: 25s to 55s


...

    [krea] clone-score=0.068 (max 0.90) attempt 1/3 seed=985994953
Scene 35/36: [low|story] Medium shot, eye-level. Frankenstein's Monster; Massive square-built man with mo...


  0%|          | 0/12 [00:00<?, ?it/s]

    [krea] clone-score=0.252 (max 0.90) attempt 1/3 seed=716882075
Scene 36/36: [low|story] Close-up, low angle. Dr. Frankenstein; Wiry German man in his mid-50s, pale weat...


  0%|          | 0/12 [00:00<?, ?it/s]

    [krea] clone-score=0.073 (max 0.90) attempt 1/3 seed=3907449806
VRAM after Z-Image release: 0.0 GB alloc, 0.0 GB reserved
Loading full song into memory for per-scene audio slicing...
  Loaded: 233.56s @ 48000Hz, 2 channel(s).
  Scene 01: render 328 frames -> trim to 313 = 13.042s  [0.00s -> 13.04s]  (song span 13.04s)
  Scene 02: render 226 frames -> trim to 219 = 9.125s  [13.04s -> 22.17s]  (song span 9.14s)
Scene 03: NOTE — exact span 90 frames is below the H3 grid minimum 107; the clip will render and stay near 107 frames, running slightly long.
  Scene 03: render 107 frames -> trim to 90 = 3.750s  [22.17s -> 25.92s]  (song span 3.76s)
  Scene 04: render 192 frames -> trim to 182 = 7.583s  [25.92s -> 33.50s]  (song span 7.55s)
Scene 05: NOTE — exact span 90 frames is below the H3 grid minimum 107; the clip will render and stay near 107 frames, running slightly long.
  Scene 05: render 107 frames -> trim to 90 = 3.750s  [33.50s -> 37.25s]  (song span 3.79s)
  Scene 06: render 192

In [ ]:
USE_GROK = True
# run new systems
for audio_file in CONFIG["audio_files"]:
    create_video(CONFIG, audio_file)
    reset_memory(device)


Music video generator v11 — lyric-driven scenes + Wan2GP H3
  * Local work         : Whisper transcription, music analysis, LLM scene authoring,
                         Z-Image reference stills, per-scene audio slicing.
  * Render (H3)        : ONE scene = ONE image = ONE Wan2GP ref2va call; H3 generates
                         the scene's motion from the starting image + prompt + audio.
                         Every scene is batched into a single wgp.py --process call.
  * Scene length       : 3–26s (beat-driven cutting: on, energy-aware camera grammar).
  * Conda env          : activates 'wan2gp' for that call, switches back
                         to 'base' afterward regardless of outcome.
  * Model type         : minimax_h3_ref2va
  * dry_run            : False
  * CGI movie magic    : on
  * Scene heat         : follows each scene's own lines (ceiling: explicit; adults only)
  * Upscale            : Real-ESRGAN 4x after generation
Processing audio file: /mnt/d/Share/Audio/Fran

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

Total audio duration: 233.56s
 > Processing chunk: 0s to 30s


[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_ne

 > Processing chunk: 25s to 55s


...

 > Processing chunk: 225s to 233.56s
[genre] No genre in song_metadata -- classifying audio with CLAP (laion/clap-htsat-unfused)...


Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

[genre] CLAP: pop (28%), runner-up heavy metal (16%) -> using 'pop metal'
    [transcript]  120.1s  'Frankensteins Accord, yeah' -> "Frankenstein's accordion, yeah"
    [transcript]  132.0s  'Metal riffs with polka whales' -> 'Metal riffs with polka wails'
    [transcript]  166.0s  'From crypts and tombs the dead arise, dancing under starlit skies, a fusion sound both wild and keen, the anthem of this mencarp scene.' -> 'From crypts and tombs the dead arise, dancing under starlit skies, a fusion sound both wild and keen, the anthem of this madcap scene.'
    [transcript]  185.2s  "Frankenstein's a courting year" -> "Frankenstein's accordion"
    [transcript]  200.0s  'Metal hearts and pokesouls, let the haunted music roll' -> 'Metal hearts and polka souls, let the haunted music roll'
  [transcript] 5 line(s) corrected, 6 dropped.
Reading the whole lyric (subject, shape, heat)...
  [content] heat=none, shape=narrative -- Frankenstein's monster wakes under a pale moon and leads ghosts, w

Loading weights:   0%|          | 0/447 [00:00<?, ?it/s]

[vocals] CLAP: duet 51%, instrumental 18%, male 10%
Generating Film Bible (performers, cast, places, chapters)...


## Re-author a previous song (skips Whisper/CLAP; new folder)

In [ ]:
# Step 1: re-run the LLM authoring with the current SONG_DIRECTION / prompts
# (skips Whisper/CLAP, writes a NEW folder). render=False stops after
# scenes.json so the plan can be reviewed first.
USE_GROK = False
scenes, out_dir = reauthor_video(CONFIG, "FrankensteinsAccordion", "20261016_180955",
                                 CONFIG["audio_files"][0], render=False)

# Step 2: when the plan reads well, make the stills and render it:
# resume_video(CONFIG, "FrankensteinsAccordion", out_dir[-15:],   # its YYYYmmdd_HHMMSS
#              CONFIG["audio_files"][0], make_stills=True)


## Run Previous Generation

In [ ]:
# resume a saved run (reuses scenes.json + stills; only renders)
timestamp = '20261014_173335'
title = 'Boyz in the Hood_mod'

resume_video(CONFIG, title=title, timestamp=timestamp,
            audio_file=CONFIG["audio_files"][0])

In [ ]:
resume_video(CONFIG, "01_ItsCountry_mod", "20260816_042443",
             "/mnt/d/Share/Audio/01_ItsCountry_mod.mp3",
             skip_existing_output=False)